
# Notebook 06 — Stealth & Injection Count Feasibility

## Objective
Evaluate and select the most suitable injection-count formula and stealth multipliers (High, Medium, Low) based on natural user behavior and historical Baseline statistics.

The analysis focuses on event counts across Email, USB, and HTTP, without modifying the original data.

## Roadmap
1. **Data Preparation:** Load and validate Notebook 03 and 05 outputs and existing Baseline statistics.
2. **Formula Evaluation:** Compare candidate injection formulas without assuming the V1 formula is optimal.
3. **Multiplier Optimization:** Test different High, Medium, and Low stealth multipliers.
4. **Behavioral Impact:** Evaluate daily, cumulative, and cross-channel changes using hypothetical injections.
5. **Integrated Feasibility:** Assess compatibility with structurally feasible users and windows from Notebook 05.
6. **Final Decision:** Select and justify the formula and multipliers.
7. **Validation & Documentation:** Verify results, generate data dictionaries, and organize outputs.

## Expected Outcomes
- Which injection formula is most suitable, and why?
- Which multipliers best represent High, Medium, and Low Stealth?
- How do hypothetical injections affect natural user behavior?
- Are the selected rules compatible with Notebook 05?
- Are the results independently validated and documented?

**Next:** Notebook 07 — Scenario Catalog Design.



### 1.1 — Input Discovery & Source Inventory

**Objective:** Identify and inspect the available outputs from Notebooks 02, 03, and 05 before selecting the inputs for stealth and injection-count analysis.

**Source Roles:**
- **Notebook 02:** Temporal windows and natural channel availability.
- **Notebook 03:** Natural event frequencies and historical Baseline comparisons.
- **Notebook 05:** Approved structural rules and compatible attack windows.

**Expected Outcome:** A verified inventory of available datasets, their dimensions, and relevant columns.

**Constraint:** Read-only inspection. No original datasets or reports will be modified.


In [2]:

from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq

# ============================================================
# CELL 1.1 — INPUT DISCOVERY & SOURCE INVENTORY
# ============================================================

# 1. Locate New_Attack_Simulation
cwd = Path.cwd().resolve()

search_locations = [
    cwd,
    *cwd.parents,
    cwd / "New_Attack_Simulation",
    cwd.parent / "New_Attack_Simulation",
]

ROOT = next(
    (
        p for p in search_locations
        if (p / "reports").is_dir()
        and any(
            x.is_dir() and x.name.startswith("05_")
            for x in (p / "reports").iterdir()
        )
    ),
    None
)

if ROOT is None:
    raise FileNotFoundError(
        f"Could not locate New_Attack_Simulation from {cwd}"
    )

REPORTS = ROOT / "reports"

print("=" * 70)
print("STEP 1 — PROJECT LOCATION")
print("=" * 70)
print("Project:", ROOT)
print("Reports:", REPORTS)

# 2. Find the three notebook report folders
notebooks = {
    "NB02": "02_",
    "NB03": "03_",
    "NB05": "05_",
}

source_dirs = {}

for name, prefix in notebooks.items():
    matches = sorted(
        p for p in REPORTS.iterdir()
        if p.is_dir() and p.name.startswith(prefix)
    )

    if len(matches) != 1:
        raise RuntimeError(
            f"{name}: expected one report folder, "
            f"found {len(matches)}: {[p.name for p in matches]}"
        )

    source_dirs[name] = matches[0]

    print(f"{name}: {matches[0].name} — FOUND")

# 3. Inventory existing Parquet and CSV datasets
print("\n" + "=" * 70)
print("STEP 2 — DATASET INVENTORY")
print("=" * 70)

inventory = []
schemas = {}

for notebook, folder in source_dirs.items():

    files = sorted(
        p for p in folder.rglob("*")
        if p.is_file()
        and p.suffix.lower() in {".parquet", ".csv"}
        and "data_dictionaries"
            not in p.relative_to(folder).parts
        and "dictionary" not in p.name.lower()
        and "manifest" not in p.name.lower()
    )

    print(f"\n{notebook} — {folder.name}")
    print(f"Datasets found: {len(files)}")

    for path in files:
        relative = str(path.relative_to(folder))

        if path.suffix.lower() == ".parquet":
            metadata = pq.read_metadata(path)
            columns = pq.ParquetFile(
                path
            ).schema_arrow.names
            rows = metadata.num_rows

        else:
            columns = pd.read_csv(
                path, nrows=0
            ).columns.tolist()
            rows = None

        schemas[(notebook, relative)] = columns

        inventory.append({
            "notebook": notebook,
            "file": relative,
            "format": path.suffix.lower(),
            "rows": rows,
            "columns": len(columns),
        })

        row_text = f"{rows:,}" if rows is not None else "—"

        print(
            f"  {relative} | "
            f"Rows: {row_text} | "
            f"Columns: {len(columns)}"
        )

inventory_df = pd.DataFrame(inventory)

# 4. Inspect Notebook 05 integrated output
print("\n" + "=" * 70)
print("STEP 3 — NOTEBOOK 05 INTEGRATED DATASET")
print("=" * 70)

STRUCTURAL_PATH = (
    source_dirs["NB05"]
    / "integrated_structural_window_audit.parquet"
)

if not STRUCTURAL_PATH.is_file():
    raise FileNotFoundError(
        f"Missing integrated structural dataset: {STRUCTURAL_PATH}"
    )

metadata = pq.read_metadata(STRUCTURAL_PATH)
columns = pq.ParquetFile(
    STRUCTURAL_PATH
).schema_arrow.names

print("File:", STRUCTURAL_PATH.name)
print("Rows:", f"{metadata.num_rows:,}")
print("Columns:", len(columns))

print("\nColumn names:")
for col in columns:
    print(" -", col)

# 5. Final inventory summary
print("\n" + "=" * 70)
print("STEP 4 — INVENTORY SUMMARY")
print("=" * 70)

print(
    inventory_df.groupby("notebook")
    .size()
    .rename("dataset_count")
    .to_string()
)

print("\nPASS — Source discovery completed.")
print("No files were modified or created.")


STEP 1 — PROJECT LOCATION
Project: C:\Users\hp\Documents\FYP_Data_Exfiltration_Organized\FYP_Data_Exfiltration\New_Attack_Simulation
Reports: C:\Users\hp\Documents\FYP_Data_Exfiltration_Organized\FYP_Data_Exfiltration\New_Attack_Simulation\reports
NB02: 02_Temporal_Window_Audit — FOUND
NB03: 03_Channel_Behavior_Audit — FOUND
NB05: 05_Simulation_Rules_Feasibility — FOUND

STEP 2 — DATASET INVENTORY

NB02 — 02_Temporal_Window_Audit
Datasets found: 2
  rotation_feasibility_audit.parquet | Rows: 22,296 | Columns: 52
  user_window_length_audit.parquet | Rows: 387,162 | Columns: 116

NB03 — 03_Channel_Behavior_Audit
Datasets found: 2
  candidate_behavior_profile.parquet | Rows: 995 | Columns: 20
  user_channel_frequency_comparison.parquet | Rows: 2,281 | Columns: 27

NB05 — 05_Simulation_Rules_Feasibility
Datasets found: 17
  02_Switching\natural_window_switching_audit.parquet | Rows: 472,667 | Columns: 20
  02_Switching\natural_window_switching_enriched.parquet | Rows: 472,667 | Columns: 30


### 1.2 — Source Schema & Key Verification

**Objective:** Inspect the schemas of Notebook 02 and 03 outputs, verify user coverage, and confirm the structurally compatible population from Notebook 05.

**Expected Outcome:** Identify available columns, possible join keys, and any user-coverage differences between sources.

**Constraint:** Read-only verification without modifying existing datasets.


In [3]:

# ============================================================
# CELL 1.2 — SOURCE SCHEMA & KEY VERIFICATION
# ============================================================

source_files = {
    "NB02 — Window Length Audit":
        source_dirs["NB02"] / "user_window_length_audit.parquet",

    "NB02 — Rotation Feasibility":
        source_dirs["NB02"] / "rotation_feasibility_audit.parquet",

    "NB03 — Candidate Profile":
        source_dirs["NB03"] / "candidate_behavior_profile.parquet",

    "NB03 — Channel Frequency":
        source_dirs["NB03"] / "user_channel_frequency_comparison.parquet",
}

# ------------------------------------------------------------
# 1. Inspect source columns
# ------------------------------------------------------------
print("=" * 70)
print("STEP 1 — SOURCE SCHEMAS")
print("=" * 70)

source_users = {}

for name, path in source_files.items():

    if not path.is_file():
        raise FileNotFoundError(path)

    parquet_file = pq.ParquetFile(path)
    columns = parquet_file.schema_arrow.names

    print(f"\n{name}")
    print("-" * 60)
    print("Rows:", f"{parquet_file.metadata.num_rows:,}")
    print("Columns:", len(columns))

    if name.startswith("NB03"):
        # Show all columns in the behavioral reference files.
        for col in columns:
            print(" -", col)
    else:
        # Focus on potentially relevant Notebook 02 columns.
        keywords = (
            "user", "window", "channel", "switch",
            "rotation", "support", "start", "end",
            "duration", "count", "day"
        )

        relevant = [
            col for col in columns
            if any(k in col.lower() for k in keywords)
        ]

        print("Potentially relevant columns:")
        for col in relevant:
            print(" -", col)

    if "user" in columns:
        users = set(
            pd.read_parquet(path, columns=["user"])
              ["user"]
              .dropna()
              .unique()
        )
        source_users[name] = users
        print("Unique users:", f"{len(users):,}")
    else:
        print("User coverage: no 'user' column")

# ------------------------------------------------------------
# 2. Verify Notebook 05 structural population
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("STEP 2 — STRUCTURAL COMPATIBILITY VERIFICATION")
print("=" * 70)

structural = pd.read_parquet(
    STRUCTURAL_PATH,
    columns=[
        "user",
        "window_workdays",
        "window_start",
        "window_end",
        "structural_pass",
    ]
)

if structural["structural_pass"].isna().any():
    raise ValueError("Null structural_pass values detected.")

pass_mask = structural["structural_pass"].eq(True)

all_users = set(structural["user"].dropna().unique())
passed_users = set(
    structural.loc[pass_mask, "user"].dropna().unique()
)

print("All audited windows:", f"{len(structural):,}")
print("Compatible windows:", f"{pass_mask.sum():,}")
print("Audited users:", f"{len(all_users):,}")
print("Compatible users:", f"{len(passed_users):,}")
print("Users without compatible windows:",
      f"{len(all_users - passed_users):,}")

assert len(structural) == 472_667
assert int(pass_mask.sum()) == 288_486
assert len(all_users) == 995
assert len(passed_users) == 936

# ------------------------------------------------------------
# 3. Compare user coverage across notebooks
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("STEP 3 — USER COVERAGE RECONCILIATION")
print("=" * 70)

for name, users in source_users.items():
    missing = passed_users - users

    print(f"\n{name}")
    print("Compatible NB05 users present:",
          f"{len(passed_users & users):,} / 936")
    print("Compatible NB05 users missing:", len(missing))

    if missing:
        print("Missing user examples:", sorted(missing)[:10])

print("\n" + "=" * 70)
print("CELL 1.2 COMPLETED")
print("=" * 70)
print("No source files were modified.")


STEP 1 — SOURCE SCHEMAS

NB02 — Window Length Audit
------------------------------------------------------------
Rows: 387,162
Columns: 116
Potentially relevant columns:
 - user
 - window_workdays
 - eligible_windows
 - calendar_contiguous_windows
 - cross_gap_windows
 - has_calendar_contiguous_option
 - windows_NONE
 - windows_E
 - windows_U
 - windows_EU
 - windows_H
 - windows_EH
 - windows_UH
 - windows_EUH
 - windows_one_observed_channel
 - windows_two_observed_channels
 - windows_three_observed_channels
 - windows_2plus_observed_channels
 - windows_any_observed_channel
 - observed_2plus_windows
 - distinct_day_2plus_windows
 - same_day_only_2plus_windows
 - observed_three_windows
 - distinct_day_three_windows
 - three_without_assignment_windows
 - three_pair_constraint_fail_windows
 - windows_any_ordered_pair
 - windows_any_three_step_change
 - windows_three_distinct_order
 - windows_return_pattern
 - supported_pair_pattern_types
 - supported_three_step_pattern_types
 - windows_a


### 1.3 — Natural Counts & Baseline Source Verification

**Objective:** Locate the original daily event-count datasets and verify the availability of historical Baseline frequency statistics for each candidate user and channel.

**Expected Outcomes:**
- Identify the source of daily Email, USB, and HTTP event counts.
- Verify historical Baseline coverage by channel.
- Identify missing or incomplete Baseline references.
- Confirm which datasets are required for subsequent formula evaluation.

**Constraint:** Read-only inspection. No Baseline recalculation or source modification.


In [4]:

# ============================================================
# CELL 1.3 — NATURAL COUNTS & BASELINE SOURCE VERIFICATION
# ============================================================

import os
from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq

# ------------------------------------------------------------
# STEP 1 — Discover original data sources
# ------------------------------------------------------------

print("=" * 70)
print("STEP 1 — ORIGINAL DATA SOURCE DISCOVERY")
print("=" * 70)

SEARCH_ROOT = ROOT.parent

skip_dirs = {
    ".git", ".venv", "venv", "__pycache__",
    "node_modules", "reports", "data_dictionaries"
}

source_candidates = []

for directory, subdirs, files in os.walk(SEARCH_ROOT):
    subdirs[:] = [
        d for d in subdirs
        if d.lower() not in skip_dirs
    ]

    for filename in files:
        name = filename.lower()

        if not name.endswith(".parquet"):
            continue

        if (
            "user_day_r62_v1" in name
            or "behavior_features_r62_v1" in name
            or "baseline" in name
        ):
            source_candidates.append(
                Path(directory) / filename
            )

source_candidates = sorted(set(source_candidates))

print("Search root:", SEARCH_ROOT)
print("Matching Parquet files:", len(source_candidates))

required_daily_columns = [
    "user",
    "date",
    "external_email_count",
    "usb_copy_count",
    "www_upload_count",
]

for path in source_candidates:
    print("\n" + "-" * 60)
    print("File:", path.name)
    print("Path:", path.relative_to(SEARCH_ROOT))

    try:
        parquet = pq.ParquetFile(path)
        columns = parquet.schema_arrow.names

        print("Rows:", f"{parquet.metadata.num_rows:,}")
        print("Columns:", len(columns))

        present = [
            col for col in required_daily_columns
            if col in columns
        ]
        missing = [
            col for col in required_daily_columns
            if col not in columns
        ]

        print("Daily count columns present:", present)
        print("Daily count columns missing:", missing)

        additional = [
            col for col in columns
            if any(
                key in col.lower()
                for key in (
                    "baseline", "p50", "p75",
                    "p90", "p95", "mad",
                    "logon", "weekend"
                )
            )
        ]

        if additional:
            print("Relevant reference columns:", additional)

    except Exception as exc:
        print("Metadata inspection error:", repr(exc))

# ------------------------------------------------------------
# STEP 2 — Load existing Notebook 03 frequency references
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("STEP 2 — HISTORICAL BASELINE COVERAGE")
print("=" * 70)

frequency_path = (
    source_dirs["NB03"]
    / "user_channel_frequency_comparison.parquet"
)

freq = pd.read_parquet(frequency_path)

required_reference_columns = [
    "user",
    "channel",
    "baseline_active_days",
    "baseline_p50",
    "baseline_p75",
    "baseline_p90",
    "baseline_p95",
    "baseline_mad",
    "has_baseline_reference",
]

missing_columns = [
    col for col in required_reference_columns
    if col not in freq.columns
]

if missing_columns:
    raise ValueError(
        f"Missing frequency reference columns: {missing_columns}"
    )

if freq.duplicated(["user", "channel"]).any():
    raise ValueError(
        "Duplicate user-channel records detected."
    )

freq["reference_available_check"] = (
    freq["has_baseline_reference"].eq(True)
)

coverage = (
    freq.groupby("channel", dropna=False)
    .agg(
        user_channel_records=("user", "size"),
        users=("user", "nunique"),
        baseline_available=("reference_available_check", "sum"),
    )
)

coverage["baseline_unavailable"] = (
    coverage["user_channel_records"]
    - coverage["baseline_available"]
)

print("\nBaseline coverage by channel:")
print(coverage.to_string())

print("\nTotal user-channel records:", f"{len(freq):,}")
print(
    "Available baseline references:",
    f"{freq['reference_available_check'].sum():,}"
)
print(
    "Unavailable baseline references:",
    f"{(~freq['reference_available_check']).sum():,}"
)

# ------------------------------------------------------------
# STEP 3 — Check reference statistic completeness
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("STEP 3 — BASELINE STATISTIC COMPLETENESS")
print("=" * 70)

stats = [
    "baseline_p50",
    "baseline_p75",
    "baseline_p90",
    "baseline_p95",
    "baseline_mad",
]

available_rows = freq.loc[
    freq["reference_available_check"]
]

print("Rows with available Baseline:", len(available_rows))

print("\nMissing statistics within available references:")
print(available_rows[stats].isna().sum().to_string())

print("\nUser-channel records with no Baseline reference:")
missing_reference = freq.loc[
    ~freq["reference_available_check"],
    ["user", "channel", "baseline_active_days"]
]

print("Count:", len(missing_reference))

if not missing_reference.empty:
    print(missing_reference.head(10).to_string(index=False))

# ------------------------------------------------------------
# STEP 4 — Final summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CELL 1.3 — SUMMARY")
print("=" * 70)

print("Original source candidates:", len(source_candidates))
print("Frequency reference records:", len(freq))
print("Unique reference users:", freq["user"].nunique())
print("Unique channels:", freq["channel"].nunique())

print("\nPASS — Read-only source and reference inspection completed.")
print("No original files were modified or created.")


STEP 1 — ORIGINAL DATA SOURCE DISCOVERY
Search root: C:\Users\hp\Documents\FYP_Data_Exfiltration_Organized\FYP_Data_Exfiltration
Matching Parquet files: 4

------------------------------------------------------------
File: behavior_features_r62_v1.parquet
Path: Data_Engineering\Data_engineering\data\features\behavior_features_r62_v1.parquet
Rows: 387,162
Columns: 85
Daily count columns present: ['user', 'external_email_count', 'usb_copy_count', 'www_upload_count']
Daily count columns missing: ['date']
Relevant reference columns: ['is_weekend', 'logon_count', 'first_logon', 'baseline_workdays', 'email_count_p50', 'email_count_p90', 'email_count_p95', 'email_count_mad', 'email_bytes_p50', 'email_bytes_p90', 'email_bytes_p95', 'email_bytes_mad', 'usb_copy_p50', 'usb_copy_p90', 'usb_copy_p95', 'usb_copy_mad', 'http_upload_p50', 'http_upload_p90', 'http_upload_p95', 'http_upload_mad', 'email_count_above_p95', 'email_bytes_above_p95', 'usb_copy_above_p95', 'http_upload_above_p95', 'email_cou


### 1.4 — Date Key & Baseline Edge-Case Audit

**Objective:** Identify the actual date columns in the daily datasets and examine historical Baseline edge cases among structurally compatible users.

**Expected Outcomes:**
- Identify the daily date keys for future data reconciliation.
- Measure zero or insufficient historical activity by channel.
- Identify zero-P95 and zero-MAD references.
- Determine which edge cases require attention during formula evaluation.

**Constraint:** Read-only analysis. No Baseline recalculation or user exclusion.


In [5]:

# ============================================================
# CELL 1.4 — DATE KEY & BASELINE EDGE-CASE AUDIT
# ============================================================

import pyarrow as pa

# ------------------------------------------------------------
# STEP 1 — Locate daily source files
# ------------------------------------------------------------

print("=" * 70)
print("STEP 1 — DAILY DATE KEY DISCOVERY")
print("=" * 70)

def get_source_file(filename):
    matches = [
        p for p in source_candidates
        if p.name == filename
    ]
    if len(matches) != 1:
        raise RuntimeError(
            f"Expected one file named {filename}; "
            f"found {len(matches)}."
        )
    return matches[0]

FEATURES_PATH = get_source_file(
    "behavior_features_r62_v1.parquet"
)

USER_DAY_PATH = get_source_file(
    "user_day_r62_v1.parquet"
)

BASELINE_PATH = get_source_file(
    "user_baseline_r62_v1.parquet"
)

daily_sources = {
    "Future Behavior Features": FEATURES_PATH,
    "Original User-Day": USER_DAY_PATH,
}

for name, path in daily_sources.items():
    schema = pq.read_schema(path)

    date_candidates = []

    for field in schema:
        col = field.name.lower()

        is_date_type = (
            pa.types.is_date32(field.type)
            or pa.types.is_date64(field.type)
            or pa.types.is_timestamp(field.type)
        )

        is_date_name = (
            col in {
                "date", "day", "event_date",
                "activity_date", "event_day",
                "date_only", "day_date",
                "timestamp", "datetime"
            }
            or col.endswith("_date")
        )

        if is_date_type or is_date_name:
            date_candidates.append(
                (field.name, str(field.type))
            )

    print(f"\n{name}")
    print("File:", path.name)
    print("Possible date columns:")

    for col, dtype in date_candidates:
        print(f" - {col}: {dtype}")

    if not date_candidates:
        print(" - No obvious date column identified.")
        print("   Full column list:", schema.names)

# ------------------------------------------------------------
# STEP 2 — Restrict Baseline audit to NB05 compatible users
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("STEP 2 — COMPATIBLE USER BASELINE COVERAGE")
print("=" * 70)

freq_936 = freq.loc[
    freq["user"].isin(passed_users)
].copy()

assert freq_936["user"].nunique() == 936
assert not freq_936.duplicated(
    ["user", "channel"]
).any()

print("Compatible users:", freq_936["user"].nunique())
print("User-channel records:", len(freq_936))
print(
    "References marked available:",
    int(freq_936["has_baseline_reference"].eq(True).sum())
)

# ------------------------------------------------------------
# STEP 3 — Examine Baseline edge cases by channel
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("STEP 3 — BASELINE EDGE CASES BY CHANNEL")
print("=" * 70)

rows = []

for channel, group in freq_936.groupby("channel"):
    rows.append({
        "channel": channel,
        "records": len(group),
        "zero_baseline_active_days": int(
            group["baseline_active_days"].eq(0).sum()
        ),
        "zero_p95": int(
            group["baseline_p95"].eq(0).sum()
        ),
        "zero_mad": int(
            group["baseline_mad"].eq(0).sum()
        ),
        "p95_missing": int(
            group["baseline_p95"].isna().sum()
        ),
        "mad_missing": int(
            group["baseline_mad"].isna().sum()
        ),
        "median_baseline_active_days": (
            group["baseline_active_days"].median()
        ),
    })

edge_summary = pd.DataFrame(rows)

print(edge_summary.to_string(index=False))

# ------------------------------------------------------------
# STEP 4 — Inspect affected records
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("STEP 4 — BASELINE EDGE-CASE EXAMPLES")
print("=" * 70)

edge_mask = (
    freq_936["baseline_active_days"].eq(0)
    | freq_936["baseline_p95"].eq(0)
    | freq_936["baseline_mad"].eq(0)
)

edge_records = freq_936.loc[
    edge_mask,
    [
        "user", "channel", "future_active_days",
        "baseline_active_days", "baseline_p50",
        "baseline_p95", "baseline_mad"
    ]
]

print("Records with at least one zero condition:",
      f"{len(edge_records):,}")

if not edge_records.empty:
    print("\nSample records:")
    print(edge_records.head(12).to_string(index=False))

# ------------------------------------------------------------
# STEP 5 — Final summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CELL 1.4 — SUMMARY")
print("=" * 70)

print("Compatible users audited: 936")
print("Historical statistics were not recalculated.")
print("No users were excluded.")
print("No source files were modified.")
print("\nPASS — Baseline edge-case inspection completed.")


STEP 1 — DAILY DATE KEY DISCOVERY

Future Behavior Features
File: behavior_features_r62_v1.parquet
Possible date columns:
 - day: date32[day]
 - first_logon: timestamp[us]
 - last_logoff: timestamp[us]

Original User-Day
File: user_day_r62_v1.parquet
Possible date columns:
 - day: date32[day]
 - first_logon: timestamp[us]
 - last_logoff: timestamp[us]

STEP 2 — COMPATIBLE USER BASELINE COVERAGE
Compatible users: 936
User-channel records: 2159
References marked available: 2159

STEP 3 — BASELINE EDGE CASES BY CHANNEL
channel  records  zero_baseline_active_days  zero_p95  zero_mad  p95_missing  mad_missing  median_baseline_active_days
  EMAIL      936                          0         0       343            0            0                        166.0
   HTTP      533                          0         0       243            0            0                         71.0
    USB      690                          0         0       403            0            0                         82.0

S


### 1.5 — Daily Data Reconciliation

**Objective:** Verify that natural Email, USB, and HTTP event counts in the Future behavior features match the original processed User-Day dataset.

**Validation:**
- Verify User–Day keys and eligible Future workdays.
- Reconcile daily records for the 936 structurally compatible users.
- Compare event counts across all three channels.
- Identify missing, duplicated, or inconsistent records.

**Expected Outcome:** A verified and consistent natural daily activity source for formula and multiplier evaluation.

**Constraint:** Read-only validation. No data modification or injection.


In [6]:

# ============================================================
# CELL 1.5 — DAILY DATA RECONCILIATION
# ============================================================

import pandas as pd

KEYS = ["user", "day"]
COUNTS = [
    "external_email_count",
    "usb_copy_count",
    "www_upload_count",
]
COLUMNS = KEYS + COUNTS + ["is_weekend", "logon_count"]

FUTURE_START = pd.Timestamp("2011-01-01")
FUTURE_END = pd.Timestamp("2011-06-01")

# ------------------------------------------------------------
# STEP 1 — Load only required columns
# ------------------------------------------------------------
print("=" * 70)
print("STEP 1 — SOURCE LOADING")
print("=" * 70)

features = pd.read_parquet(
    FEATURES_PATH, columns=COLUMNS
)
original = pd.read_parquet(
    USER_DAY_PATH, columns=COLUMNS
)

for df in (features, original):
    df["day"] = pd.to_datetime(df["day"])

# Restrict to structurally compatible users
features = features[
    features["user"].isin(passed_users)
].copy()

original = original[
    original["user"].isin(passed_users)
    & original["day"].between(FUTURE_START, FUTURE_END)
].copy()

# Eligible Future workdays, as used in previous audits
eligible = (
    original["is_weekend"].eq(0)
    & original["logon_count"].gt(0)
)

original_eligible = original.loc[eligible].copy()

print("Features rows:", f"{len(features):,}")
print("Original Future rows:", f"{len(original):,}")
print("Original eligible rows:", f"{len(original_eligible):,}")

print("Features users:", features["user"].nunique())
print("Original eligible users:",
      original_eligible["user"].nunique())

# ------------------------------------------------------------
# STEP 2 — Key and eligibility validation
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("STEP 2 — KEY & ELIGIBILITY CHECK")
print("=" * 70)

for name, df in [
    ("Features", features),
    ("Original eligible", original_eligible),
]:
    duplicates = int(df.duplicated(KEYS).sum())
    missing_keys = int(df[KEYS].isna().any(axis=1).sum())

    print(f"{name}:")
    print("  Duplicate keys:", duplicates)
    print("  Missing keys:", missing_keys)

    assert duplicates == 0
    assert missing_keys == 0

features_ineligible = (
    ~features["day"].between(FUTURE_START, FUTURE_END)
    | ~features["is_weekend"].eq(0)
    | ~features["logon_count"].gt(0)
)

print("Ineligible Features rows:",
      int(features_ineligible.sum()))

assert not features_ineligible.any(), (
    "Features contain unexpected ineligible user-days."
)

# ------------------------------------------------------------
# STEP 3 — Reconcile User-Day keys and channel counts
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("STEP 3 — DAILY RECORD RECONCILIATION")
print("=" * 70)

comparison = features[KEYS + COUNTS].merge(
    original_eligible[KEYS + COUNTS],
    on=KEYS,
    how="outer",
    suffixes=("_features", "_original"),
    indicator=True,
    validate="one_to_one",
)

print("\nKey reconciliation:")
print(comparison["_merge"].value_counts().to_string())

matched = comparison[
    comparison["_merge"].eq("both")
].copy()

mismatch_counts = {}

for channel in COUNTS:
    left = matched[f"{channel}_features"]
    right = matched[f"{channel}_original"]

    mismatches = ~left.eq(right)
    mismatch_counts[channel] = int(mismatches.sum())

    print(f"\n{channel}")
    print("  Matched records:", f"{len(matched):,}")
    print("  Count mismatches:", mismatch_counts[channel])
    print("  Null values in Features:", int(left.isna().sum()))
    print("  Null values in Original:", int(right.isna().sum()))
    print("  Negative values in Features:", int(left.lt(0).sum()))
    print("  Negative values in Original:", int(right.lt(0).sum()))

# ------------------------------------------------------------
# STEP 4 — Final verification
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("STEP 4 — FINAL VERIFICATION")
print("=" * 70)

missing_from_original = int(
    comparison["_merge"].eq("left_only").sum()
)
missing_from_features = int(
    comparison["_merge"].eq("right_only").sum()
)

print("Keys missing from Original:", missing_from_original)
print("Keys missing from Features:", missing_from_features)
print("Total count mismatches:", sum(mismatch_counts.values()))

issues = (
    missing_from_original
    + missing_from_features
    + sum(mismatch_counts.values())
    + int(
        matched[
            [f"{c}_{s}" for c in COUNTS
             for s in ("features", "original")]
        ].isna().sum().sum()
    )
)

assert features["user"].nunique() == 936
assert original_eligible["user"].nunique() == 936
assert issues == 0, (
    "Daily reconciliation failed. Review differences."
)

print("\nPASS — Natural daily counts match the original User-Day.")
print("No source files were modified or created.")


STEP 1 — SOURCE LOADING
Features rows: 97,298
Original Future rows: 102,764
Original eligible rows: 97,298
Features users: 936
Original eligible users: 936

STEP 2 — KEY & ELIGIBILITY CHECK
Features:
  Duplicate keys: 0
  Missing keys: 0
Original eligible:
  Duplicate keys: 0
  Missing keys: 0
Ineligible Features rows: 0

STEP 3 — DAILY RECORD RECONCILIATION

Key reconciliation:
_merge
both          97298
left_only         0
right_only        0

external_email_count
  Matched records: 97,298
  Count mismatches: 0
  Null values in Features: 0
  Null values in Original: 0
  Negative values in Features: 0
  Negative values in Original: 0

usb_copy_count
  Matched records: 97,298
  Count mismatches: 0
  Null values in Features: 0
  Null values in Original: 0
  Negative values in Features: 0
  Negative values in Original: 0

www_upload_count
  Matched records: 97,298
  Count mismatches: 0
  Null values in Features: 0
  Null values in Original: 0
  Negative values in Features: 0
  Negative v


### 2.1 — Candidate Injection Formula Definition

**Objective:** Define and verify two candidate injection-count formulas:

- **Formula A — Baseline-Based:** `I = max(1, ceil(m × P95))`
- **Formula B — Natural-Activity-Based:** `I = max(1, ceil(m × N))`

Where `N` is the natural daily event count, `P95` is the historical channel P95, and `m` is the multiplier.

**Expected Outcome:** Mathematically valid formulas ready for real-data evaluation.

No formula or multiplier is approved at this stage.


In [7]:

import numpy as np
import pandas as pd

# ============================================================
# CELL 2.1 — CANDIDATE FORMULA DEFINITION
# ============================================================

def formula_A(N, P95, m):
    if m <= 0:
        raise ValueError("Multiplier must be positive.")
    return np.maximum(
        1, np.ceil(m * np.asarray(P95, dtype=float))
    ).astype(np.int64)


def formula_B(N, P95, m):
    if m <= 0:
        raise ValueError("Multiplier must be positive.")
    return np.maximum(
        1, np.ceil(m * np.asarray(N, dtype=float))
    ).astype(np.int64)


FORMULAS = {
    "A": formula_A,
    "B": formula_B,
}

# Illustrative examples only
examples = [
    ("Low activity", 1, 3.0),
    ("Moderate activity", 6, 10.0),
    ("Above historical P95", 20, 10.0),
]

# V1 multipliers used only to demonstrate the formulas
example_multipliers = {
    "High": 0.5,
    "Medium": 1.0,
    "Low": 1.5,
}

rows = []

for case, N, P95 in examples:
    for level, m in example_multipliers.items():
        A = int(formula_A(N, P95, m))
        B = int(formula_B(N, P95, m))

        rows.append({
            "case": case,
            "stealth": level,
            "N": N,
            "P95": P95,
            "m": m,
            "A_added": A,
            "B_added": B,
            "A_final": N + A,
            "B_final": N + B,
        })

formula_examples = pd.DataFrame(rows)

print("=" * 65)
print("CANDIDATE FORMULA COMPARISON")
print("=" * 65)
print(formula_examples.to_string(index=False))

assert (formula_examples[["A_added", "B_added"]] >= 1).all().all()
assert (
    formula_examples["A_final"]
    == formula_examples["N"] + formula_examples["A_added"]
).all()
assert (
    formula_examples["B_final"]
    == formula_examples["N"] + formula_examples["B_added"]
).all()

print("\nPASS — Formula A and B verified.")
print("No formula or multiplier approved.")
print("No files modified.")


CANDIDATE FORMULA COMPARISON
                case stealth  N  P95   m  A_added  B_added  A_final  B_final
        Low activity    High  1  3.0 0.5        2        1        3        2
        Low activity  Medium  1  3.0 1.0        3        1        4        2
        Low activity     Low  1  3.0 1.5        5        2        6        3
   Moderate activity    High  6 10.0 0.5        5        3       11        9
   Moderate activity  Medium  6 10.0 1.0       10        6       16       12
   Moderate activity     Low  6 10.0 1.5       15        9       21       15
Above historical P95    High 20 10.0 0.5        5       10       25       30
Above historical P95  Medium 20 10.0 1.0       10       20       30       40
Above historical P95     Low 20 10.0 1.5       15       30       35       50

PASS — Formula A and B verified.
No formula or multiplier approved.
No files modified.



### 2.2 — Real-Data Formula Input Preparation

**Objective:** Prepare naturally active User–Day–Channel records for the 936 structurally compatible users and link them to existing historical Baseline statistics.

**Expected Outcomes:**
- Identify naturally active Email, USB, and HTTP observations.
- Match each observation with its user-channel Baseline.
- Verify count validity, keys, and reference completeness.
- Establish a common dataset for fair comparison of Formulas A and B.

**Constraint:** Read-only preparation. No injection, Baseline recalculation, or user exclusion.


In [8]:

# ============================================================
# CELL 2.2 — REAL-DATA FORMULA INPUT PREPARATION
# ============================================================

import numpy as np
import pandas as pd

CHANNEL_COLUMNS = {
    "external_email_count": "EMAIL",
    "usb_copy_count": "USB",
    "www_upload_count": "HTTP",
}

# ------------------------------------------------------------
# STEP 1 — Load verified natural Future activity
# ------------------------------------------------------------

print("=" * 70)
print("STEP 1 — NATURAL DAILY ACTIVITY")
print("=" * 70)

daily = pd.read_parquet(
    FEATURES_PATH,
    columns=["user", "day"] + list(CHANNEL_COLUMNS)
)

daily = daily.loc[
    daily["user"].isin(passed_users)
].copy()

daily["day"] = pd.to_datetime(daily["day"])

assert len(daily) == 97_298
assert daily["user"].nunique() == 936
assert not daily.duplicated(["user", "day"]).any()
assert not daily[["user", "day"]].isna().any().any()

count_columns = list(CHANNEL_COLUMNS)

assert daily[count_columns].notna().all().all()
assert daily[count_columns].ge(0).all().all()

print("Compatible users:", daily["user"].nunique())
print("Eligible user-days:", f"{len(daily):,}")

# ------------------------------------------------------------
# STEP 2 — Convert to User-Day-Channel records
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("STEP 2 — ACTIVE USER-DAY-CHANNEL RECORDS")
print("=" * 70)

natural_long = daily.melt(
    id_vars=["user", "day"],
    value_vars=count_columns,
    var_name="source_column",
    value_name="N"
)

natural_long["channel"] = (
    natural_long["source_column"].map(CHANNEL_COLUMNS)
)

# Only naturally active channels are eligible for evaluation
natural_long = natural_long.loc[
    natural_long["N"] > 0,
    ["user", "day", "channel", "N"]
].copy()

assert not natural_long.duplicated(
    ["user", "day", "channel"]
).any()

assert np.isfinite(natural_long["N"]).all()
assert (natural_long["N"] == np.floor(natural_long["N"])).all()

natural_long["N"] = natural_long["N"].astype("int64")

print("Active observations:", f"{len(natural_long):,}")
print("Active users:", natural_long["user"].nunique())

print("\nObservations by channel:")
print(
    natural_long.groupby("channel")
    .agg(
        observations=("N", "size"),
        users=("user", "nunique"),
        median_N=("N", "median"),
        max_N=("N", "max")
    ).to_string()
)

# ------------------------------------------------------------
# STEP 3 — Load existing historical Baseline references
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("STEP 3 — HISTORICAL BASELINE JOIN")
print("=" * 70)

reference_columns = [
    "user",
    "channel",
    "baseline_active_days",
    "baseline_p50",
    "baseline_p75",
    "baseline_p90",
    "baseline_p95",
    "baseline_mad",
    "has_baseline_reference",
]

references = pd.read_parquet(
    source_dirs["NB03"]
    / "user_channel_frequency_comparison.parquet",
    columns=reference_columns
)

assert not references.duplicated(
    ["user", "channel"]
).any()

evaluation_data = natural_long.merge(
    references,
    on=["user", "channel"],
    how="left",
    validate="many_to_one",
    indicator=True
)

missing_references = int(
    evaluation_data["_merge"].ne("both").sum()
)

print("Matched records:",
      f"{(evaluation_data['_merge'] == 'both').sum():,}")
print("Unmatched records:", missing_references)

assert missing_references == 0

required_stats = [
    "baseline_p50",
    "baseline_p75",
    "baseline_p90",
    "baseline_p95",
    "baseline_mad",
]

assert evaluation_data["has_baseline_reference"].eq(True).all()
assert evaluation_data[required_stats].notna().all().all()
assert evaluation_data["baseline_p95"].gt(0).all()
assert evaluation_data["baseline_active_days"].gt(0).all()

evaluation_data = evaluation_data.drop(columns=["_merge"])

# ------------------------------------------------------------
# STEP 4 — Final evaluation dataset verification
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("STEP 4 — FINAL INPUT VERIFICATION")
print("=" * 70)

print("Eligible user-days:", f"{len(daily):,}")
print("Evaluation observations:", f"{len(evaluation_data):,}")
print("Unique users:", evaluation_data["user"].nunique())
print("Unique channels:", evaluation_data["channel"].nunique())
print("Missing references:", missing_references)

print("\nBaseline coverage by channel:")
print(
    evaluation_data.groupby("channel")
    .agg(
        observations=("N", "size"),
        users=("user", "nunique"),
        median_N=("N", "median"),
        median_P95=("baseline_p95", "median"),
        zero_MAD_observations=(
            "baseline_mad",
            lambda x: int(x.eq(0).sum())
        )
    ).to_string()
)

assert evaluation_data["user"].nunique() == 936
assert evaluation_data["N"].gt(0).all()

print("\nPASS — Real-data formula inputs are ready.")
print("No injection performed.")
print("No original files modified or created.")


STEP 1 — NATURAL DAILY ACTIVITY
Compatible users: 936
Eligible user-days: 97,298

STEP 2 — ACTIVE USER-DAY-CHANNEL RECORDS
Active observations: 100,585
Active users: 936

Observations by channel:
         observations  users  median_N  max_N
channel                                      
EMAIL           57815    936       2.0     16
HTTP            13158    533       2.0     18
USB             29612    690       2.0     16

STEP 3 — HISTORICAL BASELINE JOIN
Matched records: 100,585
Unmatched records: 0

STEP 4 — FINAL INPUT VERIFICATION
Eligible user-days: 97,298
Evaluation observations: 100,585
Unique users: 936
Unique channels: 3
Missing references: 0

Baseline coverage by channel:
         observations  users  median_N  median_P95  zero_MAD_observations
channel                                                                  
EMAIL           57815    936       2.0         4.0                  12350
HTTP            13158    533       2.0         4.0                   4976
USB         


### 2.3 — Formula & Multiplier Impact Comparison

**Objective:** Compare Formulas A and B using three candidate multipliers (0.5, 1.0, 1.5) on identical natural User–Day–Channel observations.

**Evaluation:**
- Added-event counts and relative increases.
- Final activity and new historical P95 exceedances.
- Differences between formulas at each multiplier.
- Stealth-level separation and rounding-related ties.
- Results across Email, USB, and HTTP.

**Expected Outcome:** Determine whether formula behavior and daily impact change meaningfully across multipliers.

**Constraint:** Hypothetical daily evaluation only. No final formula, multiplier, or attack schedule is approved.


In [9]:

# ============================================================
# CELL 2.3 — FORMULA & MULTIPLIER IMPACT COMPARISON
# ============================================================

import numpy as np
import pandas as pd

# STEP 1 — Common evaluation inputs
N = evaluation_data["N"].to_numpy(dtype=np.int64)
P95 = evaluation_data["baseline_p95"].to_numpy(dtype=float)
CHANNELS = evaluation_data["channel"].to_numpy()

MULTIPLIERS = {
    "High": 0.5,
    "Medium": 1.0,
    "Low": 1.5,
}

assert set(FORMULAS) == {"A", "B"}
assert len(N) == 100_585
assert np.all(N > 0)
assert np.all(np.isfinite(P95) & (P95 > 0))

print("=" * 70)
print("STEP 1 — EVALUATION SETTINGS")
print("=" * 70)
print("Observations:", f"{len(N):,}")
print("Users:", evaluation_data["user"].nunique())
print("Formulas: A, B")
print("Multipliers:", MULTIPLIERS)

# STEP 2 — Evaluate six combinations
results = []
additions = {}

for formula_name, formula_fn in FORMULAS.items():
    for level, m in MULTIPLIERS.items():

        I = formula_fn(N, P95, m)
        F = N + I

        additions[(formula_name, level)] = I

        assert np.all(I >= 1)
        assert np.all(F == N + I)

        for channel in ["ALL", "EMAIL", "USB", "HTTP"]:
            mask = (
                np.ones(len(N), dtype=bool)
                if channel == "ALL"
                else CHANNELS == channel
            )

            n = N[mask]
            p = P95[mask]
            added = I[mask]
            final = F[mask]

            natural_above = n > p
            final_above = final > p

            results.append({
                "formula": formula_name,
                "level": level,
                "m": m,
                "channel": channel,
                "observations": int(mask.sum()),
                "total_added": int(added.sum()),
                "median_added": float(np.median(added)),
                "p95_added": float(np.percentile(added, 95)),
                "median_increase_pct": float(
                    np.median(100 * added / n)
                ),
                "median_final_to_P95": float(
                    np.median(final / p)
                ),
                "final_above_P95_pct": float(
                    100 * final_above.mean()
                ),
                "newly_above_P95_pct": float(
                    100 * ((~natural_above) & final_above).mean()
                ),
            })

impact_results = pd.DataFrame(results)

print("\n" + "=" * 70)
print("STEP 2 — OVERALL DAILY IMPACT")
print("=" * 70)

overall = impact_results.loc[
    impact_results["channel"] == "ALL"
].copy()

print(overall.round(2).to_string(index=False))

print("\n" + "=" * 70)
print("STEP 3 — IMPACT BY CHANNEL")
print("=" * 70)

by_channel = impact_results.loc[
    impact_results["channel"] != "ALL"
].copy()

print(by_channel.round(2).to_string(index=False))

# STEP 4 — Compare formulas at each multiplier
print("\n" + "=" * 70)
print("STEP 4 — FORMULA A VS B")
print("=" * 70)

formula_comparison_rows = []

for level, m in MULTIPLIERS.items():

    A = additions[("A", level)]
    B = additions[("B", level)]

    formula_comparison_rows.append({
        "level": level,
        "m": m,
        "A_more_pct": float(100 * (A > B).mean()),
        "B_more_pct": float(100 * (B > A).mean()),
        "equal_pct": float(100 * (A == B).mean()),
        "median_A": float(np.median(A)),
        "median_B": float(np.median(B)),
    })

formula_comparison = pd.DataFrame(formula_comparison_rows)

print(formula_comparison.round(2).to_string(index=False))

# STEP 5 — Test separation between multipliers
print("\n" + "=" * 70)
print("STEP 5 — STEALTH-LEVEL SEPARATION")
print("=" * 70)

separation_rows = []

for formula_name in FORMULAS:

    high = additions[(formula_name, "High")]
    medium = additions[(formula_name, "Medium")]
    low = additions[(formula_name, "Low")]

    assert np.all(high <= medium)
    assert np.all(medium <= low)

    all_equal = (high == medium) & (medium == low)
    strict = (high < medium) & (medium < low)

    for condition, mask in [
        ("ALL", np.ones(len(N), dtype=bool)),
        ("N <= 3", N <= 3),
        ("N > 3", N > 3),
    ]:
        if not mask.any():
            continue

        separation_rows.append({
            "formula": formula_name,
            "condition": condition,
            "observations": int(mask.sum()),
            "all_equal_pct": float(
                100 * all_equal[mask].mean()
            ),
            "strict_separation_pct": float(
                100 * strict[mask].mean()
            ),
            "high_equals_medium_pct": float(
                100 * (high[mask] == medium[mask]).mean()
            ),
            "medium_equals_low_pct": float(
                100 * (medium[mask] == low[mask]).mean()
            ),
        })

stealth_separation = pd.DataFrame(separation_rows)

print(stealth_separation.round(2).to_string(index=False))

# STEP 6 — Validation
print("\n" + "=" * 70)
print("STEP 6 — VALIDATION")
print("=" * 70)

assert len(impact_results) == 24
assert len(formula_comparison) == 3
assert len(stealth_separation) == 6
assert len(additions) == 6

for _, row in formula_comparison.iterrows():
    assert np.isclose(
        row["A_more_pct"]
        + row["B_more_pct"]
        + row["equal_pct"],
        100.0
    )

print("PASS — All six combinations evaluated.")
print("Formula approved: None")
print("Multiplier approved: None")
print("No source files modified or created.")


STEP 1 — EVALUATION SETTINGS
Observations: 100,585
Users: 936
Formulas: A, B
Multipliers: {'High': 0.5, 'Medium': 1.0, 'Low': 1.5}



STEP 2 — OVERALL DAILY IMPACT
formula  level   m channel  observations  total_added  median_added  p95_added  median_increase_pct  median_final_to_P95  final_above_P95_pct  newly_above_P95_pct
      A   High 0.5     ALL        100585       269384           2.0        5.0               100.00                 1.00                41.37                37.71
      A Medium 1.0     ALL        100585       497698           4.0        9.0               200.00                 1.50               100.00                96.34
      A    Low 1.5     ALL        100585       761391           6.0       14.0               300.00                 2.00               100.00                96.34
      B   High 0.5     ALL        100585       162953           1.0        4.0                66.67                 0.75                26.84                23.18
      B Medium 1.0     ALL        100585       263389           2.0        7.0               100.00                 1.00                39.43             


### 3.1 — Multiplier Configuration Sensitivity Analysis

**Objective:** Compare six stealth multiplier configurations using Formulas A and B on identical natural User–Day–Channel observations.

**Tested Configurations:**

| Configuration | High | Medium | Low | Source |
|---|---:|---:|---:|---|
| V1-A | 0.25 | 0.75 | 1.25 | Original V1 |
| V1-B | 0.50 | 1.00 | 1.50 | Original V1 — Previously Selected |
| V1-C | 0.50 | 1.25 | 2.00 | Original V1 |
| Conservative | 0.10 | 0.25 | 0.50 | New |
| Intermediate | 0.25 | 0.50 | 1.00 | New |
| Balanced | 0.25 | 0.50 | 0.75 | New |

**Evaluation:**
- Added-event counts and relative increases.
- Final activity and newly introduced P95 exceedances.
- Differences across Email, USB, and HTTP.
- High/Medium/Low separation and rounding-related ties.
- Effects on low-frequency activity (N <= 3).

**Expected Outcome:** Identify configurations that balance meaningful stealth-level separation with controlled behavioral deviation.

**Constraint:** 36 hypothetical formula–multiplier–level experiments on 100,585 observations. No configuration is approved, and no source data is modified.


In [10]:

# ============================================================
# CELL 3.1 — MULTIPLIER CONFIGURATION SENSITIVITY ANALYSIS
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# STEP 1 — Define all six candidate configurations
# ------------------------------------------------------------

MULTIPLIER_GROUPS = {
    "V1-A": {
        "High": 0.25, "Medium": 0.75, "Low": 1.25
    },
    "V1-B": {
        "High": 0.50, "Medium": 1.00, "Low": 1.50
    },
    "V1-C": {
        "High": 0.50, "Medium": 1.25, "Low": 2.00
    },
    "Conservative": {
        "High": 0.10, "Medium": 0.25, "Low": 0.50
    },
    "Intermediate": {
        "High": 0.25, "Medium": 0.50, "Low": 1.00
    },
    "Balanced": {
        "High": 0.25, "Medium": 0.50, "Low": 0.75
    },
}

LEVELS = ["High", "Medium", "Low"]
CHANNEL_ORDER = ["EMAIL", "USB", "HTTP"]

N = evaluation_data["N"].to_numpy(dtype=np.int64)
P95 = evaluation_data["baseline_p95"].to_numpy(dtype=float)
CHANNELS = evaluation_data["channel"].to_numpy()

assert set(FORMULAS) == {"A", "B"}
assert len(N) == 100_585
assert np.all(N > 0)
assert np.all(np.isfinite(P95) & (P95 > 0))
assert set(np.unique(CHANNELS)) == set(CHANNEL_ORDER)

for group, multipliers in MULTIPLIER_GROUPS.items():
    assert (
        0 < multipliers["High"]
        < multipliers["Medium"]
        < multipliers["Low"]
    ), f"Invalid multiplier order: {group}"

print("=" * 75)
print("STEP 1 — EVALUATION SETTINGS")
print("=" * 75)

print("Users:", evaluation_data["user"].nunique())
print("Active observations:", f"{len(N):,}")
print("Formulas:", list(FORMULAS))
print("Configurations:", len(MULTIPLIER_GROUPS))
print("Total level experiments:", 2 * 6 * 3)

for group, values in MULTIPLIER_GROUPS.items():
    print(f"{group}: {values}")

# ------------------------------------------------------------
# STEP 2 — Evaluate formulas and configurations
# ------------------------------------------------------------

impact_rows = []
separation_rows = []

channel_masks = {
    "ALL": np.ones(len(N), dtype=bool),
    **{
        channel: CHANNELS == channel
        for channel in CHANNEL_ORDER
    }
}

natural_above_p95 = N > P95
low_activity = N <= 3

for formula_name, formula_fn in FORMULAS.items():

    for group_name, multipliers in MULTIPLIER_GROUPS.items():

        additions = {}

        for level in LEVELS:

            m = multipliers[level]

            I = np.asarray(
                formula_fn(N, P95, m),
                dtype=np.int64
            )
            F = N + I

            assert len(I) == len(N)
            assert np.all(I >= 1)
            assert np.all(F == N + I)

            additions[level] = I

            for channel, mask in channel_masks.items():

                n = N[mask]
                p = P95[mask]
                added = I[mask]
                final = F[mask]

                natural_above = natural_above_p95[mask]
                final_above = final > p

                newly_above = (
                    ~natural_above & final_above
                )

                impact_rows.append({
                    "formula": formula_name,
                    "configuration": group_name,
                    "level": level,
                    "m": m,
                    "channel": channel,
                    "observations": int(mask.sum()),
                    "total_added": int(added.sum()),
                    "median_added": float(np.median(added)),
                    "p95_added": float(
                        np.percentile(added, 95)
                    ),
                    "median_increase_pct": float(
                        np.median(100 * added / n)
                    ),
                    "median_final_to_P95": float(
                        np.median(final / p)
                    ),
                    "final_above_P95_pct": float(
                        100 * final_above.mean()
                    ),
                    "newly_above_P95_pct": float(
                        100 * newly_above.mean()
                    ),
                })

        # ----------------------------------------------------
        # STEP 3 — Stealth-level separation
        # ----------------------------------------------------

        high = additions["High"]
        medium = additions["Medium"]
        low = additions["Low"]

        assert np.all(high <= medium)
        assert np.all(medium <= low)

        strict = (high < medium) & (medium < low)
        all_equal = (high == medium) & (medium == low)

        separation_rows.append({
            "formula": formula_name,
            "configuration": group_name,
            "observations": len(N),
            "strict_separation_pct": float(
                100 * strict.mean()
            ),
            "all_equal_pct": float(
                100 * all_equal.mean()
            ),
            "high_equals_medium_pct": float(
                100 * (high == medium).mean()
            ),
            "medium_equals_low_pct": float(
                100 * (medium == low).mean()
            ),
            "strict_when_N_le_3_pct": float(
                100 * strict[low_activity].mean()
            ),
            "all_equal_when_N_le_3_pct": float(
                100 * all_equal[low_activity].mean()
            ),
        })

# ------------------------------------------------------------
# STEP 4 — Overall impact comparison
# ------------------------------------------------------------

multiplier_sensitivity = pd.DataFrame(impact_rows)
stealth_separation = pd.DataFrame(separation_rows)

overall = multiplier_sensitivity.loc[
    multiplier_sensitivity["channel"] == "ALL"
].copy()

print("\n" + "=" * 75)
print("STEP 2 — OVERALL MULTIPLIER IMPACT")
print("=" * 75)

display_columns = [
    "formula",
    "configuration",
    "level",
    "m",
    "median_added",
    "p95_added",
    "median_increase_pct",
    "median_final_to_P95",
    "final_above_P95_pct",
    "newly_above_P95_pct",
]

print(
    overall[display_columns]
    .round(2)
    .to_string(index=False)
)

# ------------------------------------------------------------
# STEP 5 — Channel-specific P95 impact
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("STEP 3 — NEW P95 EXCEEDANCES BY CHANNEL (%)")
print("=" * 75)

channel_results = multiplier_sensitivity.loc[
    multiplier_sensitivity["channel"] != "ALL"
].copy()

channel_p95_comparison = channel_results.pivot(
    index=["formula", "configuration"],
    columns=["channel", "level"],
    values="newly_above_P95_pct"
)

channel_p95_comparison.columns = [
    f"{channel}_{level}"
    for channel, level in channel_p95_comparison.columns
]

ordered_columns = [
    f"{channel}_{level}"
    for channel in CHANNEL_ORDER
    for level in LEVELS
]

print(
    channel_p95_comparison[ordered_columns]
    .round(2)
    .to_string()
)

# ------------------------------------------------------------
# STEP 6 — Stealth-level separation
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("STEP 4 — STEALTH-LEVEL SEPARATION")
print("=" * 75)

print(
    stealth_separation
    .round(2)
    .to_string(index=False)
)

# ------------------------------------------------------------
# STEP 7 — Independent consistency checks
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("STEP 5 — VALIDATION")
print("=" * 75)

assert len(multiplier_sensitivity) == 144
assert len(overall) == 36
assert len(channel_results) == 108
assert len(stealth_separation) == 12

# Check that channel totals equal overall totals
for (formula, config, level), group in (
    multiplier_sensitivity.groupby(
        ["formula", "configuration", "level"]
    )
):
    all_row = group.loc[group["channel"] == "ALL"]
    channel_rows = group.loc[group["channel"] != "ALL"]

    assert len(all_row) == 1
    assert int(channel_rows["observations"].sum()) == len(N)
    assert (
        int(channel_rows["total_added"].sum())
        == int(all_row["total_added"].iloc[0])
    )

# All expected combinations must be present
assert set(overall["formula"]) == {"A", "B"}
assert set(overall["configuration"]) == set(
    MULTIPLIER_GROUPS
)
assert set(overall["level"]) == set(LEVELS)

# Previously tested V1-B coefficients must be reproduced
for formula_name in FORMULAS:
    for level in LEVELS:
        previous = impact_results.loc[
            (impact_results["formula"] == formula_name)
            & (impact_results["level"] == level)
            & (impact_results["channel"] == "ALL")
        ]

        current = overall.loc[
            (overall["formula"] == formula_name)
            & (overall["configuration"] == "V1-B")
            & (overall["level"] == level)
        ]

        assert len(previous) == 1
        assert len(current) == 1

        assert np.isclose(
            previous["newly_above_P95_pct"].iloc[0],
            current["newly_above_P95_pct"].iloc[0]
        )

        assert (
            int(previous["total_added"].iloc[0])
            == int(current["total_added"].iloc[0])
        )

print("Formulas:", 2)
print("Multiplier configurations:", 6)
print("Overall level experiments:", len(overall))
print("Stealth separation comparisons:", len(stealth_separation))
print("V1-B results reproduced: YES")

print("\nPASS — Multiplier sensitivity analysis completed.")
print("Formula approved: None")
print("Configuration approved: None")
print("No source files modified or created.")


STEP 1 — EVALUATION SETTINGS
Users: 936
Active observations: 100,585
Formulas: ['A', 'B']
Configurations: 6
Total level experiments: 36
V1-A: {'High': 0.25, 'Medium': 0.75, 'Low': 1.25}
V1-B: {'High': 0.5, 'Medium': 1.0, 'Low': 1.5}
V1-C: {'High': 0.5, 'Medium': 1.25, 'Low': 2.0}
Conservative: {'High': 0.1, 'Medium': 0.25, 'Low': 0.5}
Intermediate: {'High': 0.25, 'Medium': 0.5, 'Low': 1.0}
Balanced: {'High': 0.25, 'Medium': 0.5, 'Low': 0.75}

STEP 2 — OVERALL MULTIPLIER IMPACT
formula configuration  level    m  median_added  p95_added  median_increase_pct  median_final_to_P95  final_above_P95_pct  newly_above_P95_pct
      A          V1-A   High 0.25           1.0        3.0                66.67                 0.80                19.44                15.78
      A          V1-A Medium 0.75           3.0        7.0               160.00                 1.33                78.11                74.45
      A          V1-A    Low 1.25           5.0       12.0               260.00          


### 3.2 — Low-Activity & Zero-MAD Sensitivity

**Objective:** Investigate how natural event frequency and historical MAD conditions affect stealth-level separation across the six multiplier configurations and Formulas A and B.

**Evaluation:**
- Analyze N = 1, N = 2, N = 3, and N >= 4 separately.
- Compare historical MAD = 0 and MAD > 0.
- Measure strict separation and complete equality between High, Medium, and Low.
- Identify configurations affected by integer rounding.

**Expected Outcome:** Explain why certain configurations fail to distinguish stealth levels, particularly under low-frequency activity.

**Constraint:** Diagnostic analysis only. No formula or multiplier selection, injection, or source modification.


In [11]:

# ============================================================
# CELL 3.2 — LOW-ACTIVITY & ZERO-MAD SENSITIVITY
# ============================================================

import numpy as np
import pandas as pd

N = evaluation_data["N"].to_numpy(dtype=np.int64)
P95 = evaluation_data["baseline_p95"].to_numpy(dtype=float)
MAD = evaluation_data["baseline_mad"].to_numpy(dtype=float)

assert len(N) == 100_585
assert np.all(N > 0)
assert np.all(np.isfinite(P95) & (P95 > 0))
assert np.all(np.isfinite(MAD) & (MAD >= 0))

# ------------------------------------------------------------
# STEP 1 — Define activity conditions
# ------------------------------------------------------------

CONDITIONS = {
    "N1": N == 1,
    "N2": N == 2,
    "N3": N == 3,
    "N4plus": N >= 4,
    "MAD0": MAD == 0,
    "MADplus": MAD > 0,
}

print("=" * 70)
print("STEP 1 — ACTIVITY CONDITION DISTRIBUTION")
print("=" * 70)

for name, mask in CONDITIONS.items():
    print(
        f"{name:8} | Records: {mask.sum():>7,} "
        f"| Share: {100 * mask.mean():6.2f}%"
    )

assert sum(
    CONDITIONS[k].sum()
    for k in ["N1", "N2", "N3", "N4plus"]
) == len(N)

assert (
    CONDITIONS["MAD0"].sum()
    + CONDITIONS["MADplus"].sum()
) == len(N)

# ------------------------------------------------------------
# STEP 2 — Evaluate stealth-level separation
# ------------------------------------------------------------

rows = []

for formula_name, formula_fn in FORMULAS.items():
    for config_name, multipliers in MULTIPLIER_GROUPS.items():

        H = np.asarray(
            formula_fn(N, P95, multipliers["High"])
        )
        M = np.asarray(
            formula_fn(N, P95, multipliers["Medium"])
        )
        L = np.asarray(
            formula_fn(N, P95, multipliers["Low"])
        )

        assert np.all((H >= 1) & (H <= M) & (M <= L))

        strict = (H < M) & (M < L)
        all_equal = (H == M) & (M == L)

        row = {
            "formula": formula_name,
            "configuration": config_name,
            "strict_ALL": 100 * strict.mean(),
            "equal_ALL": 100 * all_equal.mean(),
        }

        for condition, mask in CONDITIONS.items():
            row[f"strict_{condition}"] = (
                100 * strict[mask].mean()
                if mask.any() else np.nan
            )
            row[f"equal_{condition}"] = (
                100 * all_equal[mask].mean()
                if mask.any() else np.nan
            )

        rows.append(row)

edge_sensitivity = pd.DataFrame(rows)

# ------------------------------------------------------------
# STEP 3 — Analyze natural activity counts
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("STEP 2 — STRICT SEPARATION BY NATURAL COUNT (%)")
print("=" * 70)

print(
    edge_sensitivity[
        [
            "formula", "configuration",
            "strict_N1", "strict_N2",
            "strict_N3", "strict_N4plus",
            "strict_ALL",
        ]
    ].round(2).to_string(index=False)
)

print("\n" + "=" * 70)
print("STEP 3 — ALL THREE LEVELS EQUAL (%)")
print("=" * 70)

print(
    edge_sensitivity[
        [
            "formula", "configuration",
            "equal_N1", "equal_N2",
            "equal_N3", "equal_N4plus",
            "equal_ALL",
        ]
    ].round(2).to_string(index=False)
)

# ------------------------------------------------------------
# STEP 4 — Compare historical MAD conditions
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("STEP 4 — ZERO-MAD VS POSITIVE-MAD (%)")
print("=" * 70)

print(
    edge_sensitivity[
        [
            "formula", "configuration",
            "strict_MAD0", "strict_MADplus",
            "equal_MAD0", "equal_MADplus",
        ]
    ].round(2).to_string(index=False)
)

# ------------------------------------------------------------
# STEP 5 — Reconcile with Cell 3.1
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("STEP 5 — VALIDATION")
print("=" * 70)

assert len(edge_sensitivity) == 12

check = edge_sensitivity.merge(
    stealth_separation[
        [
            "formula", "configuration",
            "strict_separation_pct", "all_equal_pct",
        ]
    ],
    on=["formula", "configuration"],
    validate="one_to_one",
)

assert len(check) == 12

assert np.allclose(
    check["strict_ALL"],
    check["strict_separation_pct"]
)

assert np.allclose(
    check["equal_ALL"],
    check["all_equal_pct"]
)

print("Formula-configuration combinations:", len(check))
print("Behavioral conditions:", len(CONDITIONS))
print("Cell 3.1 reconciliation: PASS")
print("No formulas or configurations approved.")
print("No source files modified or created.")

print("\nPASS — Edge-case sensitivity analysis completed.")


STEP 1 — ACTIVITY CONDITION DISTRIBUTION
N1       | Records:  38,547 | Share:  38.32%
N2       | Records:  23,697 | Share:  23.56%
N3       | Records:  14,904 | Share:  14.82%
N4plus   | Records:  23,437 | Share:  23.30%
MAD0     | Records:  27,865 | Share:  27.70%
MADplus  | Records:  72,720 | Share:  72.30%

STEP 2 — STRICT SEPARATION BY NATURAL COUNT (%)
formula configuration  strict_N1  strict_N2  strict_N3  strict_N4plus  strict_ALL
      A          V1-A      94.98      99.91      99.98         100.00       98.05
      A          V1-B      94.99      99.93      99.98         100.00       98.06
      A          V1-C      95.20      99.99     100.00         100.00       98.16
      A  Conservative      22.04      32.27      46.28          80.48       41.66
      A  Intermediate      83.28      94.47      98.91          99.92       92.11
      A      Balanced      81.38      93.29      98.45          99.83       91.01
      B          V1-A       0.00     100.00     100.00         100


### 3.3 — Combined Evidence & Candidate Review

**Objective:** Consolidate the daily-impact and edge-case findings to compare all 12 formula–configuration combinations.

**Evaluation:**
- Median added events at each stealth level.
- Newly introduced P95 exceedances.
- Strict separation and complete level equality.
- Separation for low-frequency activity (N = 1).
- Direct comparison of Formula A versus B within each configuration.

**Expected Outcome:** Establish an evidence-based comparison for prioritizing candidates in subsequent temporal and cross-channel impact tests.

**Constraint:** No automatic winner selection, user exclusion, injection, or source modification.


In [12]:

# ============================================================
# CELL 3.3 — COMBINED EVIDENCE & CANDIDATE REVIEW
# ============================================================

import numpy as np
import pandas as pd

# STEP 1 — Combine daily impact and separation evidence

overall = multiplier_sensitivity.loc[
    multiplier_sensitivity["channel"].eq("ALL")
].copy()

wide = overall.pivot(
    index=["formula", "configuration"],
    columns="level",
    values=["median_added", "newly_above_P95_pct"]
)

wide.columns = [
    f"{metric}_{level}"
    for metric, level in wide.columns
]

daily_evidence = (
    wide.reset_index()
    .merge(
        stealth_separation[
            ["formula", "configuration",
             "strict_separation_pct", "all_equal_pct"]
        ],
        on=["formula", "configuration"],
        validate="one_to_one"
    )
    .merge(
        edge_sensitivity[
            ["formula", "configuration",
             "strict_N1", "strict_N4plus",
             "strict_MAD0", "strict_MADplus"]
        ],
        on=["formula", "configuration"],
        validate="one_to_one"
    )
)

assert len(daily_evidence) == 12
assert not daily_evidence.duplicated(
    ["formula", "configuration"]
).any()

print("=" * 75)
print("STEP 1 — DAILY IMPACT & STEALTH SEPARATION")
print("=" * 75)

display_cols = [
    "formula", "configuration",
    "median_added_High",
    "median_added_Medium",
    "median_added_Low",
    "newly_above_P95_pct_High",
    "newly_above_P95_pct_Medium",
    "newly_above_P95_pct_Low",
    "strict_separation_pct",
    "all_equal_pct",
]

print(
    daily_evidence[display_cols]
    .round(2)
    .to_string(index=False)
)

# STEP 2 — Review low-activity and MAD-related limitations

print("\n" + "=" * 75)
print("STEP 2 — EDGE-CASE COMPARISON")
print("=" * 75)

print(
    daily_evidence[
        ["formula", "configuration",
         "strict_N1", "strict_N4plus",
         "strict_MAD0", "strict_MADplus"]
    ].round(2).to_string(index=False)
)

# STEP 3 — Compare A against B within the same configuration

print("\n" + "=" * 75)
print("STEP 3 — FORMULA A MINUS B (PERCENTAGE POINTS)")
print("=" * 75)

A = daily_evidence.loc[
    daily_evidence["formula"].eq("A")
].set_index("configuration")

B = daily_evidence.loc[
    daily_evidence["formula"].eq("B")
].set_index("configuration")

assert set(A.index) == set(B.index)

difference_rows = []

for config in MULTIPLIER_GROUPS:
    row = {"configuration": config}

    for level in ["High", "Medium", "Low"]:
        column = f"newly_above_P95_pct_{level}"

        row[f"new_P95_diff_{level}_pp"] = (
            A.loc[config, column] - B.loc[config, column]
        )

    row["strict_separation_diff_pp"] = (
        A.loc[config, "strict_separation_pct"]
        - B.loc[config, "strict_separation_pct"]
    )

    difference_rows.append(row)

formula_tradeoffs = pd.DataFrame(difference_rows)

print(
    formula_tradeoffs.round(2).to_string(index=False)
)

print(
    "\nPositive P95 difference = A causes more new P95 "
    "exceedances than B."
)
print(
    "Positive separation difference = A has stronger "
    "daily strict separation than B."
)

# STEP 4 — Verify completeness and consistency

print("\n" + "=" * 75)
print("STEP 4 — VALIDATION")
print("=" * 75)

assert len(formula_tradeoffs) == 6
assert daily_evidence["formula"].value_counts().to_dict() == {
    "A": 6, "B": 6
}

for metric in [
    "median_added",
    "newly_above_P95_pct"
]:
    H = daily_evidence[f"{metric}_High"]
    M = daily_evidence[f"{metric}_Medium"]
    L = daily_evidence[f"{metric}_Low"]

    assert (H <= M).all()
    assert (M <= L).all()

# Validate that all percentages are between 0 and 100
percentage_columns = [
    "strict_separation_pct",
    "all_equal_pct",
]

assert daily_evidence[percentage_columns].ge(0).all().all()
assert daily_evidence[percentage_columns].le(100).all().all()
assert daily_evidence[percentage_columns].notna().all().all()

print("Formula–configuration combinations: 12")
print("Configurations compared directly: 6")
print("Sources reconciled: Cells 3.1 and 3.2")
print("Candidates excluded: None")
print("Final decision: Not made")

print("\nPASS — Combined evidence review completed.")
print("No source files modified or created.")


STEP 1 — DAILY IMPACT & STEALTH SEPARATION
formula configuration  median_added_High  median_added_Medium  median_added_Low  newly_above_P95_pct_High  newly_above_P95_pct_Medium  newly_above_P95_pct_Low  strict_separation_pct  all_equal_pct
      A      Balanced                1.0                  2.0               3.0                     15.78                       37.71                    74.45                  91.01           1.94
      A  Conservative                1.0                  1.0               2.0                     10.20                       15.78                    37.71                  41.66           7.89
      A  Intermediate                1.0                  2.0               4.0                     15.78                       37.71                    96.34                  92.11           1.84
      A          V1-A                1.0                  3.0               5.0                     15.78                       74.45                    96.34           


### 4.1 — Compatible Window Timeline Reconciliation

**Objective:** Reconcile structurally compatible windows from Notebook 05 with the actual eligible Future user-day timeline.

**Validation:**
- Match each window's start and end dates to actual eligible workdays.
- Verify window duration in eligible workdays.
- Recalculate naturally active channel days within each window.
- Reconcile active days and forced Off-Days with Notebook 05.

**Expected Outcome:** A verified set of 288,486 compatible windows across 936 users, ready for controlled temporal impact evaluation.

**Constraint:** Read-only validation. No attack scheduling, injection, or modification of original datasets.


In [13]:

# ============================================================
# CELL 4.1 — COMPATIBLE WINDOW TIMELINE RECONCILIATION
# ============================================================

import numpy as np
import pandas as pd

print("=" * 70)
print("STEP 1 — LOAD COMPATIBLE WINDOWS")
print("=" * 70)

window_columns = [
    "user", "window_workdays",
    "window_start", "window_end",
    "structural_pass", "available_channel_days",
    "forced_offdays", "proposed_active_days"
]

compatible_windows = pd.read_parquet(
    STRUCTURAL_PATH,
    columns=window_columns
)

compatible_windows = compatible_windows.loc[
    compatible_windows["structural_pass"].eq(True)
].copy()

assert len(compatible_windows) == 288_486
assert compatible_windows["user"].nunique() == 936

for col in ["window_start", "window_end"]:
    compatible_windows[col] = pd.to_datetime(
        compatible_windows[col]
    ).dt.normalize()

print("Compatible windows:", f"{len(compatible_windows):,}")
print("Users:", compatible_windows["user"].nunique())

# ------------------------------------------------------------
# STEP 2 — Build actual eligible-day timeline
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("STEP 2 — BUILD NATURAL USER TIMELINES")
print("=" * 70)

count_cols = [
    "external_email_count",
    "usb_copy_count",
    "www_upload_count"
]

timeline = daily[
    ["user", "day"] + count_cols
].copy()

timeline["day"] = pd.to_datetime(
    timeline["day"]
).dt.normalize()

timeline = timeline.sort_values(
    ["user", "day"]
).reset_index(drop=True)

assert len(timeline) == 97_298
assert not timeline.duplicated(["user", "day"]).any()

timeline["eligible_pos"] = (
    timeline.groupby("user").cumcount()
)

timeline["any_channel_active"] = (
    timeline[count_cols].gt(0).any(axis=1).astype("int64")
)

timeline["cumulative_active"] = (
    timeline.groupby("user")["any_channel_active"].cumsum()
)

# ------------------------------------------------------------
# STEP 3 — Match window boundaries
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("STEP 3 — MATCH WINDOW BOUNDARIES")
print("=" * 70)

start_lookup = timeline[
    ["user", "day", "eligible_pos",
     "any_channel_active", "cumulative_active"]
].rename(columns={
    "day": "window_start",
    "eligible_pos": "start_pos",
    "any_channel_active": "start_active",
    "cumulative_active": "start_cumulative"
})

end_lookup = timeline[
    ["user", "day", "eligible_pos", "cumulative_active"]
].rename(columns={
    "day": "window_end",
    "eligible_pos": "end_pos",
    "cumulative_active": "end_cumulative"
})

window_timeline = compatible_windows.merge(
    start_lookup,
    on=["user", "window_start"],
    how="left",
    validate="many_to_one"
).merge(
    end_lookup,
    on=["user", "window_end"],
    how="left",
    validate="many_to_one"
)

boundary_missing = int(
    window_timeline[
        ["start_pos", "end_pos"]
    ].isna().any(axis=1).sum()
)

print("Unmatched window boundaries:", boundary_missing)

assert boundary_missing == 0

# ------------------------------------------------------------
# STEP 4 — Verify durations and channel-active days
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("STEP 4 — WINDOW RECONCILIATION")
print("=" * 70)

window_timeline["observed_duration"] = (
    window_timeline["end_pos"]
    - window_timeline["start_pos"] + 1
)

window_timeline["observed_active_days"] = (
    window_timeline["end_cumulative"]
    - window_timeline["start_cumulative"]
    + window_timeline["start_active"]
)

window_timeline["observed_offdays"] = (
    window_timeline["observed_duration"]
    - window_timeline["observed_active_days"]
)

duration_mismatch = int((
    window_timeline["observed_duration"]
    != window_timeline["window_workdays"]
).sum())

active_mismatch = int((
    window_timeline["observed_active_days"]
    != window_timeline["available_channel_days"]
).sum())

offday_mismatch = int((
    window_timeline["observed_offdays"]
    != window_timeline["forced_offdays"]
).sum())

proposed_mismatch = int((
    window_timeline["observed_active_days"]
    != window_timeline["proposed_active_days"]
).sum())

print("Duration mismatches:", duration_mismatch)
print("Active-day mismatches:", active_mismatch)
print("Forced Off-Day mismatches:", offday_mismatch)
print("Proposed active-day mismatches:", proposed_mismatch)

assert duration_mismatch == 0
assert active_mismatch == 0
assert offday_mismatch == 0
assert proposed_mismatch == 0

# ------------------------------------------------------------
# STEP 5 — Summarize compatible windows
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("STEP 5 — COMPATIBLE WINDOW SUMMARY")
print("=" * 70)

window_summary = (
    window_timeline.groupby("window_workdays")
    .agg(
        windows=("user", "size"),
        users=("user", "nunique"),
        median_active_days=("observed_active_days", "median"),
        median_forced_offdays=("observed_offdays", "median")
    )
    .reset_index()
)

print(window_summary.to_string(index=False))

assert int(window_summary["windows"].sum()) == 288_486

print("\n" + "=" * 70)
print("CELL 4.1 — VALIDATION")
print("=" * 70)

print("Compatible windows:", f"{len(window_timeline):,}")
print("Compatible users:", window_timeline["user"].nunique())
print("Timeline reconciliation: PASS")
print("Attack schedules created: 0")
print("Source files modified: 0")

print("\nPASS — Compatible window timelines verified.")


STEP 1 — LOAD COMPATIBLE WINDOWS
Compatible windows: 288,486
Users: 936

STEP 2 — BUILD NATURAL USER TIMELINES

STEP 3 — MATCH WINDOW BOUNDARIES
Unmatched window boundaries: 0

STEP 4 — WINDOW RECONCILIATION
Duration mismatches: 0
Active-day mismatches: 0
Forced Off-Day mismatches: 0
Proposed active-day mismatches: 0

STEP 5 — COMPATIBLE WINDOW SUMMARY
 window_workdays  windows  users  median_active_days  median_forced_offdays
               5    30803    893                 5.0                    0.0
              10    60011    931                 9.0                    1.0
              20    57812    853                17.0                    3.0
              30    53966    827                25.0                    5.0
              40    46580    789                34.0                    6.0
              50    39314    773                42.0                    8.0

CELL 4.1 — VALIDATION
Compatible windows: 288,486
Compatible users: 936
Timeline reconciliation: PASS
Attack sch


### 4.2 — Cumulative Injection Count Bounds

**Objective:** Evaluate cumulative added-event count ranges across all structurally compatible windows without constructing attack schedules.

**Scope:**
- Six window durations: 5, 10, 20, 30, 40, and 50 eligible workdays.
- All 288,486 structurally compatible windows across 936 users.
- Formulas A and B with all six multiplier configurations.
- Three stealth levels: High, Medium, and Low.

**Method:**
- Calculate the minimum and maximum added-event counts among naturally available channels on each active day.
- Assume one naturally active channel per day for this diagnostic.
- Aggregate daily bounds across each window.
- Maintain zero injection on forced Off-Days.
- Compare cumulative counts across formulas, multipliers, and window durations.
- Examine duration-specific results for A–Balanced, B–V1-A, and B–Balanced.

**Expected Outcome:** Quantify how cumulative added-event counts vary with formula, multiplier, channel availability, and window duration.

**Limitation:** The bounds do not enforce switching requirements and therefore do not represent validated attack schedules or confirmed stealth.

**Constraint:** Read-only analysis. No user selection, attack scheduling, injection, or source modification.


In [14]:

# ============================================================
# CELL 4.2 — CUMULATIVE INJECTION COUNT BOUNDS
# ALL SIX WINDOW DURATIONS: 5, 10, 20, 30, 40, 50
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# STEP 1 — Define evaluation scope
# ------------------------------------------------------------

print("=" * 75)
print("STEP 1 — EVALUATION SCOPE")
print("=" * 75)

DURATIONS = [5, 10, 20, 30, 40, 50]
LEVELS = ["High", "Medium", "Low"]

evaluation_windows = window_timeline.loc[
    window_timeline["window_workdays"].isin(DURATIONS)
].copy().reset_index(drop=True)

assert len(evaluation_windows) == 288_486
assert evaluation_windows["user"].nunique() == 936

print("Evaluation windows:", f"{len(evaluation_windows):,}")
print("Users:", evaluation_windows["user"].nunique())
print("Window durations:", DURATIONS)
print("Formulas:", list(FORMULAS))
print("Multiplier configurations:", len(MULTIPLIER_GROUPS))
print("Stealth levels:", LEVELS)
print("Total level experiments:",
      len(FORMULAS) * len(MULTIPLIER_GROUPS) * len(LEVELS))

# ------------------------------------------------------------
# STEP 2 — Reconcile natural channel observations
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("STEP 2 — NATURAL OBSERVATION ALIGNMENT")
print("=" * 75)

count_columns = [
    "external_email_count",
    "usb_copy_count",
    "www_upload_count",
]

channel_positions = {
    "EMAIL": 0,
    "USB": 1,
    "HTTP": 2,
}

# Build timeline keys.
timeline_keys = pd.MultiIndex.from_frame(
    timeline[["user", "day"]].assign(
        day=lambda x: pd.to_datetime(x["day"]).dt.normalize()
    )
)

assert timeline_keys.is_unique

# Build observation keys.
observation_keys = pd.MultiIndex.from_frame(
    evaluation_data[["user", "day"]].assign(
        day=lambda x: pd.to_datetime(x["day"]).dt.normalize()
    )
)

observation_idx = timeline_keys.get_indexer(
    observation_keys
)

assert np.all(observation_idx >= 0)

assert not evaluation_data.duplicated(
    ["user", "day", "channel"]
).any()

# Map channel names to count-column positions.
channel_idx = (
    evaluation_data["channel"]
    .astype(str)
    .str.upper()
    .map(channel_positions)
)

assert channel_idx.notna().all()

channel_idx = channel_idx.to_numpy(
    dtype=np.int64
)

# Natural event counts.
natural_counts = timeline[
    count_columns
].to_numpy(dtype=np.int64)

N = evaluation_data["N"].to_numpy(
    dtype=np.int64
)

P95 = evaluation_data[
    "baseline_p95"
].to_numpy(dtype=float)

assert np.all(N > 0)
assert np.all(np.isfinite(P95))
assert np.all(P95 > 0)

# Confirm natural counts match their channels.
assert np.array_equal(
    N,
    natural_counts[observation_idx, channel_idx]
)

# Every positive channel count must be represented.
assert len(N) == int(
    (natural_counts > 0).sum()
)

active_day_mask = (
    natural_counts > 0
).any(axis=1)

print("Matched channel observations:", f"{len(N):,}")
print("Naturally active user-days:",
      f"{active_day_mask.sum():,}")
print("Missing observations: 0")
print("Count reconciliation: PASS")

# ------------------------------------------------------------
# STEP 3 — Match all window boundaries
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("STEP 3 — WINDOW BOUNDARY ALIGNMENT")
print("=" * 75)

start_keys = pd.MultiIndex.from_frame(
    evaluation_windows[
        ["user", "window_start"]
    ].rename(columns={"window_start": "day"})
)

end_keys = pd.MultiIndex.from_frame(
    evaluation_windows[
        ["user", "window_end"]
    ].rename(columns={"window_end": "day"})
)

start_idx = timeline_keys.get_indexer(start_keys)
end_idx = timeline_keys.get_indexer(end_keys)

assert np.all(start_idx >= 0)
assert np.all(end_idx >= start_idx)

# Verify that each range belongs to one user.
timeline_users = timeline["user"].to_numpy()
window_users = evaluation_windows["user"].to_numpy()

assert np.array_equal(
    timeline_users[start_idx],
    window_users
)

assert np.array_equal(
    timeline_users[end_idx],
    window_users
)

active_days = evaluation_windows[
    "observed_active_days"
].to_numpy(dtype=np.int64)

assert np.all(active_days > 0)

def cumulative_window_sum(daily_values):
    """
    Calculate cumulative daily counts inside each
    user's window, including both boundary days.
    """

    daily_values = np.asarray(
        daily_values,
        dtype=np.int64
    )

    assert len(daily_values) == len(timeline)

    cumulative = (
        pd.Series(daily_values)
        .groupby(
            timeline["user"],
            sort=False
        )
        .cumsum()
        .to_numpy(dtype=np.int64)
    )

    return (
        cumulative[end_idx]
        - cumulative[start_idx]
        + daily_values[start_idx]
    )

# Independently verify active days in every window.
observed_active = cumulative_window_sum(
    active_day_mask.astype(np.int64)
)

assert np.array_equal(
    observed_active,
    active_days
)

print("Matched windows:", f"{len(evaluation_windows):,}")
print("Active-day reconciliation: PASS")
print("Unmatched boundaries: 0")

# ------------------------------------------------------------
# STEP 4 — Evaluate cumulative added-count bounds
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("STEP 4 — CUMULATIVE ADDED-COUNT BOUNDS")
print("=" * 75)

results = []
duration_results = []

# Focus combinations for detailed duration analysis.
# All 12 combinations are still evaluated overall.
focus_combinations = {
    ("A", "Balanced"),
    ("B", "V1-A"),
    ("B", "Balanced"),
}

window_durations = evaluation_windows[
    "window_workdays"
].to_numpy(dtype=np.int64)

n_days = len(timeline)

for formula_name, formula_fn in FORMULAS.items():

    for config_name, multipliers in MULTIPLIER_GROUPS.items():

        for level in LEVELS:

            m = multipliers[level]

            # Calculate hypothetical added event counts
            # for every naturally active channel.
            added = np.asarray(
                formula_fn(N, P95, m),
                dtype=np.int64
            )

            assert np.all(added >= 1)

            # Minimum and maximum among available
            # channels on the same user-day.
            daily_min = np.full(
                n_days,
                np.iinfo(np.int64).max,
                dtype=np.int64
            )

            daily_max = np.zeros(
                n_days,
                dtype=np.int64
            )

            np.minimum.at(
                daily_min,
                observation_idx,
                added
            )

            np.maximum.at(
                daily_max,
                observation_idx,
                added
            )

            # No channel available = no injection.
            daily_min[~active_day_mask] = 0

            assert np.all(
                daily_min[active_day_mask] >= 1
            )

            assert np.all(
                daily_min <= daily_max
            )

            assert np.all(
                daily_min[~active_day_mask] == 0
            )

            assert np.all(
                daily_max[~active_day_mask] == 0
            )

            # Aggregate daily bounds into each window.
            minimum_added = cumulative_window_sum(
                daily_min
            )

            maximum_added = cumulative_window_sum(
                daily_max
            )

            assert np.all(
                minimum_added >= active_days
            )

            assert np.all(
                maximum_added >= minimum_added
            )

            # Overall results across all six durations.
            results.append({
                "formula": formula_name,
                "configuration": config_name,
                "level": level,
                "m": m,
                "windows": len(evaluation_windows),

                "median_min_added": np.median(
                    minimum_added
                ),

                "median_max_added": np.median(
                    maximum_added
                ),

                "p95_max_added": np.percentile(
                    maximum_added, 95
                ),

                "median_min_per_active_day": np.median(
                    minimum_added / active_days
                ),

                "median_max_per_active_day": np.median(
                    maximum_added / active_days
                ),

                "channel_choice_effect_pct": (
                    100 * np.mean(
                        minimum_added < maximum_added
                    )
                ),
            })

            # Detailed breakdown by all six durations
            # for the three focus combinations.
            if (
                formula_name,
                config_name
            ) in focus_combinations:

                for duration in DURATIONS:

                    mask = (
                        window_durations == duration
                    )

                    duration_results.append({
                        "formula": formula_name,
                        "configuration": config_name,
                        "level": level,
                        "workdays": duration,
                        "windows": int(mask.sum()),

                        "median_min_added": np.median(
                            minimum_added[mask]
                        ),

                        "median_max_added": np.median(
                            maximum_added[mask]
                        ),

                        "median_min_per_active_day": np.median(
                            minimum_added[mask]
                            / active_days[mask]
                        ),

                        "median_max_per_active_day": np.median(
                            maximum_added[mask]
                            / active_days[mask]
                        ),
                    })

cumulative_bounds = pd.DataFrame(results)

cumulative_by_duration = pd.DataFrame(
    duration_results
)

print(
    cumulative_bounds
    .round(2)
    .to_string(index=False)
)

# ------------------------------------------------------------
# STEP 5 — Duration-specific sensitivity
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("STEP 5 — DURATION SENSITIVITY")
print("=" * 75)

print("Focus combinations:")
print("  A — Balanced")
print("  B — V1-A")
print("  B — Balanced")
print()
print("All six durations and three stealth levels:")
print()

cumulative_by_duration = (
    cumulative_by_duration
    .sort_values(
        ["formula", "configuration",
         "level", "workdays"]
    )
    .reset_index(drop=True)
)

print(
    cumulative_by_duration
    .round(2)
    .to_string(index=False)
)

# ------------------------------------------------------------
# STEP 6 — Final validation
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("STEP 6 — VALIDATION")
print("=" * 75)

# 2 formulas × 6 configurations × 3 levels.
assert len(cumulative_bounds) == 36

# 3 focus combinations × 3 levels × 6 durations.
assert len(cumulative_by_duration) == 54

assert not cumulative_bounds.duplicated(
    ["formula", "configuration", "level"]
).any()

assert not cumulative_by_duration.duplicated(
    ["formula", "configuration", "level", "workdays"]
).any()

assert set(
    cumulative_by_duration["workdays"]
) == set(DURATIONS)

# Higher multiplier levels must not produce
# lower cumulative added-event counts.
for _, group in cumulative_bounds.groupby(
    ["formula", "configuration"]
):

    ordered = (
        group.set_index("level")
        .loc[LEVELS]
    )

    for col in [
        "median_min_added",
        "median_max_added",
        "p95_max_added",
    ]:

        assert np.all(
            np.diff(
                ordered[col].to_numpy()
            ) >= 0
        )

# Check bounded percentages.
assert cumulative_bounds[
    "channel_choice_effect_pct"
].between(0, 100).all()

# Check missing values.
assert cumulative_bounds[
    [
        "median_min_added",
        "median_max_added",
        "p95_max_added",
        "median_min_per_active_day",
        "median_max_per_active_day",
    ]
].notna().all().all()

assert cumulative_by_duration[
    [
        "median_min_added",
        "median_max_added",
        "median_min_per_active_day",
        "median_max_per_active_day",
    ]
].notna().all().all()

assert (
    cumulative_bounds["median_min_added"]
    <= cumulative_bounds["median_max_added"]
).all()

print("Window durations:", DURATIONS)
print("Evaluation windows:", f"{len(evaluation_windows):,}")
print("Users:", evaluation_windows["user"].nunique())

print(
    "Formula–configuration–level experiments:",
    len(cumulative_bounds)
)

print(
    "Duration-specific comparisons:",
    len(cumulative_by_duration)
)

print("Off-day injection: 0")
print("Attack schedules created: 0")
print("Switching constraints applied: No")
print("Users selected or excluded: 0")
print("Source files modified or created: 0")

print(
    "\nPASS — Cumulative count bounds evaluated "
    "across all six durations."
)


STEP 1 — EVALUATION SCOPE
Evaluation windows: 288,486
Users: 936
Window durations: [5, 10, 20, 30, 40, 50]
Formulas: ['A', 'B']
Multiplier configurations: 6
Stealth levels: ['High', 'Medium', 'Low']
Total level experiments: 36

STEP 2 — NATURAL OBSERVATION ALIGNMENT
Matched channel observations: 100,585
Naturally active user-days: 74,701
Missing observations: 0
Count reconciliation: PASS

STEP 3 — WINDOW BOUNDARY ALIGNMENT
Matched windows: 288,486
Active-day reconciliation: PASS
Unmatched boundaries: 0

STEP 4 — CUMULATIVE ADDED-COUNT BOUNDS
formula configuration  level    m  windows  median_min_added  median_max_added  p95_max_added  median_min_per_active_day  median_max_per_active_day  channel_choice_effect_pct
      A          V1-A   High 0.25   288486              26.0              30.0           91.0                       1.28                       1.83                      60.65
      A          V1-A Medium 0.75   288486              71.0              80.0          240.0         


### 4.3 — Cumulative Level Separation & Natural Activity Impact

**Objective:** Assess cumulative stealth-level separation and the relative impact of added events across all six window durations.

**Evaluation:**
- Compare cumulative High, Medium, and Low bounds for all 12 formula–configuration combinations.
- Measure strict separation using minimum and maximum cumulative counts.
- Identify windows with non-overlapping level ranges.
- Compare added-event counts against total natural event counts within each window.
- Examine duration-specific results for A–Balanced, B–V1-A, and B–Balanced.

**Expected Outcome:** Determine whether cumulative aggregation improves level differentiation and quantify the relative behavioral impact of hypothetical additions.

**Limitations:** Bounds assume one naturally active channel per active day and do not enforce switching requirements. Non-overlapping ranges are sufficient evidence of separation under these assumptions, not proof of a feasible attack schedule or stealth.

**Constraint:** No attack scheduling, injection, user exclusion, or source modification.


In [15]:

# ============================================================
# CELL 4.3 — CUMULATIVE LEVEL SEPARATION & NATURAL IMPACT
# ============================================================

import numpy as np
import pandas as pd

print("=" * 75)
print("STEP 1 — VERIFY INPUTS AND NATURAL ACTIVITY")
print("=" * 75)

assert len(evaluation_windows) == 288_486
assert len(timeline) == len(natural_counts)
assert len(N) == len(observation_idx)
assert len(start_idx) == len(evaluation_windows)
assert len(end_idx) == len(evaluation_windows)

DURATIONS = [5, 10, 20, 30, 40, 50]
LEVELS = ["High", "Medium", "Low"]

# Total natural event counts across all three channels.
natural_daily_total = natural_counts.sum(
    axis=1
).astype(np.int64)

natural_window_total = cumulative_window_sum(
    natural_daily_total
)

assert np.all(natural_window_total > 0)

window_durations = evaluation_windows[
    "window_workdays"
].to_numpy(dtype=np.int64)

print("Windows:", f"{len(evaluation_windows):,}")
print("Users:", evaluation_windows["user"].nunique())
print("Durations:", DURATIONS)
print("Median natural events per window:",
      np.median(natural_window_total))

# ------------------------------------------------------------
# STEP 2 — Helper for cumulative count bounds
# ------------------------------------------------------------

def compute_cumulative_bounds(formula_fn, multiplier):
    """Calculate one-channel-per-day cumulative bounds."""

    additions = np.asarray(
        formula_fn(N, P95, multiplier),
        dtype=np.int64
    )

    daily_minimum = np.full(
        len(timeline),
        np.iinfo(np.int64).max,
        dtype=np.int64
    )

    daily_maximum = np.zeros(
        len(timeline),
        dtype=np.int64
    )

    np.minimum.at(
        daily_minimum,
        observation_idx,
        additions
    )

    np.maximum.at(
        daily_maximum,
        observation_idx,
        additions
    )

    # Zero additions on days without active channels.
    daily_minimum[~active_day_mask] = 0

    assert np.all(
        daily_minimum[~active_day_mask] == 0
    )
    assert np.all(
        daily_maximum[~active_day_mask] == 0
    )

    minimum = cumulative_window_sum(daily_minimum)
    maximum = cumulative_window_sum(daily_maximum)

    assert np.all(minimum <= maximum)

    return minimum, maximum

# ------------------------------------------------------------
# STEP 3 — Evaluate cumulative level separation
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("STEP 2 — CUMULATIVE STEALTH-LEVEL SEPARATION")
print("=" * 75)

separation_rows = []
duration_rows = []

focus_combinations = {
    ("A", "Balanced"),
    ("B", "V1-A"),
    ("B", "Balanced"),
}

for formula_name, formula_fn in FORMULAS.items():

    for config_name, multipliers in MULTIPLIER_GROUPS.items():

        level_bounds = {}

        for level in LEVELS:
            level_bounds[level] = compute_cumulative_bounds(
                formula_fn,
                multipliers[level]
            )

        H_min, H_max = level_bounds["High"]
        M_min, M_max = level_bounds["Medium"]
        L_min, L_max = level_bounds["Low"]

        # Separation when all levels use their minimum
        # cumulative bound.
        strict_min = (
            (H_min < M_min) &
            (M_min < L_min)
        )

        # Separation when all levels use their maximum
        # cumulative bound.
        strict_max = (
            (H_max < M_max) &
            (M_max < L_max)
        )

        # Stronger condition:
        # all possible cumulative ranges are disjoint
        # in High < Medium < Low order.
        nonoverlap = (
            (H_max < M_min) &
            (M_max < L_min)
        )

        # All-channel natural counts are the denominator.
        relative_impact = {}

        for level in LEVELS:
            minimum, maximum = level_bounds[level]

            relative_impact[level] = {
                "minimum": (
                    100 * minimum / natural_window_total
                ),
                "maximum": (
                    100 * maximum / natural_window_total
                ),
            }

        row = {
            "formula": formula_name,
            "configuration": config_name,
            "windows": len(evaluation_windows),
            "strict_min_pct": 100 * strict_min.mean(),
            "strict_max_pct": 100 * strict_max.mean(),
            "nonoverlap_pct": 100 * nonoverlap.mean(),
        }

        for level in LEVELS:
            row[f"{level}_min_vs_natural_pct"] = np.median(
                relative_impact[level]["minimum"]
            )

            row[f"{level}_max_vs_natural_pct"] = np.median(
                relative_impact[level]["maximum"]
            )

        separation_rows.append(row)

        # Duration-specific analysis for focus combinations.
        if (
            formula_name,
            config_name
        ) in focus_combinations:

            for duration in DURATIONS:
                mask = window_durations == duration

                duration_rows.append({
                    "formula": formula_name,
                    "configuration": config_name,
                    "workdays": duration,
                    "windows": int(mask.sum()),
                    "strict_min_pct": (
                        100 * strict_min[mask].mean()
                    ),
                    "strict_max_pct": (
                        100 * strict_max[mask].mean()
                    ),
                    "nonoverlap_pct": (
                        100 * nonoverlap[mask].mean()
                    ),
                    "medium_min_vs_natural_pct": np.median(
                        relative_impact["Medium"]["minimum"][mask]
                    ),
                    "medium_max_vs_natural_pct": np.median(
                        relative_impact["Medium"]["maximum"][mask]
                    ),
                })

cumulative_separation = pd.DataFrame(
    separation_rows
)

separation_by_duration = pd.DataFrame(
    duration_rows
)

print(
    cumulative_separation[
        [
            "formula",
            "configuration",
            "strict_min_pct",
            "strict_max_pct",
            "nonoverlap_pct",
        ]
    ].round(2).to_string(index=False)
)

# ------------------------------------------------------------
# STEP 4 — Compare additions with natural activity
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("STEP 3 — ADDED EVENTS RELATIVE TO NATURAL ACTIVITY (%)")
print("=" * 75)

print(
    cumulative_separation[
        [
            "formula",
            "configuration",
            "High_min_vs_natural_pct",
            "High_max_vs_natural_pct",
            "Medium_min_vs_natural_pct",
            "Medium_max_vs_natural_pct",
            "Low_min_vs_natural_pct",
            "Low_max_vs_natural_pct",
        ]
    ].round(2).to_string(index=False)
)

print(
    "\nNOTE: Percentages use total natural counts "
    "across Email, USB, and HTTP as the denominator."
)

# ------------------------------------------------------------
# STEP 5 — Examine duration sensitivity
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("STEP 4 — DURATION-SPECIFIC SEPARATION")
print("=" * 75)

separation_by_duration = (
    separation_by_duration
    .sort_values(
        ["formula", "configuration", "workdays"]
    )
    .reset_index(drop=True)
)

print(
    separation_by_duration
    .round(2)
    .to_string(index=False)
)

# ------------------------------------------------------------
# STEP 6 — Validate results
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("STEP 5 — VALIDATION")
print("=" * 75)

assert len(cumulative_separation) == 12
assert len(separation_by_duration) == 18

assert not cumulative_separation.duplicated(
    ["formula", "configuration"]
).any()

assert not separation_by_duration.duplicated(
    ["formula", "configuration", "workdays"]
).any()

# Non-overlapping ranges imply strict separation
# for both minimum and maximum bounds.
assert (
    cumulative_separation["nonoverlap_pct"]
    <= cumulative_separation["strict_min_pct"] + 1e-9
).all()

assert (
    cumulative_separation["nonoverlap_pct"]
    <= cumulative_separation["strict_max_pct"] + 1e-9
).all()

percentage_cols = [
    "strict_min_pct",
    "strict_max_pct",
    "nonoverlap_pct",
]

assert cumulative_separation[
    percentage_cols
].ge(0).all().all()

assert cumulative_separation[
    percentage_cols
].le(100).all().all()

assert cumulative_separation[
    percentage_cols
].notna().all().all()

assert set(
    separation_by_duration["workdays"]
) == set(DURATIONS)

assert cumulative_separation[
    [
        f"{level}_{bound}_vs_natural_pct"
        for level in LEVELS
        for bound in ["min", "max"]
    ]
].notna().all().all()

print("Formula–configuration comparisons: 12")
print("Duration-specific comparisons: 18")
print("Window durations:", DURATIONS)
print("Switching constraints applied: No")
print("Attack schedules created: 0")
print("Users excluded: 0")
print("Source files modified or created: 0")

print(
    "\nPASS — Cumulative separation and natural "
    "activity impact evaluated."
)


STEP 1 — VERIFY INPUTS AND NATURAL ACTIVITY
Windows: 288,486
Users: 936
Durations: [5, 10, 20, 30, 40, 50]
Median natural events per window: 64.0

STEP 2 — CUMULATIVE STEALTH-LEVEL SEPARATION
formula configuration  strict_min_pct  strict_max_pct  nonoverlap_pct
      A          V1-A          100.00          100.00           92.80
      A          V1-B          100.00          100.00           88.46
      A          V1-C          100.00          100.00           88.43
      A  Conservative           63.67           67.71           56.94
      A  Intermediate           99.88           99.96           94.35
      A      Balanced           99.88           99.95           85.61
      B          V1-A           99.56           99.92           85.53
      B          V1-B           99.56           99.92           78.67
      B          V1-C           99.56           99.92           64.89
      B  Conservative           61.27           75.38           52.02
      B  Intermediate           94.99 


### 4.4 — Cumulative P95 Exceedance Exposure

**Objective:** Evaluate how frequently hypothetical added events introduce new historical P95 exceedances within structurally compatible windows.

**Scope:**
- All 288,486 compatible windows across 936 users.
- Six durations: 5, 10, 20, 30, 40, and 50 workdays.
- Both formulas, all six multiplier configurations, and all three stealth levels.

**Method:**
- Identify naturally active channels whose original count is at or below their historical P95.
- Determine whether hypothetical additions push each channel above P95.
- Calculate daily minimum and maximum exceedance indicators across available channels.
- Aggregate these indicators into window-level exceedance-day bounds.
- Compare exceedance frequency and exposure across durations.

**Expected Outcome:** Identify configurations that limit newly introduced P95 exceedance days while maintaining meaningful cumulative stealth-level differences.

**Limitations:** One naturally active channel is assumed per active day. Daily choices are independent and do not enforce switching constraints. P95 exceedances are behavioral indicators, not confirmed detections.

**Constraint:** Diagnostic analysis only. No attack schedules, injection, user exclusion, or source modification.


In [16]:

# ============================================================
# CELL 4.4 — CUMULATIVE P95 EXCEEDANCE EXPOSURE
# ============================================================

import numpy as np
import pandas as pd

DURATIONS = [5, 10, 20, 30, 40, 50]
LEVELS = ["High", "Medium", "Low"]

# ------------------------------------------------------------
# STEP 1 — Verify evaluation inputs
# ------------------------------------------------------------

print("=" * 75)
print("STEP 1 — EVALUATION INPUTS")
print("=" * 75)

assert len(evaluation_windows) == 288_486
assert evaluation_windows["user"].nunique() == 936
assert len(evaluation_data) == 100_585
assert len(observation_idx) == len(evaluation_data)
assert np.all(active_days > 0)
assert np.all(N > 0)
assert np.all(np.isfinite(P95) & (P95 > 0))

print("Windows:", f"{len(evaluation_windows):,}")
print("Users:", evaluation_windows["user"].nunique())
print("Channel observations:", f"{len(N):,}")
print("Durations:", DURATIONS)

natural_above = N > P95

print(
    "Already above P95 before injection:",
    f"{natural_above.sum():,}",
    f"({100 * natural_above.mean():.2f}%)"
)

# ------------------------------------------------------------
# STEP 2 — Evaluate daily and window-level exceedances
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("STEP 2 — WINDOW-LEVEL P95 EXCEEDANCE BOUNDS")
print("=" * 75)

overall_rows = []
duration_rows = []

focus_combinations = {
    ("A", "Balanced"),
    ("B", "V1-A"),
    ("B", "Balanced"),
}

window_durations = evaluation_windows[
    "window_workdays"
].to_numpy(dtype=np.int64)

for formula_name, formula_fn in FORMULAS.items():

    for config_name, multipliers in MULTIPLIER_GROUPS.items():

        previous_min = None
        previous_max = None

        for level in LEVELS:

            m = multipliers[level]

            additions = np.asarray(
                formula_fn(N, P95, m),
                dtype=np.int64
            )

            final_counts = N + additions

            # Count only exceedances introduced by
            # hypothetical added events.
            newly_above = (
                (N <= P95) &
                (final_counts > P95)
            ).astype(np.int64)

            # Minimum and maximum possible exceedance
            # indicator on each naturally active day.
            daily_min = np.ones(
                len(timeline),
                dtype=np.int64
            )

            daily_max = np.zeros(
                len(timeline),
                dtype=np.int64
            )

            np.minimum.at(
                daily_min,
                observation_idx,
                newly_above
            )

            np.maximum.at(
                daily_max,
                observation_idx,
                newly_above
            )

            # No available channel means no injection.
            daily_min[~active_day_mask] = 0

            assert np.all(daily_min <= daily_max)
            assert np.all(daily_min[~active_day_mask] == 0)
            assert np.all(daily_max[~active_day_mask] == 0)

            # Aggregate exceedance-day indicators
            # across every compatible window.
            min_new_days = cumulative_window_sum(
                daily_min
            )

            max_new_days = cumulative_window_sum(
                daily_max
            )

            assert np.all(min_new_days <= max_new_days)
            assert np.all(min_new_days >= 0)
            assert np.all(max_new_days <= active_days)

            # Increasing the multiplier cannot reduce
            # new exceedance-day counts.
            if previous_min is not None:
                assert np.all(min_new_days >= previous_min)
                assert np.all(max_new_days >= previous_max)

            previous_min = min_new_days
            previous_max = max_new_days

            min_share = (
                100 * min_new_days / active_days
            )

            max_share = (
                100 * max_new_days / active_days
            )

            overall_rows.append({
                "formula": formula_name,
                "configuration": config_name,
                "level": level,
                "m": m,
                "windows": len(evaluation_windows),

                "median_min_new_days": np.median(
                    min_new_days
                ),
                "median_max_new_days": np.median(
                    max_new_days
                ),

                "median_min_active_days_pct": np.median(
                    min_share
                ),
                "median_max_active_days_pct": np.median(
                    max_share
                ),

                "unavoidable_any_new_pct": (
                    100 * np.mean(min_new_days > 0)
                ),
                "possible_any_new_pct": (
                    100 * np.mean(max_new_days > 0)
                ),

                "channel_choice_effect_pct": (
                    100 * np.mean(
                        min_new_days < max_new_days
                    )
                ),
            })

            # Detailed duration evaluation for
            # the three previously selected focus groups.
            if (
                formula_name,
                config_name
            ) in focus_combinations:

                for duration in DURATIONS:

                    mask = (
                        window_durations == duration
                    )

                    duration_rows.append({
                        "formula": formula_name,
                        "configuration": config_name,
                        "level": level,
                        "workdays": duration,
                        "windows": int(mask.sum()),

                        "median_min_new_days": np.median(
                            min_new_days[mask]
                        ),
                        "median_max_new_days": np.median(
                            max_new_days[mask]
                        ),

                        "median_min_active_days_pct": np.median(
                            min_share[mask]
                        ),
                        "median_max_active_days_pct": np.median(
                            max_share[mask]
                        ),

                        "unavoidable_any_new_pct": (
                            100 * np.mean(
                                min_new_days[mask] > 0
                            )
                        ),
                    })

window_p95_exceedance = pd.DataFrame(
    overall_rows
)

exceedance_by_duration = pd.DataFrame(
    duration_rows
)

print(
    window_p95_exceedance
    .round(2)
    .to_string(index=False)
)

# ------------------------------------------------------------
# STEP 3 — Duration sensitivity
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("STEP 3 — DURATION SENSITIVITY (MEDIUM LEVEL)")
print("=" * 75)

medium_duration = exceedance_by_duration.loc[
    exceedance_by_duration["level"].eq("Medium")
].sort_values(
    ["formula", "configuration", "workdays"]
)

print(
    medium_duration
    .round(2)
    .to_string(index=False)
)

# ------------------------------------------------------------
# STEP 4 — Validate all results
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("STEP 4 — VALIDATION")
print("=" * 75)

assert len(window_p95_exceedance) == 36
assert len(exceedance_by_duration) == 54

assert not window_p95_exceedance.duplicated(
    ["formula", "configuration", "level"]
).any()

assert not exceedance_by_duration.duplicated(
    ["formula", "configuration", "level", "workdays"]
).any()

assert set(
    exceedance_by_duration["workdays"]
) == set(DURATIONS)

percentage_columns = [
    "median_min_active_days_pct",
    "median_max_active_days_pct",
    "unavoidable_any_new_pct",
    "possible_any_new_pct",
    "channel_choice_effect_pct",
]

assert window_p95_exceedance[
    percentage_columns
].ge(0).all().all()

assert window_p95_exceedance[
    percentage_columns
].le(100).all().all()

assert window_p95_exceedance[
    percentage_columns
].notna().all().all()

assert (
    window_p95_exceedance["unavoidable_any_new_pct"]
    <= window_p95_exceedance["possible_any_new_pct"]
).all()

assert (
    window_p95_exceedance["median_min_new_days"]
    <= window_p95_exceedance["median_max_new_days"]
).all()

print("Formula–configuration–level evaluations: 36")
print("Duration-specific evaluations: 54")
print("Window durations:", DURATIONS)
print("Switching constraints applied: No")
print("Attack schedules created: 0")
print("Injection performed: No")
print("Source files modified or created: 0")

print("\nPASS — Cumulative P95 exposure evaluated.")


STEP 1 — EVALUATION INPUTS
Windows: 288,486
Users: 936
Channel observations: 100,585
Durations: [5, 10, 20, 30, 40, 50]
Already above P95 before injection: 3,682 (3.66%)

STEP 2 — WINDOW-LEVEL P95 EXCEEDANCE BOUNDS
formula configuration  level    m  windows  median_min_new_days  median_max_new_days  median_min_active_days_pct  median_max_active_days_pct  unavoidable_any_new_pct  possible_any_new_pct  channel_choice_effect_pct
      A          V1-A   High 0.25   288486                  1.0                  3.0                        8.00                       17.65                    74.72                 90.48                      73.12
      A          V1-A Medium 0.75   288486                 13.0                 15.0                       62.50                       79.41                    99.85                100.00                      80.03
      A          V1-A    Low 1.25   288486                 19.0                 20.0                       96.55                      100.00


### 4.5 — Final P95 Exposure & Pre-existing Exceedances

**Objective:** Evaluate total final P95 exceedance exposure while distinguishing pre-existing natural exceedances from newly introduced exceedances.

**Scope:**
- All 288,486 compatible windows across 936 users.
- Six durations: 5, 10, 20, 30, 40, and 50 workdays.
- Both formulas, six multiplier configurations, and three stealth levels.

**Method:**
- Identify channel observations already above historical P95.
- Recalculate final P95 exceedances after hypothetical additions.
- Aggregate minimum and maximum final exceedance-day counts across each window.
- Compare final exposure with natural baseline exposure and the new-exceedance results from Cell 4.4.
- Examine duration-specific results for the three focus combinations.

**Expected Outcome:** Determine whether configurations that minimize new exceedances also limit total final P95 exposure.

**Limitations:** One naturally active channel per active day; independent daily channel choices without switching enforcement. P95 exceedances are behavioral indicators, not confirmed detections.

**Constraint:** No attack scheduling, injection, user exclusion, or source modification.


In [17]:

# ============================================================
# CELL 4.5 — FINAL P95 EXPOSURE
# ============================================================

import numpy as np
import pandas as pd

DURATIONS = [5, 10, 20, 30, 40, 50]
LEVELS = ["High", "Medium", "Low"]

# ------------------------------------------------------------
# STEP 1 — Verify inputs and natural exceedances
# ------------------------------------------------------------

print("=" * 75)
print("STEP 1 — NATURAL P95 EXCEEDANCE BASELINE")
print("=" * 75)

assert len(evaluation_windows) == 288_486
assert len(N) == len(P95) == len(observation_idx)
assert len(active_day_mask) == len(timeline)
assert len(active_days) == len(evaluation_windows)

natural_above = N > P95

assert int(natural_above.sum()) == 3_682

n_days = len(timeline)

# Daily bounds for pre-existing exceedances.
natural_daily_min = np.ones(n_days, dtype=np.int64)
natural_daily_max = np.zeros(n_days, dtype=np.int64)

np.minimum.at(
    natural_daily_min,
    observation_idx,
    natural_above.astype(np.int64)
)

np.maximum.at(
    natural_daily_max,
    observation_idx,
    natural_above.astype(np.int64)
)

natural_daily_min[~active_day_mask] = 0

natural_window_min = cumulative_window_sum(
    natural_daily_min
)

natural_window_max = cumulative_window_sum(
    natural_daily_max
)

assert np.all(natural_window_min <= natural_window_max)
assert np.all(natural_window_max <= active_days)

print("Channel observations:", f"{len(N):,}")
print("Naturally above P95:", f"{natural_above.sum():,}")
print("Natural exceedance rate:",
      f"{100 * natural_above.mean():.2f}%")

print("Median minimum natural exceedance days:",
      np.median(natural_window_min))

print("Median maximum natural exceedance days:",
      np.median(natural_window_max))

print("Windows with unavoidable natural exceedance:",
      f"{100 * np.mean(natural_window_min > 0):.2f}%")

print("Windows with possible natural exceedance:",
      f"{100 * np.mean(natural_window_max > 0):.2f}%")

# ------------------------------------------------------------
# STEP 2 — Calculate final exceedance bounds
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("STEP 2 — FINAL P95 EXCEEDANCE EXPOSURE")
print("=" * 75)

final_rows = []
duration_rows = []

focus_combinations = {
    ("A", "Balanced"),
    ("B", "V1-A"),
    ("B", "Balanced"),
}

window_durations = evaluation_windows[
    "window_workdays"
].to_numpy(dtype=np.int64)

for formula_name, formula_fn in FORMULAS.items():

    for config_name, multipliers in MULTIPLIER_GROUPS.items():

        previous_min = None
        previous_max = None

        for level in LEVELS:

            m = multipliers[level]

            additions = np.asarray(
                formula_fn(N, P95, m),
                dtype=np.int64
            )

            final_above = (N + additions) > P95

            newly_above = (
                ~natural_above & final_above
            )

            # Every final exceedance is either
            # pre-existing or newly introduced.
            assert np.array_equal(
                final_above,
                natural_above | newly_above
            )

            # Daily bounds across naturally
            # available channels.
            daily_min = np.ones(
                n_days, dtype=np.int64
            )

            daily_max = np.zeros(
                n_days, dtype=np.int64
            )

            np.minimum.at(
                daily_min,
                observation_idx,
                final_above.astype(np.int64)
            )

            np.maximum.at(
                daily_max,
                observation_idx,
                final_above.astype(np.int64)
            )

            daily_min[~active_day_mask] = 0

            assert np.all(daily_min <= daily_max)
            assert np.all(daily_min >= natural_daily_min)
            assert np.all(daily_max >= natural_daily_max)
            assert np.all(daily_max <= active_day_mask.astype(int))

            # Aggregate into compatible windows.
            final_min_days = cumulative_window_sum(
                daily_min
            )

            final_max_days = cumulative_window_sum(
                daily_max
            )

            assert np.all(final_min_days <= final_max_days)
            assert np.all(final_min_days >= natural_window_min)
            assert np.all(final_max_days >= natural_window_max)
            assert np.all(final_max_days <= active_days)

            if previous_min is not None:
                assert np.all(
                    final_min_days >= previous_min
                )
                assert np.all(
                    final_max_days >= previous_max
                )

            previous_min = final_min_days
            previous_max = final_max_days

            min_share = (
                100 * final_min_days / active_days
            )

            max_share = (
                100 * final_max_days / active_days
            )

            final_rows.append({
                "formula": formula_name,
                "configuration": config_name,
                "level": level,
                "m": m,
                "windows": len(evaluation_windows),

                "final_channel_above_pct":
                    100 * final_above.mean(),

                "new_channel_above_pct":
                    100 * newly_above.mean(),

                "median_min_final_days":
                    np.median(final_min_days),

                "median_max_final_days":
                    np.median(final_max_days),

                "median_min_active_days_pct":
                    np.median(min_share),

                "median_max_active_days_pct":
                    np.median(max_share),

                "unavoidable_any_final_pct":
                    100 * np.mean(final_min_days > 0),

                "possible_any_final_pct":
                    100 * np.mean(final_max_days > 0),
            })

            if (
                formula_name,
                config_name
            ) in focus_combinations:

                for duration in DURATIONS:

                    mask = (
                        window_durations == duration
                    )

                    # Medium-level duration detail.
                    if level == "Medium":

                        duration_rows.append({
                            "formula": formula_name,
                            "configuration": config_name,
                            "workdays": duration,
                            "windows": int(mask.sum()),

                            "median_min_final_days":
                                np.median(final_min_days[mask]),

                            "median_max_final_days":
                                np.median(final_max_days[mask]),

                            "median_min_active_days_pct":
                                np.median(min_share[mask]),

                            "median_max_active_days_pct":
                                np.median(max_share[mask]),

                            "unavoidable_any_final_pct":
                                100 * np.mean(
                                    final_min_days[mask] > 0
                                ),
                        })

final_p95_exposure = pd.DataFrame(final_rows)
final_exposure_by_duration = pd.DataFrame(duration_rows)

print(
    final_p95_exposure
    .round(2)
    .to_string(index=False)
)

# ------------------------------------------------------------
# STEP 3 — Compare with Cell 4.4
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("STEP 3 — NEW VS FINAL P95 EXCEEDANCE DAYS")
print("=" * 75)

comparison = final_p95_exposure.merge(
    window_p95_exceedance[
        [
            "formula", "configuration", "level",
            "median_min_new_days",
            "median_max_new_days",
        ]
    ],
    on=["formula", "configuration", "level"],
    validate="one_to_one"
)

assert len(comparison) == 36

assert (
    comparison["median_min_final_days"]
    >= comparison["median_min_new_days"]
).all()

assert (
    comparison["median_max_final_days"]
    >= comparison["median_max_new_days"]
).all()

print(
    comparison[
        [
            "formula",
            "configuration",
            "level",
            "median_min_new_days",
            "median_min_final_days",
            "median_max_new_days",
            "median_max_final_days",
        ]
    ].round(2).to_string(index=False)
)

# ------------------------------------------------------------
# STEP 4 — Duration sensitivity
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("STEP 4 — DURATION SENSITIVITY (MEDIUM LEVEL)")
print("=" * 75)

final_exposure_by_duration = (
    final_exposure_by_duration
    .sort_values(
        ["formula", "configuration", "workdays"]
    )
    .reset_index(drop=True)
)

print(
    final_exposure_by_duration
    .round(2)
    .to_string(index=False)
)

# ------------------------------------------------------------
# STEP 5 — Final validation
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("STEP 5 — VALIDATION")
print("=" * 75)

assert len(final_p95_exposure) == 36
assert len(final_exposure_by_duration) == 18

assert not final_p95_exposure.duplicated(
    ["formula", "configuration", "level"]
).any()

assert not final_exposure_by_duration.duplicated(
    ["formula", "configuration", "workdays"]
).any()

assert set(
    final_exposure_by_duration["workdays"]
) == set(DURATIONS)

percentage_cols = [
    "final_channel_above_pct",
    "new_channel_above_pct",
    "median_min_active_days_pct",
    "median_max_active_days_pct",
    "unavoidable_any_final_pct",
    "possible_any_final_pct",
]

assert final_p95_exposure[
    percentage_cols
].ge(0).all().all()

assert final_p95_exposure[
    percentage_cols
].le(100).all().all()

assert final_p95_exposure[
    percentage_cols
].notna().all().all()

assert (
    final_p95_exposure["unavoidable_any_final_pct"]
    <= final_p95_exposure["possible_any_final_pct"]
).all()

assert (
    final_p95_exposure["median_min_final_days"]
    <= final_p95_exposure["median_max_final_days"]
).all()

print("Formula–configuration–level evaluations: 36")
print("Duration-specific comparisons: 18")
print("Cell 4.4 reconciliation: PASS")
print("Window durations:", DURATIONS)
print("Switching constraints applied: No")
print("Attack schedules created: 0")
print("Injection performed: No")
print("Source files modified or created: 0")

print("\nPASS — Final P95 exposure evaluated.")


STEP 1 — NATURAL P95 EXCEEDANCE BASELINE
Channel observations: 100,585
Naturally above P95: 3,682
Natural exceedance rate: 3.66%
Median minimum natural exceedance days: 0.0
Median maximum natural exceedance days: 1.0
Windows with unavoidable natural exceedance: 36.87%
Windows with possible natural exceedance: 58.34%

STEP 2 — FINAL P95 EXCEEDANCE EXPOSURE
formula configuration  level    m  windows  final_channel_above_pct  new_channel_above_pct  median_min_final_days  median_max_final_days  median_min_active_days_pct  median_max_active_days_pct  unavoidable_any_final_pct  possible_any_final_pct
      A          V1-A   High 0.25   288486                    19.44                  15.78                    2.0                    4.0                       11.11                       21.62                      81.78                   93.96
      A          V1-A Medium 0.75   288486                    78.11                  74.45                   14.0                   16.0                  


### 4.6 — Controlled Temporal & Cross-Channel Behavioral Impact

**Objective:** Evaluate the behavioral impact of all 12 formula–configuration combinations using an identical, controlled hypothetical injection pattern.

**Evaluation Protocol:**
- Select every third naturally active eligible workday per user.
- Select one naturally active channel per selected day, rotating among available channels.
- Keep the same evaluation days and channels for every formula, configuration, and stealth level.
- Preserve all original data, baseline references, and ineligible days.

**Evaluation:**
1. **Daily P90/P95:** Measure new and final historical threshold exceedances.
2. **Temporal Impact:** Measure added-event increases within rolling 7-, 14-, and 30-calendar-day windows.
3. **Cross-Channel Impact:** Measure changes in Email, USB, and HTTP activity proportions and dominant-channel behavior.

**Expected Outcome:** Identify which combinations produce controlled behavioral changes while maintaining meaningful differences between stealth levels.

**Limitations:** This is one standardized hypothetical exposure pattern, not a final attack schedule. Notebook 05 switching constraints are not enforced in this experiment.

**Constraint:** Read-only analysis. No final attack-user selection, scenario assignment, injection into source datasets, or source modification.


In [18]:

# ============================================================
# CELL 4.6 — CONTROLLED TEMPORAL & CROSS-CHANNEL IMPACT
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

CHANNELS = ["EMAIL", "USB", "HTTP"]
LEVELS = ["High", "Medium", "Low"]
ROLLING_DAYS = [7, 14, 30]

# ------------------------------------------------------------
# STEP 1 — Load complete natural calendar-day activity
# ------------------------------------------------------------

print("=" * 75)
print("STEP 1 — SOURCE DATA & EVALUATION SCOPE")
print("=" * 75)

simulation_root = Path(STRUCTURAL_PATH).resolve().parents[2]

source_path = (
    simulation_root.parent
    / "Data_Engineering"
    / "Data_engineering"
    / "data"
    / "processed"
    / "user_day_r62_v1.parquet"
)

assert source_path.is_file(), (
    f"Original User-Day source not found: {source_path}"
)

count_columns = [
    "external_email_count",
    "usb_copy_count",
    "www_upload_count",
]

future_start = pd.to_datetime(timeline["day"].min())
future_end = pd.to_datetime(timeline["day"].max())
history_start = future_start - pd.Timedelta(days=29)

full_daily = pd.read_parquet(
    source_path,
    columns=["user", "day"] + count_columns
)

full_daily["day"] = pd.to_datetime(
    full_daily["day"]
).dt.normalize()

full_daily = full_daily.loc[
    full_daily["user"].isin(timeline["user"].unique())
    & full_daily["day"].between(history_start, future_end)
].copy()

full_daily = full_daily.sort_values(
    ["user", "day"]
).reset_index(drop=True)

assert not full_daily.duplicated(["user", "day"]).any()
assert full_daily[count_columns].notna().all().all()
assert full_daily[count_columns].ge(0).all().all()

natural_full = full_daily[
    count_columns
].to_numpy(dtype=np.int64)

full_keys = pd.MultiIndex.from_frame(
    full_daily[["user", "day"]]
)

eligible_keys = pd.MultiIndex.from_frame(
    timeline[["user", "day"]]
)

eligible_full_idx = full_keys.get_indexer(eligible_keys)

assert np.all(eligible_full_idx >= 0)
assert np.array_equal(
    natural_full[eligible_full_idx],
    natural_counts
)

print("Original User-Day source: VERIFIED")
print("Calendar-day source rows:", f"{len(full_daily):,}")
print("Eligible Future user-days:", f"{len(timeline):,}")
print("Users:", timeline["user"].nunique())
print("Source count reconciliation: PASS")

# ------------------------------------------------------------
# STEP 2 — Create one fixed hypothetical test pattern
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("STEP 2 — FIXED HYPOTHETICAL EVALUATION PATTERN")
print("=" * 75)

# Number naturally active eligible days independently
# for each user.
active_order = (
    timeline["any_channel_active"]
    .groupby(timeline["user"])
    .cumsum()
    .to_numpy(dtype=np.int64)
)

# First active day, fourth, seventh, etc.
test_mask = (
    active_day_mask
    & ((active_order - 1) % 3 == 0)
)

test_timeline_idx = np.flatnonzero(test_mask)

available = (
    natural_counts[test_timeline_idx] > 0
)

available_count = available.sum(axis=1)

assert np.all(available_count >= 1)

# Rotate among naturally available channels.
test_round = (
    (active_order[test_timeline_idx] - 1) // 3
)

target_position = test_round % available_count

available_rank = np.cumsum(
    available, axis=1
) - 1

chosen = (
    available
    & (available_rank == target_position[:, None])
)

assert np.all(chosen.sum(axis=1) == 1)

chosen_channel_idx = np.argmax(
    chosen, axis=1
)

test_full_idx = eligible_full_idx[
    test_timeline_idx
]

test_N = natural_counts[
    test_timeline_idx,
    chosen_channel_idx
]

assert np.all(test_N > 0)
assert len(np.unique(test_full_idx)) == len(test_full_idx)

# Build per-user-day-channel historical references.
p90_lookup = np.full(
    natural_counts.shape, np.nan
)

p95_lookup = np.full(
    natural_counts.shape, np.nan
)

p90_lookup[observation_idx, channel_idx] = (
    evaluation_data["baseline_p90"].to_numpy(float)
)

p95_lookup[observation_idx, channel_idx] = (
    evaluation_data["baseline_p95"].to_numpy(float)
)

test_P90 = p90_lookup[
    test_timeline_idx,
    chosen_channel_idx
]

test_P95 = p95_lookup[
    test_timeline_idx,
    chosen_channel_idx
]

assert np.all(np.isfinite(test_P90))
assert np.all(np.isfinite(test_P95))
assert np.all(test_P90 <= test_P95)

print("Hypothetical injection days:", f"{len(test_full_idx):,}")
print("Users represented:",
      timeline.iloc[test_timeline_idx]["user"].nunique())

print("\nSelected channels:")

for i, channel in enumerate(CHANNELS):
    print(
        f"{channel}: "
        f"{(chosen_channel_idx == i).sum():,}"
    )

print("\nSame test pattern for all 36 experiments: YES")
print("Injection on ineligible days: NO")

# ------------------------------------------------------------
# STEP 3 — Prepare exact calendar-day rolling windows
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("STEP 3 — ROLLING CALENDAR-DAY REFERENCES")
print("=" * 75)

rolling_starts = {}

for days in ROLLING_DAYS:

    lower_dates = (
        timeline["day"]
        - pd.Timedelta(days=days - 1)
    )

    lower_keys = pd.MultiIndex.from_arrays([
        timeline["user"].to_numpy(),
        lower_dates.to_numpy()
    ])

    rolling_starts[days] = full_keys.searchsorted(
        lower_keys,
        side="left"
    )

    starts = rolling_starts[days]

    assert np.all(starts <= eligible_full_idx)

    assert np.array_equal(
        full_daily["user"].to_numpy()[starts],
        timeline["user"].to_numpy()
    )

def rolling_counts(matrix, days):
    """Inclusive calendar-day totals at eligible Future endpoints."""

    cumulative = np.vstack([
        np.zeros((1, matrix.shape[1]), dtype=np.int64),
        np.cumsum(matrix, axis=0, dtype=np.int64)
    ])

    return (
        cumulative[eligible_full_idx + 1]
        - cumulative[rolling_starts[days]]
    )

natural_rolling = {
    days: rolling_counts(natural_full, days)
    for days in ROLLING_DAYS
}

for days in ROLLING_DAYS:
    natural_total = natural_rolling[days].sum(axis=1)
    print(
        f"{days}-day windows with natural activity:",
        f"{(natural_total > 0).sum():,}"
    )

# ------------------------------------------------------------
# STEP 4 — Evaluate all formulas and multipliers
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("STEP 4 — CONTROLLED BEHAVIORAL IMPACT")
print("=" * 75)

results = []

natural30 = natural_rolling[30]
natural30_total = natural30.sum(axis=1)
valid30 = natural30_total > 0

natural_top = natural30.max(axis=1)
natural_tied = (
    (natural30 == natural_top[:, None]).sum(axis=1) > 1
)

for formula_name, formula_fn in FORMULAS.items():

    for config_name, multipliers in MULTIPLIER_GROUPS.items():

        for level in LEVELS:

            m = multipliers[level]

            added = np.asarray(
                formula_fn(test_N, test_P95, m),
                dtype=np.int64
            )

            assert np.all(added >= 1)

            # Daily P90 / P95 impacts.
            final_test = test_N + added

            new_p90 = (
                (test_N <= test_P90)
                & (final_test > test_P90)
            )

            new_p95 = (
                (test_N <= test_P95)
                & (final_test > test_P95)
            )

            final_p95 = final_test > test_P95

            # Hypothetical additions in memory only.
            added_full = np.zeros_like(
                natural_full, dtype=np.int64
            )

            added_full[
                test_full_idx,
                chosen_channel_idx
            ] = added

            row = {
                "formula": formula_name,
                "configuration": config_name,
                "level": level,
                "m": m,
                "new_P90_pct": 100 * new_p90.mean(),
                "new_P95_pct": 100 * new_p95.mean(),
                "final_P95_pct": 100 * final_p95.mean(),
            }

            for days in ROLLING_DAYS:

                rolled_add = rolling_counts(
                    added_full, days
                ).sum(axis=1)

                rolled_nat = natural_rolling[
                    days
                ].sum(axis=1)

                valid = rolled_nat > 0

                row[f"median_increase_{days}d_pct"] = (
                    np.median(
                        100 * rolled_add[valid]
                        / rolled_nat[valid]
                    )
                )

            # Cross-channel distribution changes,
            # evaluated using the same 30-day windows.
            added30 = rolling_counts(
                added_full, 30
            )

            final30 = natural30 + added30

            natural_share = (
                natural30[valid30]
                / natural30_total[valid30, None]
            )

            final_share = (
                final30[valid30]
                / final30[valid30].sum(axis=1)[:, None]
            )

            # Total variation distance in percentage points.
            share_shift = (
                50 * np.abs(
                    final_share - natural_share
                ).sum(axis=1)
            )

            row["median_share_shift_30d_pp"] = (
                np.median(share_shift)
            )

            row["p95_share_shift_30d_pp"] = (
                np.percentile(share_shift, 95)
            )

            # Dominant-channel changes without tie ambiguity.
            final_top = final30.max(axis=1)

            final_tied = (
                (final30 == final_top[:, None])
                .sum(axis=1) > 1
            )

            valid_dominance = (
                valid30
                & ~natural_tied
                & ~final_tied
            )

            assert valid_dominance.any()

            row["dominant_changed_30d_pct"] = (
                100 * np.mean(
                    np.argmax(
                        natural30[valid_dominance],
                        axis=1
                    )
                    != np.argmax(
                        final30[valid_dominance],
                        axis=1
                    )
                )
            )

            row["dominance_comparisons"] = int(
                valid_dominance.sum()
            )

            results.append(row)

controlled_behavior = pd.DataFrame(results)

print(
    controlled_behavior.drop(
        columns="dominance_comparisons"
    ).round(2).to_string(index=False)
)

# ------------------------------------------------------------
# STEP 5 — Compare focus combinations
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("STEP 5 — FOCUS COMPARISON")
print("=" * 75)

FOCUS = {
    ("B", "Balanced"),
    ("B", "Intermediate"),
    ("B", "V1-A"),
    ("A", "Balanced"),
}

focus_results = controlled_behavior.loc[
    controlled_behavior[
        ["formula", "configuration"]
    ].apply(tuple, axis=1).isin(FOCUS)
]

print(
    focus_results[
        [
            "formula",
            "configuration",
            "level",
            "new_P90_pct",
            "new_P95_pct",
            "median_increase_7d_pct",
            "median_increase_14d_pct",
            "median_increase_30d_pct",
            "median_share_shift_30d_pp",
            "dominant_changed_30d_pct",
        ]
    ].round(2).to_string(index=False)
)

# ------------------------------------------------------------
# STEP 6 — Validation
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("STEP 6 — VALIDATION")
print("=" * 75)

assert len(controlled_behavior) == 36
assert len(focus_results) == 12

assert not controlled_behavior.duplicated(
    ["formula", "configuration", "level"]
).any()

metric_columns = [
    "new_P90_pct",
    "new_P95_pct",
    "final_P95_pct",
    "median_increase_7d_pct",
    "median_increase_14d_pct",
    "median_increase_30d_pct",
    "median_share_shift_30d_pp",
    "p95_share_shift_30d_pp",
    "dominant_changed_30d_pct",
]

assert controlled_behavior[
    metric_columns
].notna().all().all()

assert controlled_behavior[
    metric_columns
].ge(0).all().all()

assert controlled_behavior[
    metric_columns
].le(100).all().all()

assert (
    controlled_behavior["new_P95_pct"]
    <= controlled_behavior["final_P95_pct"]
).all()

for _, group in controlled_behavior.groupby(
    ["formula", "configuration"]
):

    ordered = group.set_index("level").loc[LEVELS]

    for col in [
        "new_P90_pct",
        "new_P95_pct",
        "final_P95_pct",
        "median_increase_7d_pct",
        "median_increase_14d_pct",
        "median_increase_30d_pct",
    ]:
        assert np.all(
            np.diff(ordered[col].to_numpy()) >= -1e-9
        )

print("Formula–configuration–level experiments: 36")
print("Rolling windows: 7, 14, 30 calendar days")
print("Natural baseline history included: YES")
print("Natural off-day activity preserved: YES")
print("Switching constraints enforced: NO")
print("Final attack schedules created: 0")
print("Source files modified or created: 0")

print("\nPASS — Controlled behavioral evaluation completed.")


STEP 1 — SOURCE DATA & EVALUATION SCOPE
Original User-Day source: VERIFIED
Calendar-day source rows: 119,940
Eligible Future user-days: 97,298
Users: 936
Source count reconciliation: PASS

STEP 2 — FIXED HYPOTHETICAL EVALUATION PATTERN
Hypothetical injection days: 25,208
Users represented: 936

Selected channels:
EMAIL: 15,436
USB: 6,834
HTTP: 2,938

Same test pattern for all 36 experiments: YES
Injection on ineligible days: NO

STEP 3 — ROLLING CALENDAR-DAY REFERENCES
7-day windows with natural activity: 96,439
14-day windows with natural activity: 97,202
30-day windows with natural activity: 97,298

STEP 4 — CONTROLLED BEHAVIORAL IMPACT
formula configuration  level    m  new_P90_pct  new_P95_pct  final_P95_pct  median_increase_7d_pct  median_increase_14d_pct  median_increase_30d_pct  median_share_shift_30d_pp  p95_share_shift_30d_pp  dominant_changed_30d_pct
      A          V1-A   High 0.25        22.37        15.67          19.46                   14.29                    14.29    


### 5.1 — Final Evidence Comparison

**Objective:** Consolidate the existing evidence from Sections 3.3, 4.3, 4.5, and 4.6 to support a transparent, data-driven selection of the injection-count formula and stealth multipliers.

**Evaluation Scope:**
- Compare all 12 formula–multiplier configurations.
- Preserve the three stealth levels: High, Medium, and Low.
- Evaluate daily injection-count differences and low-activity sensitivity.
- Assess natural behavioral deviation and newly introduced P95 exceedances.
- Evaluate cumulative injection impact and stealth-level separation.
- Compare temporal effects across 7-, 14-, and 30-calendar-day windows.
- Examine cross-channel behavioral changes.

**Selection Principles:**
1. Favor meaningful differentiation between stealth levels.
2. Avoid excessively small or excessively large behavioral changes.
3. Preserve realistic variation across users and channels.
4. Evaluate P95 exceedances as descriptive evidence, not as an absolute rejection criterion.
5. Consider cumulative and temporal effects rather than daily impact alone.
6. Do not optimize injection rules against MUBSIR model predictions.

**Methodology:**
- Reuse existing validated results without recalculating injection counts.
- Consolidate evidence at the formula–configuration level.
- Retain level-specific measurements.
- Compare trade-offs without assigning arbitrary weighted scores.
- Do not automatically select or exclude any candidate.

**Expected Outcome:** A consolidated evidence table showing the strengths, limitations, and trade-offs of all 12 combinations before final formula and multiplier selection.

**Constraints:**
- Read-only analysis.
- No switching re-evaluation.
- No attack schedules or injection.
- No user exclusions or source modification.
- No final decision in this section.


In [19]:

# ============================================================
# CELL 5.1 — FINAL EVIDENCE COMPARISON
# Read-only | No selection | No file export
# ============================================================

import numpy as np
import pandas as pd
from IPython.display import display

KEYS_51 = ["formula", "configuration"]
LEVELS_51 = ["High", "Medium", "Low"]

# ------------------------------------------------------------
# STEP 1 — Check prerequisites
# ------------------------------------------------------------

required_names = [
    "daily_evidence",          # Section 3.3
    "cumulative_separation",   # Section 4.3
    "final_p95_exposure",      # Section 4.5
    "controlled_behavior",     # Section 4.6
]

missing = [
    name for name in required_names
    if name not in globals()
]

if missing:
    raise RuntimeError(
        "Missing results from earlier notebook cells: "
        + ", ".join(missing)
        + "\nRun all previous code cells (1.1–4.6) "
        "in order, then rerun Cell 5.1."
    )

sources = {
    name: globals()[name]
    for name in required_names
}

for name, df in sources.items():
    if not isinstance(df, pd.DataFrame):
        raise TypeError(
            f"{name} must be a pandas DataFrame."
        )

    if df.empty:
        raise ValueError(f"{name} is empty.")

    required = set(KEYS_51)

    if name in [
        "final_p95_exposure",
        "controlled_behavior",
    ]:
        required.add("level")

    absent = required - set(df.columns)

    if absent:
        raise ValueError(
            f"{name} missing columns: {sorted(absent)}"
        )

daily = daily_evidence.copy()
cumulative = cumulative_separation.copy()
p95 = final_p95_exposure.copy()
temporal = controlled_behavior.copy()

assert len(daily) == 12
assert len(cumulative) == 12
assert len(p95) == 36
assert len(temporal) == 36

assert not daily.duplicated(KEYS_51).any()
assert not cumulative.duplicated(KEYS_51).any()

for df in [p95, temporal]:
    assert not df.duplicated(
        KEYS_51 + ["level"]
    ).any()

expected_keys = set(
    map(tuple, daily[KEYS_51].to_numpy())
)

for name, df in sources.items():
    actual_keys = set(
        map(tuple, df[KEYS_51].to_numpy())
    )

    assert actual_keys == expected_keys, (
        f"Candidate mismatch in {name}"
    )

for df in [p95, temporal]:
    assert set(df["level"]) == set(LEVELS_51)

print("STEP 1 — Input validation: PASS")


# ------------------------------------------------------------
# STEP 2 — Consolidate configuration-level results
# ------------------------------------------------------------

daily_metrics = [
    "strict_separation_pct",
    "all_equal_pct",
    "strict_N1",
    "strict_N4plus",
    "strict_MAD0",
    "strict_MADplus",
]

cumulative_metrics = [
    "strict_min_pct",
    "strict_max_pct",
    "nonoverlap_pct",
]

for level in LEVELS_51:
    daily_metrics += [
        f"median_added_{level}",
        f"newly_above_P95_pct_{level}",
    ]

    cumulative_metrics += [
        f"{level}_min_vs_natural_pct",
        f"{level}_max_vs_natural_pct",
    ]

for name, frame, columns in [
    ("daily_evidence", daily, daily_metrics),
    (
        "cumulative_separation",
        cumulative,
        cumulative_metrics,
    ),
]:
    absent = set(columns) - set(frame.columns)

    if absent:
        raise ValueError(
            f"Missing metrics in {name}: {sorted(absent)}"
        )

evidence_51_summary = daily[
    KEYS_51 + daily_metrics
].merge(
    cumulative[KEYS_51 + cumulative_metrics],
    on=KEYS_51,
    how="inner",
    validate="one_to_one",
)

assert len(evidence_51_summary) == 12

print("STEP 2 — Configuration evidence: PASS")


# ------------------------------------------------------------
# STEP 3 — Consolidate level-specific results
# ------------------------------------------------------------

p95_metrics = [
    "median_min_final_days",
    "median_max_final_days",
]

temporal_metrics = [
    "new_P90_pct",
    "new_P95_pct",
    "final_P95_pct",
    "median_increase_7d_pct",
    "median_increase_14d_pct",
    "median_increase_30d_pct",
    "median_share_shift_30d_pp",
    "p95_share_shift_30d_pp",
    "dominant_changed_30d_pct",
]

for name, frame, columns in [
    ("final_p95_exposure", p95, p95_metrics),
    ("controlled_behavior", temporal, temporal_metrics),
]:
    absent = set(columns) - set(frame.columns)

    if absent:
        raise ValueError(
            f"Missing metrics in {name}: {sorted(absent)}"
        )

evidence_51_levels = temporal[
    KEYS_51 + ["level"] + temporal_metrics
].merge(
    p95[KEYS_51 + ["level"] + p95_metrics],
    on=KEYS_51 + ["level"],
    how="inner",
    validate="one_to_one",
)

daily_level_rows = []

for level in LEVELS_51:
    part = daily[
        KEYS_51
        + [
            f"median_added_{level}",
            f"newly_above_P95_pct_{level}",
        ]
    ].copy()

    part["level"] = level

    part = part.rename(
        columns={
            f"median_added_{level}": "median_added",
            f"newly_above_P95_pct_{level}":
                "daily_new_P95_pct",
        }
    )

    daily_level_rows.append(part)

daily_levels = pd.concat(
    daily_level_rows,
    ignore_index=True,
)

evidence_51_levels = evidence_51_levels.merge(
    daily_levels,
    on=KEYS_51 + ["level"],
    how="inner",
    validate="one_to_one",
)

assert len(evidence_51_levels) == 36
assert not evidence_51_levels.duplicated(
    KEYS_51 + ["level"]
).any()

print("STEP 3 — Level-specific evidence: PASS")


# ------------------------------------------------------------
# STEP 4 — Main comparison of 12 candidates
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("STEP 4 — FINAL EVIDENCE COMPARISON")
print("=" * 75)

main_columns = [
    "formula",
    "configuration",
    "strict_separation_pct",
    "all_equal_pct",
    "nonoverlap_pct",
    "median_added_High",
    "median_added_Medium",
    "median_added_Low",
    "newly_above_P95_pct_High",
    "newly_above_P95_pct_Medium",
    "newly_above_P95_pct_Low",
]

display(
    evidence_51_summary[
        main_columns
    ]
    .sort_values(KEYS_51)
    .round(2)
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# STEP 5 — Temporal and cross-channel impact
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("STEP 5 — TEMPORAL & CROSS-CHANNEL IMPACT")
print("=" * 75)

impact_columns = [
    "formula",
    "configuration",
    "level",
    "median_added",
    "new_P90_pct",
    "new_P95_pct",
    "median_increase_7d_pct",
    "median_increase_14d_pct",
    "median_increase_30d_pct",
    "median_share_shift_30d_pp",
    "dominant_changed_30d_pct",
]

display(
    evidence_51_levels[
        impact_columns
    ]
    .sort_values(
        KEYS_51 + ["level"]
    )
    .round(2)
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# STEP 6 — Low-activity and cumulative assessment
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("STEP 6 — LOW-ACTIVITY SENSITIVITY")
print("=" * 75)

display(
    evidence_51_summary[
        [
            "formula",
            "configuration",
            "strict_N1",
            "strict_N4plus",
            "strict_MAD0",
            "strict_MADplus",
        ]
    ]
    .sort_values(KEYS_51)
    .round(2)
    .reset_index(drop=True)
)

print("\n" + "=" * 75)
print("STEP 6B — CUMULATIVE EFFECT")
print("=" * 75)

display(
    evidence_51_summary[
        KEYS_51 + cumulative_metrics
    ]
    .sort_values(KEYS_51)
    .round(2)
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# STEP 7 — Final consistency validation
# ------------------------------------------------------------

assert len(evidence_51_summary) == 12
assert len(evidence_51_levels) == 36

for metric in [
    "median_added",
    "newly_above_P95_pct",
]:
    H = evidence_51_summary[f"{metric}_High"]
    M = evidence_51_summary[f"{metric}_Medium"]
    L = evidence_51_summary[f"{metric}_Low"]

    assert (H <= M + 1e-9).all()
    assert (M <= L + 1e-9).all()

for _, group in evidence_51_levels.groupby(
    KEYS_51
):
    ordered = group.set_index("level").loc[
        LEVELS_51
    ]

    for metric in [
        "median_added",
        "new_P90_pct",
        "new_P95_pct",
        "final_P95_pct",
    ]:
        values = ordered[metric].to_numpy(
            dtype=float
        )

        assert np.all(
            np.diff(values) >= -1e-9
        ), metric

numeric_columns = evidence_51_summary.select_dtypes(
    include="number"
).columns

assert np.isfinite(
    evidence_51_summary[
        numeric_columns
    ].to_numpy(dtype=float)
).all()

numeric_columns = evidence_51_levels.select_dtypes(
    include="number"
).columns

assert np.isfinite(
    evidence_51_levels[
        numeric_columns
    ].to_numpy(dtype=float)
).all()

print("\n" + "=" * 75)
print("STEP 7 — FINAL VALIDATION")
print("=" * 75)

print("Formula–configuration candidates: 12")
print("Level-specific comparisons: 36")
print("Evidence sources reconciled: 4")
print("Automatic selection: NO")
print("Attack injection: NO")
print("Switching re-evaluation: NO")
print("Files created or modified: NONE")

print("\nPASS — Final evidence comparison completed.")


STEP 1 — Input validation: PASS
STEP 2 — Configuration evidence: PASS
STEP 3 — Level-specific evidence: PASS

STEP 4 — FINAL EVIDENCE COMPARISON


,formula,configuration,strict_separation_pct,all_equal_pct,nonoverlap_pct,median_added_High,median_added_Medium,median_added_Low,newly_above_P95_pct_High,newly_above_P95_pct_Medium,newly_above_P95_pct_Low
0,A,Balanced,91.01,1.94,85.61,1.0,2.0,3.0,15.78,37.71,74.45
1,A,Conservative,41.66,7.89,56.94,1.0,1.0,2.0,10.20,15.78,37.71
2,A,Intermediate,92.11,1.84,94.35,1.0,2.0,4.0,15.78,37.71,96.34
3,A,V1-A,98.05,0.00,92.80,1.0,3.0,5.0,15.78,74.45,96.34
4,A,V1-B,98.06,0.00,88.46,2.0,4.0,6.0,37.71,96.34,96.34
5,A,V1-C,98.16,0.00,88.43,2.0,5.0,8.0,37.71,96.34,96.34
6,B,Balanced,38.12,38.32,72.27,1.0,1.0,2.0,12.63,23.18,33.75
7,B,Conservative,14.54,61.88,52.02,1.0,1.0,1.0,10.16,12.63,23.18
8,B,Intermediate,38.12,38.32,78.97,1.0,1.0,2.0,12.63,23.18,35.77
9,B,V1-A,61.68,0.00,85.53,1.0,2.0,3.0,12.63,33.75,54.89



STEP 5 — TEMPORAL & CROSS-CHANNEL IMPACT


,formula,configuration,level,median_added,new_P90_pct,new_P95_pct,median_increase_7d_pct,median_increase_14d_pct,median_increase_30d_pct,median_share_shift_30d_pp,dominant_changed_30d_pct
0,A,Balanced,High,1.0,22.37,15.67,14.29,14.29,13.89,1.54,1.60
1,A,Balanced,Low,3.0,85.92,73.55,37.50,37.50,36.36,3.17,4.54
2,A,Balanced,Medium,2.0,49.44,36.90,25.00,25.00,24.24,2.34,3.05
3,A,Conservative,High,1.0,14.78,10.31,10.53,10.71,10.45,1.34,1.15
4,A,Conservative,Low,2.0,49.44,36.90,25.00,25.00,24.24,2.34,3.05
5,A,Conservative,Medium,1.0,22.37,15.67,14.29,14.29,13.89,1.54,1.60
6,A,Intermediate,High,1.0,22.37,15.67,14.29,14.29,13.89,1.54,1.60
7,A,Intermediate,Low,4.0,92.78,96.21,45.45,45.95,44.44,3.57,5.27
8,A,Intermediate,Medium,2.0,49.44,36.90,25.00,25.00,24.24,2.34,3.05
9,A,V1-A,High,1.0,22.37,15.67,14.29,14.29,13.89,1.54,1.60



STEP 6 — LOW-ACTIVITY SENSITIVITY


,formula,configuration,strict_N1,strict_N4plus,strict_MAD0,strict_MADplus
0,A,Balanced,81.38,99.83,67.85,99.89
1,A,Conservative,22.04,80.48,0.57,57.40
2,A,Intermediate,83.28,99.92,71.71,99.92
3,A,V1-A,94.98,100.00,92.97,100.00
4,A,V1-B,94.99,100.00,93.00,100.00
5,A,V1-C,95.20,100.00,93.35,100.00
6,B,Balanced,0.00,100.00,12.05,48.11
7,B,Conservative,0.00,62.41,0.86,19.78
8,B,Intermediate,0.00,100.00,12.05,48.11
9,B,V1-A,0.00,100.00,36.50,71.33



STEP 6B — CUMULATIVE EFFECT


,formula,configuration,strict_min_pct,strict_max_pct,nonoverlap_pct,High_min_vs_natural_pct,High_max_vs_natural_pct,Medium_min_vs_natural_pct,Medium_max_vs_natural_pct,Low_min_vs_natural_pct,Low_max_vs_natural_pct
0,A,Balanced,99.88,99.95,85.61,40.00,45.83,72.48,81.82,108.00,121.70
1,A,Conservative,63.67,67.71,56.94,31.43,31.71,40.00,45.83,72.48,81.82
2,A,Intermediate,99.88,99.96,94.35,40.00,45.83,72.48,81.82,133.67,154.55
3,A,V1-A,100.00,100.00,92.80,40.00,45.83,108.00,121.70,173.21,199.27
4,A,V1-B,100.00,100.00,88.46,72.48,81.82,133.67,154.55,205.00,235.05
5,A,V1-C,100.00,100.00,88.43,72.48,81.82,173.21,199.27,266.67,307.69
6,B,Balanced,94.99,98.36,72.27,33.33,35.44,43.48,51.22,61.15,75.90
7,B,Conservative,61.27,75.38,52.02,31.37,31.39,33.33,35.44,43.48,51.22
8,B,Intermediate,94.99,98.36,78.97,33.33,35.44,43.48,51.22,65.71,85.00
9,B,V1-A,99.56,99.92,85.53,33.33,35.44,61.15,75.90,100.00,121.05



STEP 7 — FINAL VALIDATION
Formula–configuration candidates: 12
Level-specific comparisons: 36
Evidence sources reconciled: 4
Automatic selection: NO
Attack injection: NO
Switching re-evaluation: NO
Files created or modified: NONE

PASS — Final evidence comparison completed.



### 5.2 — Controlled Formula & Multiplier Comparison

**Objective:** Fairly evaluate the two injection-count formulas and six multiplier configurations by changing one factor at a time.

**Part A — Fixed Multiplier Configuration**
- Fix each of the six multiplier configurations.
- Compare Formula A against Formula B using identical High, Medium, and Low multipliers.
- Examine stealth-level separation, new P95 exceedances, and temporal impact.

**Part B — Fixed Formula**
- Fix Formula A and compare all six multiplier configurations.
- Repeat independently for Formula B.
- Evaluate level separation, low-activity behavior, P95 exposure, cumulative impact, and temporal effects.

**Evaluation Principles:**
- Preserve the distinction between High, Medium, and Low Stealth.
- Avoid treating minimum P95 exposure as the sole objective.
- Balance realistic behavioral changes with meaningful variation in injection intensity.
- Compare daily and cumulative effects without optimizing for model predictions.
- Interpret descriptive metrics within their original evaluation scope.

**Expected Outcome:** Controlled comparisons that distinguish formula effects from multiplier effects and support a transparent final selection.

**Constraints:**
- Reuse existing validated evidence.
- No injection or switching re-evaluation.
- No source modification or file export.
- No automatic ranking, exclusions, or final selection.


In [20]:

# ============================================================
# CELL 5.2 — CONTROLLED FORMULA & MULTIPLIER COMPARISON
# Read-only | No automatic selection | No file export
# ============================================================

import numpy as np
import pandas as pd
from IPython.display import display

KEYS = ["formula", "configuration"]
LEVELS = ["High", "Medium", "Low"]

CONFIGS = [
    "Conservative",
    "Balanced",
    "Intermediate",
    "V1-A",
    "V1-B",
    "V1-C",
]

# ------------------------------------------------------------
# STEP 1 — Validate existing evidence
# ------------------------------------------------------------

required = [
    "daily_evidence",
    "cumulative_separation",
    "controlled_behavior",
]

missing = [x for x in required if x not in globals()]

if missing:
    raise RuntimeError(
        "Missing notebook variables: "
        + ", ".join(missing)
        + "\nRun previous cells through Section 4.6 first."
    )

daily = daily_evidence.copy()
cumulative = cumulative_separation.copy()
behavior = controlled_behavior.copy()

required_daily = [
    "formula",
    "configuration",
    "strict_separation_pct",
    "all_equal_pct",
    "strict_N1",
    "strict_N4plus",
]

for level in LEVELS:
    required_daily += [
        f"median_added_{level}",
        f"newly_above_P95_pct_{level}",
    ]

required_cumulative = KEYS + [
    "strict_min_pct",
    "strict_max_pct",
    "nonoverlap_pct",
]

required_behavior = KEYS + [
    "level",
    "new_P95_pct",
    "median_increase_7d_pct",
    "median_increase_14d_pct",
    "median_increase_30d_pct",
    "median_share_shift_30d_pp",
    "dominant_changed_30d_pct",
]

for name, df, cols in [
    ("daily_evidence", daily, required_daily),
    ("cumulative_separation", cumulative,
     required_cumulative),
    ("controlled_behavior", behavior,
     required_behavior),
]:
    if not isinstance(df, pd.DataFrame):
        raise TypeError(f"{name} must be a DataFrame")

    absent = set(cols) - set(df.columns)

    if absent:
        raise ValueError(
            f"{name} missing columns: {sorted(absent)}"
        )

assert len(daily) == 12
assert len(cumulative) == 12
assert len(behavior) == 36

assert not daily.duplicated(KEYS).any()
assert not cumulative.duplicated(KEYS).any()
assert not behavior.duplicated(KEYS + ["level"]).any()

expected = {
    (f, c)
    for f in ["A", "B"]
    for c in CONFIGS
}

for name, df in [
    ("daily", daily),
    ("cumulative", cumulative),
    ("behavior", behavior),
]:
    actual = set(map(tuple, df[KEYS].to_numpy()))

    assert actual == expected, (
        f"Candidate mismatch: {name}"
    )

for _, g in behavior.groupby(KEYS):
    assert set(g["level"]) == set(LEVELS)

print("STEP 1 — Evidence validation: PASS")


# ------------------------------------------------------------
# STEP 2 — Build consolidated evidence
# ------------------------------------------------------------

daily_columns = required_daily

comparison = daily[daily_columns].merge(
    cumulative[required_cumulative],
    on=KEYS,
    validate="one_to_one",
)

# Add temporal measurements separately for each level.
behavior_metrics = [
    "new_P95_pct",
    "median_increase_7d_pct",
    "median_increase_14d_pct",
    "median_increase_30d_pct",
    "median_share_shift_30d_pp",
    "dominant_changed_30d_pct",
]

for level in LEVELS:

    subset = behavior.loc[
        behavior["level"].eq(level),
        KEYS + behavior_metrics,
    ].copy()

    subset = subset.rename(
        columns={
            metric: f"{metric}_{level}"
            for metric in behavior_metrics
        }
    )

    comparison = comparison.merge(
        subset,
        on=KEYS,
        validate="one_to_one",
    )

assert len(comparison) == 12
assert not comparison.duplicated(KEYS).any()

print("STEP 2 — Consolidation: PASS")


# ------------------------------------------------------------
# PART A — FIX CONFIGURATION, COMPARE FORMULAS
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("PART A — FIXED MULTIPLIERS: FORMULA A vs B")
print("=" * 75)

# Each row compares A and B under the same multiplier set
# and the same stealth level.

formula_metrics = [
    "median_added",
    "newly_above_P95_pct",
    "new_P95_pct",
    "median_increase_7d_pct",
    "median_increase_14d_pct",
    "median_increase_30d_pct",
    "median_share_shift_30d_pp",
    "dominant_changed_30d_pct",
]

formula_rows = []

for config in CONFIGS:
    for level in LEVELS:

        a = comparison.loc[
            (comparison["formula"] == "A")
            & (comparison["configuration"] == config)
        ].iloc[0]

        b = comparison.loc[
            (comparison["formula"] == "B")
            & (comparison["configuration"] == config)
        ].iloc[0]

        row = {
            "configuration": config,
            "level": level,
        }

        for metric in formula_metrics:

            column = f"{metric}_{level}"

            row[f"{metric}_A"] = a[column]
            row[f"{metric}_B"] = b[column]

            row[f"{metric}_A_minus_B"] = (
                a[column] - b[column]
            )

        formula_rows.append(row)

formula_comparison_52 = pd.DataFrame(formula_rows)

assert len(formula_comparison_52) == 18

# A1 — Daily newly introduced P95 exceedances

print("\nA1 — NEW DAILY P95 EXCEEDANCES")
print("Same multipliers; only the formula changes.")

display(
    formula_comparison_52[
        [
            "configuration",
            "level",
            "newly_above_P95_pct_A",
            "newly_above_P95_pct_B",
            "newly_above_P95_pct_A_minus_B",
        ]
    ].round(2)
)

# A2 — Daily injection counts and temporal effects

print("\nA2 — INJECTION COUNT & 30-DAY IMPACT")

display(
    formula_comparison_52[
        [
            "configuration",
            "level",
            "median_added_A",
            "median_added_B",
            "median_increase_30d_pct_A",
            "median_increase_30d_pct_B",
            "dominant_changed_30d_pct_A",
            "dominant_changed_30d_pct_B",
        ]
    ].round(2)
)

# A3 — Formula effect on level separation

print("\nA3 — STEALTH-LEVEL SEPARATION")

separation_a = comparison.loc[
    comparison["formula"].eq("A"),
    [
        "configuration",
        "strict_separation_pct",
        "all_equal_pct",
        "nonoverlap_pct",
        "strict_N1",
    ],
].set_index("configuration")

separation_b = comparison.loc[
    comparison["formula"].eq("B"),
    [
        "configuration",
        "strict_separation_pct",
        "all_equal_pct",
        "nonoverlap_pct",
        "strict_N1",
    ],
].set_index("configuration")

formula_separation_52 = pd.DataFrame(
    index=CONFIGS
)

for metric in [
    "strict_separation_pct",
    "all_equal_pct",
    "nonoverlap_pct",
    "strict_N1",
]:
    formula_separation_52[f"{metric}_A"] = (
        separation_a[metric]
    )
    formula_separation_52[f"{metric}_B"] = (
        separation_b[metric]
    )
    formula_separation_52[f"{metric}_A_minus_B"] = (
        separation_a[metric] - separation_b[metric]
    )

display(
    formula_separation_52.round(2)
)


# ------------------------------------------------------------
# PART B — FIX FORMULA, COMPARE MULTIPLIERS
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("PART B — FIXED FORMULA: COMPARE SIX CONFIGS")
print("=" * 75)

# Show all configurations for each formula independently.
# No A-vs-B comparison takes place in this part.

config_metrics = [
    "strict_separation_pct",
    "all_equal_pct",
    "strict_N1",
    "strict_N4plus",
    "nonoverlap_pct",
]

for formula in ["A", "B"]:

    subset = comparison.loc[
        comparison["formula"].eq(formula)
    ].set_index("configuration").loc[CONFIGS]

    print("\n" + "-" * 75)
    print(f"FORMULA {formula} — MULTIPLIER COMPARISON")
    print("-" * 75)

    # B1 — Level differentiation and low activity

    print("\nB1 — LEVEL SEPARATION & LOW ACTIVITY")

    display(
        subset[config_metrics]
        .round(2)
        .reset_index()
    )

    # B2 — Daily newly introduced P95 exceedances

    print("\nB2 — NEW DAILY P95 EXCEEDANCES")

    display(
        subset[
            [
                f"newly_above_P95_pct_{level}"
                for level in LEVELS
            ]
        ]
        .round(2)
        .reset_index()
    )

    # B3 — 7/14/30-day effects

    print("\nB3 — TEMPORAL IMPACT")

    temporal_rows = []

    for config in CONFIGS:
        for level in LEVELS:

            r = subset.loc[config]

            temporal_rows.append({
                "configuration": config,
                "level": level,
                "median_added":
                    r[f"median_added_{level}"],
                "increase_7d_pct":
                    r[f"median_increase_7d_pct_{level}"],
                "increase_14d_pct":
                    r[f"median_increase_14d_pct_{level}"],
                "increase_30d_pct":
                    r[f"median_increase_30d_pct_{level}"],
                "channel_share_shift_pp":
                    r[
                        f"median_share_shift_30d_pp_{level}"
                    ],
                "dominant_changed_pct":
                    r[
                        f"dominant_changed_30d_pct_{level}"
                    ],
            })

    temporal_comparison = pd.DataFrame(
        temporal_rows
    )

    display(
        temporal_comparison.round(2)
    )


# ------------------------------------------------------------
# STEP 5 — Validate controlled comparisons
# ------------------------------------------------------------

assert len(formula_comparison_52) == 18

assert set(
    formula_comparison_52["configuration"]
) == set(CONFIGS)

for config in CONFIGS:
    levels = set(
        formula_comparison_52.loc[
            formula_comparison_52[
                "configuration"
            ].eq(config),
            "level",
        ]
    )
    assert levels == set(LEVELS)

# Values must be finite and comparisons must be exact.
for frame in [
    formula_comparison_52,
    formula_separation_52,
    comparison,
]:
    numbers = frame.select_dtypes(
        include="number"
    )

    assert np.isfinite(
        numbers.to_numpy(dtype=float)
    ).all()

print("\n" + "=" * 75)
print("FINAL VALIDATION")
print("=" * 75)

print("Formula comparisons:", len(formula_comparison_52))
print("Multiplier configurations:", len(CONFIGS))
print("Fixed-formula comparisons: 2")
print("Automatic selection: NO")
print("Attack injection: NO")
print("Files created or modified: NONE")

print("\nPASS — Controlled comparison completed.")


STEP 1 — Evidence validation: PASS
STEP 2 — Consolidation: PASS

PART A — FIXED MULTIPLIERS: FORMULA A vs B

A1 — NEW DAILY P95 EXCEEDANCES
Same multipliers; only the formula changes.


,configuration,level,newly_above_P95_pct_A,newly_above_P95_pct_B,newly_above_P95_pct_A_minus_B
0,Conservative,High,10.20,10.16,0.04
1,Conservative,Medium,15.78,12.63,3.14
2,Conservative,Low,37.71,23.18,14.53
3,Balanced,High,15.78,12.63,3.14
4,Balanced,Medium,37.71,23.18,14.53
5,Balanced,Low,74.45,33.75,40.71
6,Intermediate,High,15.78,12.63,3.14
7,Intermediate,Medium,37.71,23.18,14.53
8,Intermediate,Low,96.34,35.77,60.57
9,V1-A,High,15.78,12.63,3.14



A2 — INJECTION COUNT & 30-DAY IMPACT


,configuration,level,median_added_A,median_added_B,median_increase_30d_pct_A,median_increase_30d_pct_B,dominant_changed_30d_pct_A,dominant_changed_30d_pct_B
0,Conservative,High,1.0,1.0,10.45,10.42,1.15,1.18
1,Conservative,Medium,1.0,1.0,13.89,11.43,1.60,1.20
2,Conservative,Low,2.0,1.0,24.24,15.09,3.05,1.63
3,Balanced,High,1.0,1.0,13.89,11.43,1.60,1.20
4,Balanced,Medium,2.0,1.0,24.24,15.09,3.05,1.63
5,Balanced,Low,3.0,2.0,36.36,21.35,4.54,2.63
6,Intermediate,High,1.0,1.0,13.89,11.43,1.60,1.20
7,Intermediate,Medium,2.0,1.0,24.24,15.09,3.05,1.63
8,Intermediate,Low,4.0,2.0,44.44,23.08,5.27,2.84
9,V1-A,High,1.0,1.0,13.89,11.43,1.60,1.20



A3 — STEALTH-LEVEL SEPARATION


,strict_separation_pct_A,strict_separation_pct_B,strict_separation_pct_A_minus_B,all_equal_pct_A,all_equal_pct_B,all_equal_pct_A_minus_B,nonoverlap_pct_A,nonoverlap_pct_B,nonoverlap_pct_A_minus_B,strict_N1_A,strict_N1_B,strict_N1_A_minus_B
Conservative,41.66,14.54,27.12,7.89,61.88,-53.99,56.94,52.02,4.93,22.04,0.0,22.04
Balanced,91.01,38.12,52.89,1.94,38.32,-36.38,85.61,72.27,13.34,81.38,0.0,81.38
Intermediate,92.11,38.12,53.99,1.84,38.32,-36.48,94.35,78.97,15.38,83.28,0.0,83.28
V1-A,98.05,61.68,36.37,0.00,0.00,0.00,92.80,85.53,7.28,94.98,0.0,94.98
V1-B,98.06,61.68,36.38,0.00,0.00,0.00,88.46,78.67,9.78,94.99,0.0,94.99
V1-C,98.16,61.68,36.48,0.00,0.00,0.00,88.43,64.89,23.54,95.20,0.0,95.20


... (9 outputs left out)

,configuration,newly_above_P95_pct_High,newly_above_P95_pct_Medium,newly_above_P95_pct_Low
0,Conservative,10.16,12.63,23.18
1,Balanced,12.63,23.18,33.75
2,Intermediate,12.63,23.18,35.77
3,V1-A,12.63,33.75,54.89
4,V1-B,23.18,35.77,56.63
5,V1-C,23.18,54.89,60.92



B3 — TEMPORAL IMPACT


,configuration,level,median_added,increase_7d_pct,increase_14d_pct,increase_30d_pct,channel_share_shift_pp,dominant_changed_pct
0,Conservative,High,1.0,10.53,10.71,10.42,1.36,1.18
1,Conservative,Medium,1.0,11.43,11.76,11.43,1.34,1.20
2,Conservative,Low,1.0,15.00,15.38,15.09,1.59,1.63
3,Balanced,High,1.0,11.43,11.76,11.43,1.34,1.20
4,Balanced,Medium,1.0,15.00,15.38,15.09,1.59,1.63
5,Balanced,Low,2.0,21.05,21.57,21.35,2.11,2.63
6,Intermediate,High,1.0,11.43,11.76,11.43,1.34,1.20
7,Intermediate,Medium,1.0,15.00,15.38,15.09,1.59,1.63
8,Intermediate,Low,2.0,22.22,23.08,23.08,2.24,2.84
9,V1-A,High,1.0,11.43,11.76,11.43,1.34,1.20



FINAL VALIDATION
Formula comparisons: 18
Multiplier configurations: 6
Fixed-formula comparisons: 2
Automatic selection: NO
Attack injection: NO
Files created or modified: NONE

PASS — Controlled comparison completed.



### 5.3 — Channel-Level Stealth Validation

**Objective:** Evaluate whether the leading formula–multiplier candidates exhibit consistent and meaningful behavior across Email, USB, and HTTP before final selection.

**Candidates:**
- Formula A + Balanced
- Formula B + V1-A

These are complete candidate configurations. Differences between them must not be attributed solely to the formula.

**Evaluation:**

1. **Channel Coverage**
   - Count active observations and users per channel.
   - Examine natural event counts and historical P95 references.

2. **Stealth-Level Separation**
   - Measure strict High < Medium < Low separation within each channel.
   - Identify cases where all three levels produce identical injection counts.
   - Examine separation under low natural activity (N = 1).

3. **Injection Count Impact**
   - Measure median and P95 added-event counts for each channel and stealth level.
   - Compare relative increases against natural daily event counts.

4. **Historical P95 Exposure**
   - Measure newly introduced P95 exceedances per channel and level.
   - Distinguish natural exceedances from newly introduced exceedances.

5. **Cross-Channel Consistency**
   - Identify channel-specific weaknesses that may be hidden by aggregate results.
   - Assess whether levels show meaningful variation without excessive distortion.

**Methodology:**
- Reuse the validated natural User–Day–Channel observations and baseline references.
- Reuse existing channel-level results where available.
- Calculate channel-level separation diagnostics without modifying source data.
- Do not create attack schedules or inject events.

**Expected Outcome:** Channel-specific evidence for assessing the leading candidates before selecting the final injection-count formula and stealth multipliers.

**Constraints:**
- Read-only analysis.
- No switching re-evaluation.
- No automatic selection or user exclusion.
- No file export or source modification.
- No model-based optimization.


In [21]:

# ============================================================
# CELL 5.3 — CHANNEL-LEVEL STEALTH VALIDATION
# Read-only | No selection | No file export
# ============================================================

import numpy as np
import pandas as pd
from IPython.display import display

LEVELS_53 = ["High", "Medium", "Low"]
CHANNELS_53 = ["EMAIL", "USB", "HTTP"]

CANDIDATES_53 = [
    ("A", "Balanced"),
    ("B", "V1-A"),
]

# ------------------------------------------------------------
# STEP 1 — Validate existing inputs
# ------------------------------------------------------------

required_53 = [
    "evaluation_data",
    "multiplier_sensitivity",
    "stealth_separation",
    "FORMULAS",
    "MULTIPLIER_GROUPS",
]

missing_53 = [
    name for name in required_53
    if name not in globals()
]

if missing_53:
    raise RuntimeError(
        "Missing notebook variables: "
        + ", ".join(missing_53)
        + "\nRun original notebook cells through "
          "Section 4.6 first."
    )

required_columns_53 = [
    "user",
    "day",
    "channel",
    "N",
    "baseline_p95",
]

absent_53 = (
    set(required_columns_53)
    - set(evaluation_data.columns)
)

assert not absent_53, (
    f"Missing evaluation columns: {sorted(absent_53)}"
)

assert len(evaluation_data) == 100_585
assert evaluation_data["user"].nunique() == 936

assert not evaluation_data.duplicated(
    ["user", "day", "channel"]
).any()

assert set(evaluation_data["channel"]) == set(
    CHANNELS_53
)

assert evaluation_data["N"].gt(0).all()
assert evaluation_data["baseline_p95"].gt(0).all()

for formula_name, config_name in CANDIDATES_53:
    assert formula_name in FORMULAS
    assert config_name in MULTIPLIER_GROUPS

print("STEP 1 — Input validation: PASS")


# ------------------------------------------------------------
# STEP 2 — Natural channel coverage
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("STEP 2 — NATURAL CHANNEL COVERAGE")
print("=" * 70)

channel_coverage_53 = (
    evaluation_data
    .groupby("channel")
    .agg(
        observations=("N", "size"),
        users=("user", "nunique"),
        median_N=("N", "median"),
        p95_N=("N", lambda x: np.percentile(x, 95)),
        median_baseline_P95=(
            "baseline_p95", "median"
        ),
        N1_observations=(
            "N", lambda x: int((x == 1).sum())
        ),
    )
    .reindex(CHANNELS_53)
    .reset_index()
)

channel_coverage_53["N1_pct"] = (
    100
    * channel_coverage_53["N1_observations"]
    / channel_coverage_53["observations"]
)

assert (
    channel_coverage_53["observations"].sum()
    == len(evaluation_data)
)

display(channel_coverage_53.round(2))


# ------------------------------------------------------------
# STEP 3 — Channel-level stealth separation
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("STEP 3 — CHANNEL-LEVEL STEALTH SEPARATION")
print("=" * 70)

separation_rows_53 = []

for formula_name, config_name in CANDIDATES_53:

    formula_fn = FORMULAS[formula_name]
    multipliers = MULTIPLIER_GROUPS[config_name]

    for channel in CHANNELS_53:

        sub = evaluation_data.loc[
            evaluation_data["channel"].eq(channel)
        ]

        N = sub["N"].to_numpy(dtype=np.int64)

        P95 = sub["baseline_p95"].to_numpy(
            dtype=float
        )

        additions = {}

        for level in LEVELS_53:
            additions[level] = np.asarray(
                formula_fn(
                    N,
                    P95,
                    multipliers[level],
                ),
                dtype=np.int64,
            )

        H = additions["High"]
        M = additions["Medium"]
        L = additions["Low"]

        assert np.all((H >= 1) & (H <= M) & (M <= L))

        strict = (H < M) & (M < L)
        equal = (H == M) & (M == L)

        mask_n1 = N == 1

        separation_rows_53.append({
            "formula": formula_name,
            "configuration": config_name,
            "channel": channel,
            "observations": len(N),
            "strict_separation_pct": (
                100 * strict.mean()
            ),
            "all_equal_pct": (
                100 * equal.mean()
            ),
            "N1_observations": int(mask_n1.sum()),
            "strict_N1_pct": (
                100 * strict[mask_n1].mean()
                if mask_n1.any()
                else np.nan
            ),
        })

channel_separation_53 = pd.DataFrame(
    separation_rows_53
)

assert len(channel_separation_53) == 6

display(
    channel_separation_53.round(2)
)


# ------------------------------------------------------------
# STEP 4 — Existing channel-level injection impact
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("STEP 4 — CHANNEL-LEVEL INJECTION IMPACT")
print("=" * 70)

impact_metrics_53 = [
    "median_added",
    "p95_added",
    "median_increase_pct",
    "final_above_P95_pct",
    "newly_above_P95_pct",
]

required_impact_columns_53 = [
    "formula",
    "configuration",
    "channel",
    "level",
    "observations",
    "total_added",
] + impact_metrics_53

assert set(required_impact_columns_53).issubset(
    multiplier_sensitivity.columns
)

selected_keys_53 = set(CANDIDATES_53)

impact_53 = multiplier_sensitivity.loc[
    multiplier_sensitivity["channel"].isin(
        CHANNELS_53
    )
    & multiplier_sensitivity[
        ["formula", "configuration"]
    ].apply(tuple, axis=1).isin(selected_keys_53),
    required_impact_columns_53,
].copy()

assert len(impact_53) == 18

assert not impact_53.duplicated(
    ["formula", "configuration", "channel", "level"]
).any()

display(
    impact_53.sort_values(
        ["formula", "configuration", "channel", "level"]
    )
    .round(2)
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# STEP 5 — Distinguish natural and new P95 exceedances
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("STEP 5 — P95 EXPOSURE BY CHANNEL")
print("=" * 70)

natural_p95_53 = (
    evaluation_data.assign(
        naturally_above_P95=(
            evaluation_data["N"]
            > evaluation_data["baseline_p95"]
        )
    )
    .groupby("channel")["naturally_above_P95"]
    .mean()
    .mul(100)
    .rename("natural_above_P95_pct")
    .reset_index()
)

p95_exposure_53 = impact_53.merge(
    natural_p95_53,
    on="channel",
    validate="many_to_one",
)

p95_exposure_53["expected_final_pct"] = (
    p95_exposure_53["natural_above_P95_pct"]
    + p95_exposure_53["newly_above_P95_pct"]
)

assert np.allclose(
    p95_exposure_53["final_above_P95_pct"],
    p95_exposure_53["expected_final_pct"],
    atol=1e-8,
)

display(
    p95_exposure_53[
        [
            "formula",
            "configuration",
            "channel",
            "level",
            "natural_above_P95_pct",
            "newly_above_P95_pct",
            "final_above_P95_pct",
        ]
    ]
    .sort_values(
        ["formula", "configuration", "channel", "level"]
    )
    .round(2)
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# STEP 6 — Independent consistency checks
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("STEP 6 — CROSS-CHANNEL VALIDATION")
print("=" * 70)

for formula_name, config_name in CANDIDATES_53:

    parts = channel_separation_53.loc[
        (channel_separation_53["formula"] == formula_name)
        & (
            channel_separation_53["configuration"]
            == config_name
        )
    ]

    weights = parts["observations"].to_numpy()

    weighted_strict = np.average(
        parts["strict_separation_pct"],
        weights=weights,
    )

    weighted_equal = np.average(
        parts["all_equal_pct"],
        weights=weights,
    )

    overall = stealth_separation.loc[
        (stealth_separation["formula"] == formula_name)
        & (
            stealth_separation["configuration"]
            == config_name
        )
    ]

    assert len(overall) == 1

    assert np.isclose(
        weighted_strict,
        overall["strict_separation_pct"].iloc[0],
        atol=1e-8,
    )

    assert np.isclose(
        weighted_equal,
        overall["all_equal_pct"].iloc[0],
        atol=1e-8,
    )

    for level in LEVELS_53:

        rows = impact_53.loc[
            (impact_53["formula"] == formula_name)
            & (
                impact_53["configuration"]
                == config_name
            )
            & (impact_53["level"] == level)
        ]

        overall_impact = multiplier_sensitivity.loc[
            (multiplier_sensitivity["formula"] == formula_name)
            & (
                multiplier_sensitivity["configuration"]
                == config_name
            )
            & (multiplier_sensitivity["channel"] == "ALL")
            & (multiplier_sensitivity["level"] == level)
        ]

        assert len(rows) == 3
        assert len(overall_impact) == 1

        assert (
            rows["observations"].sum()
            == len(evaluation_data)
        )

        assert (
            rows["total_added"].sum()
            == overall_impact["total_added"].iloc[0]
        )

print("Candidates analyzed:", len(CANDIDATES_53))
print("Channels:", len(CHANNELS_53))
print("Channel-level experiments:", len(impact_53))
print("Separation reconciled: YES")
print("Injection totals reconciled: YES")
print("Automatic selection: NO")
print("Attack injection: NO")
print("Files exported or modified: NONE")

print("\nPASS — Channel-level validation completed.")


STEP 1 — Input validation: PASS

STEP 2 — NATURAL CHANNEL COVERAGE


,channel,observations,users,median_N,p95_N,median_baseline_P95,N1_observations,N1_pct
0,EMAIL,57815,936,2.0,5.0,4.0,24011,41.53
1,USB,29612,690,2.0,8.0,4.0,10383,35.06
2,HTTP,13158,533,2.0,11.0,4.0,4153,31.56



STEP 3 — CHANNEL-LEVEL STEALTH SEPARATION


,formula,configuration,channel,observations,strict_separation_pct,all_equal_pct,N1_observations,strict_N1_pct
0,A,Balanced,EMAIL,57815,89.96,3.28,24011,79.28
1,A,Balanced,USB,29612,92.07,0.18,10383,83.72
2,A,Balanced,HTTP,13158,93.22,0.00,4153,87.60
3,B,V1-A,EMAIL,57815,58.47,0.00,24011,0.00
4,B,V1-A,USB,29612,64.94,0.00,10383,0.00
5,B,V1-A,HTTP,13158,68.44,0.00,4153,0.00



STEP 4 — CHANNEL-LEVEL INJECTION IMPACT


,formula,configuration,channel,level,observations,total_added,median_added,p95_added,median_increase_pct,final_above_P95_pct,newly_above_P95_pct
0,A,Balanced,EMAIL,High,57815,79920,1.0,2.0,100.00,18.67,15.14
1,A,Balanced,EMAIL,Low,57815,201881,3.0,5.0,166.67,70.60,67.06
2,A,Balanced,EMAIL,Medium,57815,135054,2.0,4.0,100.00,36.07,32.54
3,A,Balanced,HTTP,High,13158,24391,1.0,4.0,50.00,22.35,18.57
4,A,Balanced,HTTP,Low,13158,66266,3.0,12.0,150.00,94.36,90.58
5,A,Balanced,HTTP,Medium,13158,44478,2.0,8.0,100.00,54.56,50.78
6,A,Balanced,USB,High,29612,51286,1.0,3.0,66.67,19.64,15.79
7,A,Balanced,USB,Low,29612,131560,3.0,8.0,166.67,85.57,81.72
8,A,Balanced,USB,Medium,29612,89852,2.0,5.0,100.00,45.85,42.00
9,B,V1-A,EMAIL,High,57815,62206,1.0,2.0,50.00,16.31,12.77



STEP 5 — P95 EXPOSURE BY CHANNEL


,formula,configuration,channel,level,natural_above_P95_pct,newly_above_P95_pct,final_above_P95_pct
0,A,Balanced,EMAIL,High,3.54,15.14,18.67
1,A,Balanced,EMAIL,Low,3.54,67.06,70.60
2,A,Balanced,EMAIL,Medium,3.54,32.54,36.07
3,A,Balanced,HTTP,High,3.78,18.57,22.35
4,A,Balanced,HTTP,Low,3.78,90.58,94.36
5,A,Balanced,HTTP,Medium,3.78,50.78,54.56
6,A,Balanced,USB,High,3.85,15.79,19.64
7,A,Balanced,USB,Low,3.85,81.72,85.57
8,A,Balanced,USB,Medium,3.85,42.00,45.85
9,B,V1-A,EMAIL,High,3.54,12.77,16.31



STEP 6 — CROSS-CHANNEL VALIDATION
Candidates analyzed: 2
Channels: 3
Channel-level experiments: 18
Separation reconciled: YES
Injection totals reconciled: YES
Automatic selection: NO
Attack injection: NO
Files exported or modified: NONE

PASS — Channel-level validation completed.



### 5.4 — Final Stealth Rule Selection & Validation

**Objective:** Document and validate the selected unified injection-count formula and stealth multipliers using the evidence collected in Sections 3.1–5.3.

#### Selected Configuration

- Formula: A — Historical P95-Based
- Multiplier Configuration: Balanced
- High Stealth: 0.25
- Medium Stealth: 0.50
- Low Stealth: 0.75
- Channels: Email, USB, and HTTP

**Injection Count Formula:**

`I = max(1, ceil(m × Historical_P95))`

Where:
- I = Number of additional events
- m = Selected stealth multiplier
- Historical_P95 = User-specific historical channel reference

#### Selection Rationale

- Strong daily differentiation between stealth levels.
- Better separation under low natural activity than Formula B.
- Meaningful variation in added-event counts.
- Moderate cumulative impact relative to the more aggressive configurations.
- One unified formula and multiplier set across all three channels.

#### Known Limitations

- Low Stealth introduces substantial new P95 exceedances, particularly in USB and HTTP.
- Strict separation is not guaranteed for every individual observation.
- A P95 exceedance does not establish whether the MUBSIR model will detect an attack.
- The current evidence comes from hypothetical injection-count evaluations, not finalized attack schedules.

#### Validation

1. Verify the selected formula and multiplier values.
2. Confirm positive integer injection counts.
3. Verify High ≤ Medium ≤ Low for every observation.
4. Measure strict level separation for each channel.
5. Reconcile injection counts and P95 exceedances with prior evaluations.
6. Preserve original natural data and historical baselines.

**Expected Outcome:** A validated, unified injection-count rule ready for use during subsequent scenario planning and dry-run stages.

**Constraints:**
- Read-only validation.
- No actual attack injection.
- No user selection or exclusion.
- No switching re-evaluation.
- No source modification or file export.


In [22]:

# ============================================================
# CELL 5.4 — SELECTED STEALTH RULE VALIDATION
# Read-only | No injection | No file export
# ============================================================

import numpy as np
import pandas as pd
from IPython.display import display

# ------------------------------------------------------------
# STEP 1 — Selected configuration
# ------------------------------------------------------------

SELECTED_FORMULA = "A"
SELECTED_CONFIGURATION = "Balanced"

SELECTED_MULTIPLIERS = {
    "High": 0.25,
    "Medium": 0.50,
    "Low": 0.75,
}

LEVELS_54 = ["High", "Medium", "Low"]
CHANNELS_54 = ["EMAIL", "USB", "HTTP"]

required_vars = [
    "evaluation_data",
    "FORMULAS",
    "MULTIPLIER_GROUPS",
    "stealth_separation",
    "multiplier_sensitivity",
]

missing = [
    name for name in required_vars
    if name not in globals()
]

if missing:
    raise RuntimeError(
        "Run the earlier notebook cells first. "
        "Missing: " + ", ".join(missing)
    )

assert SELECTED_FORMULA in FORMULAS

assert (
    MULTIPLIER_GROUPS[SELECTED_CONFIGURATION]
    == SELECTED_MULTIPLIERS
)

assert (
    0 < SELECTED_MULTIPLIERS["High"]
    < SELECTED_MULTIPLIERS["Medium"]
    < SELECTED_MULTIPLIERS["Low"]
)

required_columns = {
    "user", "day", "channel",
    "N", "baseline_p95"
}

assert required_columns.issubset(
    evaluation_data.columns
)

assert not evaluation_data.duplicated(
    ["user", "day", "channel"]
).any()

print("STEP 1 — Configuration validation: PASS")


# ------------------------------------------------------------
# STEP 2 — Recalculate selected injection counts
# ------------------------------------------------------------

N = evaluation_data["N"].to_numpy(
    dtype=np.int64
)

P95 = evaluation_data[
    "baseline_p95"
].to_numpy(dtype=float)

assert np.all(N > 0)
assert np.all(np.isfinite(P95) & (P95 > 0))

formula_fn = FORMULAS[SELECTED_FORMULA]

added = {}

for level in LEVELS_54:

    m = SELECTED_MULTIPLIERS[level]

    values = np.asarray(
        formula_fn(N, P95, m),
        dtype=np.int64
    )

    assert len(values) == len(N)
    assert np.all(values >= 1)

    added[level] = values

H = added["High"]
M = added["Medium"]
L = added["Low"]

assert np.all((H <= M) & (M <= L))

print("STEP 2 — Injection-count validation: PASS")


# ------------------------------------------------------------
# STEP 3 — Level separation by channel
# ------------------------------------------------------------

strict = (H < M) & (M < L)
all_equal = (H == M) & (M == L)

rows = []

for channel in CHANNELS_54:

    mask = (
        evaluation_data["channel"]
        .to_numpy() == channel
    )

    n = N[mask]
    p = P95[mask]

    n1 = n == 1

    for level in LEVELS_54:

        injected = added[level][mask]

        natural_above = n > p
        final_above = (n + injected) > p

        newly_above = (
            ~natural_above & final_above
        )

        rows.append({
            "channel": channel,
            "level": level,
            "observations": int(mask.sum()),
            "users": int(
                evaluation_data.loc[
                    mask, "user"
                ].nunique()
            ),
            "multiplier":
                SELECTED_MULTIPLIERS[level],
            "median_added": float(
                np.median(injected)
            ),
            "p95_added": float(
                np.percentile(injected, 95)
            ),
            "strict_separation_pct": float(
                100 * strict[mask].mean()
            ),
            "all_equal_pct": float(
                100 * all_equal[mask].mean()
            ),
            "strict_N1_pct": (
                float(
                    100 * strict[mask][n1].mean()
                )
                if n1.any()
                else np.nan
            ),
            "new_P95_pct": float(
                100 * newly_above.mean()
            ),
            "final_P95_pct": float(
                100 * final_above.mean()
            ),
        })

selected_channel_validation = pd.DataFrame(rows)

assert len(selected_channel_validation) == 9

print("\nSTEP 3 — CHANNEL VALIDATION")

display(
    selected_channel_validation.round(2)
)


# ------------------------------------------------------------
# STEP 4 — Reconcile with prior evaluation
# ------------------------------------------------------------

previous = multiplier_sensitivity.loc[
    (
        multiplier_sensitivity["formula"]
        == SELECTED_FORMULA
    )
    & (
        multiplier_sensitivity["configuration"]
        == SELECTED_CONFIGURATION
    )
    & multiplier_sensitivity["channel"].isin(
        CHANNELS_54
    )
].copy()

assert len(previous) == 9

comparison_54 = (
    selected_channel_validation.merge(
        previous,
        on=["channel", "level"],
        suffixes=("_current", "_previous"),
        validate="one_to_one"
    )
)

for metric in [
    "median_added",
    "p95_added",
]:
    assert np.allclose(
        comparison_54[f"{metric}_current"],
        comparison_54[f"{metric}_previous"],
        atol=1e-8
    )

assert np.allclose(
    comparison_54["new_P95_pct"],
    comparison_54["newly_above_P95_pct"],
    atol=1e-8
)

assert np.allclose(
    comparison_54["final_P95_pct"],
    comparison_54["final_above_P95_pct"],
    atol=1e-8
)

previous_sep = stealth_separation.loc[
    (
        stealth_separation["formula"]
        == SELECTED_FORMULA
    )
    & (
        stealth_separation["configuration"]
        == SELECTED_CONFIGURATION
    )
]

assert len(previous_sep) == 1

assert np.isclose(
    100 * strict.mean(),
    previous_sep["strict_separation_pct"].iloc[0],
    atol=1e-8
)

assert np.isclose(
    100 * all_equal.mean(),
    previous_sep["all_equal_pct"].iloc[0],
    atol=1e-8
)

print("\nSTEP 4 — Prior evidence reconciliation: PASS")


# ------------------------------------------------------------
# STEP 5 — Prepare selected rule for later notebooks
# ------------------------------------------------------------

SELECTED_INJECTION_RULE = {
    "formula": SELECTED_FORMULA,
    "configuration": SELECTED_CONFIGURATION,
    "multipliers": SELECTED_MULTIPLIERS.copy(),
    "channels": CHANNELS_54.copy(),
    "minimum_added_events": 1,
    "status": "Selected; validation completed",
}

print("\n" + "=" * 65)
print("FINAL SELECTED CONFIGURATION")
print("=" * 65)

for key, value in SELECTED_INJECTION_RULE.items():
    print(f"{key}: {value}")

print("\nOverall strict separation:",
      f"{100 * strict.mean():.2f}%")

print("Overall all-equal:",
      f"{100 * all_equal.mean():.2f}%")

print("\nVALIDATION: PASS")
print("Actual attack injection: NO")
print("Source modifications: NONE")
print("Files created: NONE")


STEP 1 — Configuration validation: PASS
STEP 2 — Injection-count validation: PASS

STEP 3 — CHANNEL VALIDATION


,channel,level,observations,users,multiplier,median_added,p95_added,strict_separation_pct,all_equal_pct,strict_N1_pct,new_P95_pct,final_P95_pct
0,EMAIL,High,57815,936,0.25,1.0,2.0,89.96,3.28,79.28,15.14,18.67
1,EMAIL,Medium,57815,936,0.50,2.0,4.0,89.96,3.28,79.28,32.54,36.07
2,EMAIL,Low,57815,936,0.75,3.0,5.0,89.96,3.28,79.28,67.06,70.60
3,USB,High,29612,690,0.25,1.0,3.0,92.07,0.18,83.72,15.79,19.64
4,USB,Medium,29612,690,0.50,2.0,5.0,92.07,0.18,83.72,42.00,45.85
5,USB,Low,29612,690,0.75,3.0,8.0,92.07,0.18,83.72,81.72,85.57
6,HTTP,High,13158,533,0.25,1.0,4.0,93.22,0.00,87.60,18.57,22.35
7,HTTP,Medium,13158,533,0.50,2.0,8.0,93.22,0.00,87.60,50.78,54.56
8,HTTP,Low,13158,533,0.75,3.0,12.0,93.22,0.00,87.60,90.58,94.36



STEP 4 — Prior evidence reconciliation: PASS

FINAL SELECTED CONFIGURATION
formula: A
configuration: Balanced
multipliers: {'High': 0.25, 'Medium': 0.5, 'Low': 0.75}
channels: ['EMAIL', 'USB', 'HTTP']
minimum_added_events: 1
status: Selected; validation completed

Overall strict separation: 91.01%
Overall all-equal: 1.94%

VALIDATION: PASS
Actual attack injection: NO
Source modifications: NONE
Files created: NONE



### 5.5 — Final Validation & Rule Export

**Objective:** Perform final validation of the selected unified injection-count rule and export a reproducible configuration for subsequent scenario-planning notebooks.

**Selected Rule:**
- Formula: A — Historical P95-Based
- Configuration: Balanced
- High Stealth: 0.25
- Medium Stealth: 0.50
- Low Stealth: 0.75
- Minimum injection count: 1 event
- Channels: Email, USB, HTTP

**Validation:**
1. Verify the selected configuration against the original formula and multiplier definitions.
2. Confirm positive integer injection counts.
3. Validate the High ≤ Medium ≤ Low relationship.
4. Reconcile channel-level results with previously validated evidence.
5. Confirm that the historical baseline and natural data remain unchanged.
6. Document strict-level separation and newly introduced P95 exceedances.

**Final Outputs:**
- `selected_injection_rule.json` — Approved unified formula and multiplier configuration.
- `selected_stealth_validation.csv` — Channel-level validation metrics.
- `final_stealth_validation_report.md` — Validation findings and limitations.

**Limitations:**
- Results are based on hypothetical injection-count calculations.
- Low Stealth can introduce substantial historical P95 exceedances.
- Final attack schedules, injection feasibility, and model detection performance are not established here.

**Constraints:**
- No attack injection.
- No switching re-evaluation.
- No user assignment or exclusion.
- No modification of original datasets or baseline statistics.

**Expected Outcome:** Validated, documented injection-count rules ready for Notebook 07.


In [24]:

# ============================================================
# CELL 5.5 — FINAL VALIDATION & RULE EXPORT
# No attack injection | No source modification
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import numpy as np
import pandas as pd
from IPython.display import display

# ------------------------------------------------------------
# STEP 1 — Validate notebook state
# ------------------------------------------------------------

required_vars = [
    "ROOT",
    "evaluation_data",
    "FORMULAS",
    "MULTIPLIER_GROUPS",
    "SELECTED_INJECTION_RULE",
    "selected_channel_validation",
    "stealth_separation",
    "multiplier_sensitivity",
]

missing = [
    name for name in required_vars
    if name not in globals()
]

if missing:
    raise RuntimeError(
        "Missing required variables: "
        + ", ".join(missing)
        + "\nRun previous notebook cells through 5.4."
    )

rule = SELECTED_INJECTION_RULE.copy()

assert rule["formula"] == "A"
assert rule["configuration"] == "Balanced"
assert rule["minimum_added_events"] == 1

multipliers = rule["multipliers"]

assert multipliers == {
    "High": 0.25,
    "Medium": 0.50,
    "Low": 0.75,
}

assert MULTIPLIER_GROUPS["Balanced"] == multipliers

channels = ["EMAIL", "USB", "HTTP"]
levels = ["High", "Medium", "Low"]

assert set(rule["channels"]) == set(channels)
assert set(evaluation_data["channel"]) == set(channels)

assert len(evaluation_data) == 100_585
assert evaluation_data["user"].nunique() == 936

assert not evaluation_data.duplicated(
    ["user", "day", "channel"]
).any()

assert evaluation_data["N"].gt(0).all()
assert evaluation_data["baseline_p95"].gt(0).all()

print("STEP 1 — Configuration and inputs: PASS")


# ------------------------------------------------------------
# STEP 2 — Independent formula verification
# ------------------------------------------------------------

N = evaluation_data["N"].to_numpy(
    dtype=np.int64
)

P95 = evaluation_data[
    "baseline_p95"
].to_numpy(dtype=float)

assert np.isfinite(P95).all()

calculated = {}

for level in levels:

    m = multipliers[level]

    expected = np.maximum(
        1,
        np.ceil(m * P95)
    ).astype(np.int64)

    actual = np.asarray(
        FORMULAS["A"](N, P95, m),
        dtype=np.int64
    )

    assert np.array_equal(actual, expected)
    assert np.all(actual >= 1)

    calculated[level] = actual

H = calculated["High"]
M = calculated["Medium"]
L = calculated["Low"]

assert np.all(H <= M)
assert np.all(M <= L)

strict = (H < M) & (M < L)
all_equal = (H == M) & (M == L)

overall_strict_pct = float(
    100 * strict.mean()
)

overall_equal_pct = float(
    100 * all_equal.mean()
)

assert np.isclose(
    overall_strict_pct,
    91.01,
    atol=0.02
)

assert np.isclose(
    overall_equal_pct,
    1.94,
    atol=0.02
)

print("STEP 2 — Formula verification: PASS")


# ------------------------------------------------------------
# STEP 3 — Validate channel-level results
# ------------------------------------------------------------

validation = selected_channel_validation.copy()

key = ["channel", "level"]

assert len(validation) == 9
assert not validation.duplicated(key).any()

expected_keys = {
    (channel, level)
    for channel in channels
    for level in levels
}

assert set(
    map(tuple, validation[key].to_numpy())
) == expected_keys

for channel in channels:

    mask = (
        evaluation_data["channel"].to_numpy()
        == channel
    )

    n = N[mask]
    p95 = P95[mask]

    for level in levels:

        row = validation.loc[
            validation["channel"].eq(channel)
            & validation["level"].eq(level)
        ]

        assert len(row) == 1
        row = row.iloc[0]

        added = calculated[level][mask]

        natural_above = n > p95
        final_above = (n + added) > p95

        newly_above = (
            ~natural_above
            & final_above
        )

        assert int(row["observations"]) == len(n)

        assert np.isclose(
            row["median_added"],
            np.median(added),
            atol=1e-8
        )

        assert np.isclose(
            row["new_P95_pct"],
            100 * newly_above.mean(),
            atol=1e-8
        )

        assert np.isclose(
            row["final_P95_pct"],
            100 * final_above.mean(),
            atol=1e-8
        )

print("STEP 3 — Channel-level validation: PASS")


# ------------------------------------------------------------
# STEP 4 — Reconcile with previous evidence
# ------------------------------------------------------------

previous = multiplier_sensitivity.loc[
    multiplier_sensitivity["formula"].eq("A")
    & multiplier_sensitivity[
        "configuration"
    ].eq("Balanced")
    & multiplier_sensitivity[
        "channel"
    ].isin(channels)
].copy()

assert len(previous) == 9

merged = validation.merge(
    previous,
    on=["channel", "level"],
    suffixes=("_validated", "_previous"),
    validate="one_to_one"
)

assert len(merged) == 9

for metric in ["median_added", "p95_added"]:

    assert np.allclose(
        merged[f"{metric}_validated"],
        merged[f"{metric}_previous"],
        atol=1e-8
    )

assert np.allclose(
    merged["new_P95_pct"],
    merged["newly_above_P95_pct"],
    atol=1e-8
)

assert np.allclose(
    merged["final_P95_pct"],
    merged["final_above_P95_pct"],
    atol=1e-8
)

print("STEP 4 — Previous evidence reconciliation: PASS")


# ------------------------------------------------------------
# STEP 5 — Prepare output files
# ------------------------------------------------------------

report_dir = (
    Path(ROOT)
    / "reports"
    / "06_Stealth_and_Injection_Count_Feasibility"
)

rule_path = report_dir / "selected_injection_rule.json"

validation_path = (
    report_dir / "selected_stealth_validation.csv"
)

report_path = (
    report_dir / "final_stealth_validation_report.md"
)

timestamp = datetime.now(
    timezone.utc
).isoformat()

export_rule = {
    "project": "MUBSIR",
    "notebook": "06_Stealth_and_Injection_Count_Feasibility",
    "formula_id": "A",
    "formula_name": "Historical P95-Based",
    "formula_expression": "max(1, ceil(m * historical_P95))",
    "configuration": "Balanced",
    "stealth_multipliers": multipliers,
    "channels": channels,
    "minimum_added_events": 1,
    "baseline_mutable": False,
    "selection_status": "selected_and_validated",
    "validation_scope": "hypothetical active-channel observations",
    "observations_validated": int(len(evaluation_data)),
    "users_validated": int(
        evaluation_data["user"].nunique()
    ),
    "strict_separation_pct": round(
        overall_strict_pct, 4
    ),
    "all_equal_pct": round(
        overall_equal_pct, 4
    ),
    "created_utc": timestamp,
}

# ------------------------------------------------------------
# Build Markdown table without tabulate dependency
# ------------------------------------------------------------

def dataframe_to_markdown_no_tabulate(df):
    """
    Convert a DataFrame to a Markdown table using
    only pandas and Python standard functionality.
    """

    clean = df.round(2).copy()

    clean = clean.where(pd.notna(clean), "")

    headers = [str(col) for col in clean.columns]

    def escape_cell(value):
        return str(value).replace("|", r"\|").replace("\n", " ")

    lines = [
        "| " + " | ".join(
            escape_cell(col) for col in headers
        ) + " |",
        "| " + " | ".join(
            ["---"] * len(headers)
        ) + " |",
    ]

    for row in clean.itertuples(index=False, name=None):
        lines.append(
            "| " + " | ".join(
                escape_cell(value) for value in row
            ) + " |"
        )

    return "\n".join(lines)


validation_markdown = dataframe_to_markdown_no_tabulate(
    validation
)

report_lines = [
    "# MUBSIR — Final Stealth Rule Validation",
    "",
    "## 1. Selected Configuration",
    "",
    "- Formula: A — Historical P95-Based",
    "- Configuration: Balanced",
    "- High multiplier: 0.25",
    "- Medium multiplier: 0.50",
    "- Low multiplier: 0.75",
    "- Minimum added events: 1",
    "- Applies to EMAIL, USB, and HTTP",
    "",
    "## 2. Validation Results",
    "",
    f"- Observations: {len(evaluation_data):,}",
    f"- Users: {evaluation_data['user'].nunique():,}",
    f"- Strict level separation: {overall_strict_pct:.2f}%",
    f"- All levels equal: {overall_equal_pct:.2f}%",
    "- Formula and integer-count checks: PASS",
    "- Channel-level checks: PASS",
    "- Previous evidence reconciliation: PASS",
    "",
    "## 3. Channel-Level Results",
    "",
    validation_markdown,
    "",
    "## 4. Limitations",
    "",
    "- Validation uses hypothetical injection-count calculations.",
    "- Low Stealth introduces substantial new P95 exceedances,",
    "  particularly in USB and HTTP.",
    "- Strict level separation is not guaranteed for every observation.",
    "- Final scenario scheduling and injection feasibility are pending.",
    "- Model detection performance has not been evaluated.",
    "",
    "## 5. Handoff",
    "",
    "- Use selected_injection_rule.json in subsequent notebooks.",
    "- Preserve historical baseline statistics.",
    "- No actual attack injection was performed.",
    "",
]

rule_text = json.dumps(
    export_rule,
    indent=2,
    ensure_ascii=False
)

report_text = "\n".join(report_lines)

assert len(validation) == 9
assert len(rule_text) > 0
assert len(report_text) > 0

# Prevent accidental overwriting
output_paths = [
    rule_path,
    validation_path,
    report_path,
]

existing = [
    path for path in output_paths
    if path.exists()
]

if existing:
    raise FileExistsError(
        "Output files already exist. Review before replacing:\n"
        + "\n".join(map(str, existing))
    )

report_dir.mkdir(
    parents=True,
    exist_ok=True
)

rule_path.write_text(
    rule_text,
    encoding="utf-8"
)

validation.to_csv(
    validation_path,
    index=False
)

report_path.write_text(
    report_text,
    encoding="utf-8"
)

print("STEP 5 — Rule export: PASS")


# ------------------------------------------------------------
# STEP 6 — Verify exported files
# ------------------------------------------------------------

with rule_path.open(
    "r",
    encoding="utf-8"
) as f:
    verified_rule = json.load(f)

verified_validation = pd.read_csv(
    validation_path
)

assert verified_rule["formula_id"] == "A"
assert verified_rule["configuration"] == "Balanced"

assert (
    verified_rule["stealth_multipliers"]
    == multipliers
)

assert len(verified_validation) == 9

assert set(
    map(
        tuple,
        verified_validation[key].to_numpy()
    )
) == expected_keys

assert report_path.is_file()
assert report_path.stat().st_size > 0

print("\n" + "=" * 65)
print("FINAL VALIDATION & EXPORT")
print("=" * 65)

print("Formula: A")
print("Configuration: Balanced")

print(
    "Overall strict separation:",
    f"{overall_strict_pct:.2f}%"
)

print(
    "Overall all-equal:",
    f"{overall_equal_pct:.2f}%"
)

print("\nFiles created:")

for path in output_paths:
    print(" -", path)

print("\nSource datasets modified: NO")
print("Actual attack injection: NO")
print("Final validation: PASS")


STEP 1 — Configuration and inputs: PASS
STEP 2 — Formula verification: PASS
STEP 3 — Channel-level validation: PASS
STEP 4 — Previous evidence reconciliation: PASS
STEP 5 — Rule export: PASS

FINAL VALIDATION & EXPORT
Formula: A
Configuration: Balanced
Overall strict separation: 91.01%
Overall all-equal: 1.94%

Files created:
 - C:\Users\hp\Documents\FYP_Data_Exfiltration_Organized\FYP_Data_Exfiltration\New_Attack_Simulation\reports\06_Stealth_and_Injection_Count_Feasibility\selected_injection_rule.json
 - C:\Users\hp\Documents\FYP_Data_Exfiltration_Organized\FYP_Data_Exfiltration\New_Attack_Simulation\reports\06_Stealth_and_Injection_Count_Feasibility\selected_stealth_validation.csv
 - C:\Users\hp\Documents\FYP_Data_Exfiltration_Organized\FYP_Data_Exfiltration\New_Attack_Simulation\reports\06_Stealth_and_Injection_Count_Feasibility\final_stealth_validation_report.md

Source datasets modified: NO
Actual attack injection: NO
Final validation: PASS



### 5.6 — Final Data Dictionary

**Objective:** Document and validate the schema of the final channel-level stealth validation output.

**Input:**
- `selected_stealth_validation.csv`

**Validation:**
1. Verify the source file exists and is readable.
2. Check that the nine channel-level records are complete.
3. Verify the uniqueness of Channel–Level keys.
4. Document every column with its data type and meaning.
5. Confirm that no undocumented columns remain.

**Output:**
- `selected_stealth_validation_dictionary.csv`

**Constraints:**
- No injection or formula recalculation.
- No modification of original datasets.
- No replacement of existing output files.


In [25]:

# ============================================================
# CELL 5.6 — FINAL DATA DICTIONARY
# Documentation only | No source modification
# ============================================================

from pathlib import Path
import pandas as pd

# STEP 1 — Locate verified results

report_dir = (
    Path(ROOT)
    / "reports"
    / "06_Stealth_and_Injection_Count_Feasibility"
)

source_path = report_dir / "selected_stealth_validation.csv"
dictionary_path = (
    report_dir / "selected_stealth_validation_dictionary.csv"
)

assert source_path.is_file(), f"Missing: {source_path}"

validation_df = pd.read_csv(source_path)

assert len(validation_df) == 9
assert not validation_df.duplicated(
    ["channel", "level"]
).any()

assert set(validation_df["channel"]) == {
    "EMAIL", "USB", "HTTP"
}

assert set(validation_df["level"]) == {
    "High", "Medium", "Low"
}

assert validation_df.groupby("channel")["level"].nunique().eq(3).all()

print("STEP 1 — Source validation: PASS")


# STEP 2 — Define column documentation

descriptions = {
    "channel": "Target activity channel: EMAIL, USB, or HTTP.",
    "level": "Selected stealth level: High, Medium, or Low.",
    "observations": "Number of evaluated active User-Day-Channel observations.",
    "users": "Number of distinct users represented in this channel.",
    "multiplier": "Selected multiplier applied to historical P95.",
    "median_added": "Median number of additional events per observation.",
    "p95_added": "95th percentile of additional event counts.",
    "strict_separation_pct": "Percentage of observations satisfying High < Medium < Low injection counts.",
    "all_equal_pct": "Percentage of observations where all three stealth levels have identical counts.",
    "strict_N1_pct": "Strict separation percentage for observations with one natural event.",
    "new_P95_pct": "Percentage of observations newly exceeding historical P95 after hypothetical injection.",
    "final_P95_pct": "Percentage of observations exceeding historical P95 after hypothetical injection, including pre-existing exceedances.",
}

# Ensure exact schema coverage.
actual_columns = list(validation_df.columns)

missing_docs = set(actual_columns) - set(descriptions)
unused_docs = set(descriptions) - set(actual_columns)

assert not missing_docs, (
    f"Undocumented columns: {sorted(missing_docs)}"
)

assert not unused_docs, (
    f"Unexpected dictionary entries: {sorted(unused_docs)}"
)


# STEP 3 — Build dictionary

dictionary_df = pd.DataFrame([
    {
        "column": col,
        "dtype": str(validation_df[col].dtype),
        "description": descriptions[col],
        "source": "selected_stealth_validation.csv",
    }
    for col in actual_columns
])

assert len(dictionary_df) == len(actual_columns)
assert dictionary_df["column"].is_unique
assert dictionary_df["description"].str.strip().ne("").all()

print("STEP 2 — Column documentation: PASS")


# STEP 4 — Export without overwriting

if dictionary_path.exists():
    raise FileExistsError(
        f"Dictionary already exists: {dictionary_path}"
    )

dictionary_df.to_csv(dictionary_path, index=False)

# STEP 5 — Read-back verification

verified = pd.read_csv(dictionary_path)

assert len(verified) == len(actual_columns)
assert verified["column"].tolist() == actual_columns
assert verified["description"].notna().all()

print("STEP 3 — Export verification: PASS")
print(f"Documented columns: {len(verified)}/{len(actual_columns)}")
print("Undocumented columns: 0")
print("Dictionary:", dictionary_path)

display(dictionary_df)


STEP 1 — Source validation: PASS
STEP 2 — Column documentation: PASS
STEP 3 — Export verification: PASS
Documented columns: 12/12
Undocumented columns: 0
Dictionary: C:\Users\hp\Documents\FYP_Data_Exfiltration_Organized\FYP_Data_Exfiltration\New_Attack_Simulation\reports\06_Stealth_and_Injection_Count_Feasibility\selected_stealth_validation_dictionary.csv


,column,dtype,description,source
0,channel,str,"Target activity channel: EMAIL, USB, or HTTP.",selected_stealth_validation.csv
1,level,str,"Selected stealth level: High, Medium, or Low.",selected_stealth_validation.csv
2,observations,int64,Number of evaluated active User-Day-Channel ob...,selected_stealth_validation.csv
3,users,int64,Number of distinct users represented in this c...,selected_stealth_validation.csv
4,multiplier,float64,Selected multiplier applied to historical P95.,selected_stealth_validation.csv
5,median_added,float64,Median number of additional events per observa...,selected_stealth_validation.csv
6,p95_added,float64,95th percentile of additional event counts.,selected_stealth_validation.csv
7,strict_separation_pct,float64,Percentage of observations satisfying High < M...,selected_stealth_validation.csv
8,all_equal_pct,float64,Percentage of observations where all three ste...,selected_stealth_validation.csv
9,strict_N1_pct,float64,Strict separation percentage for observations ...,selected_stealth_validation.csv



### 5.7 — Comprehensive Notebook Summary & Final Handoff

**Objective:** Generate a comprehensive, evidence-based summary of Notebook 06, documenting the complete analysis, comparison methodology, final rule selection, validation results, and handoff to subsequent simulation stages.

**Summary Coverage:**

1. **Data Scope & Integrity:** Source verification, eligible users, active observations, and structurally compatible windows.
2. **Candidate Formulas:** Definition and comparison of Formula A (Historical P95-Based) and Formula B (Natural Daily Count-Based).
3. **Multiplier Configurations:** Documentation of all six tested configurations and their High, Medium, and Low Stealth multipliers.
4. **Evaluation Criteria:** Daily level separation, low-activity behavior, injection counts, cumulative separation, P95 exposure, temporal impact, and cross-channel consistency.
5. **Comparative Results:** Numerical comparison of all 12 formula–configuration combinations and their controlled behavioral effects.
6. **Selection Rationale:** Evidence supporting Formula A + Balanced, including its advantages, trade-offs, and limitations.
7. **Final Validation & Deliverables:** Verification results, exported artifacts, and rules passed to Notebook 07.

**Selected Configuration:**

- Formula A: `I = max(1, ceil(m × historical_P95))`
- High Stealth: 0.25
- Medium Stealth: 0.50
- Low Stealth: 0.75
- Unified across EMAIL, USB, and HTTP.

**Output:**

- `reports/06_Stealth_and_Injection_Count_Feasibility/notebook_summary.md`

**Execution Rules:**

- Overwrite the existing `notebook_summary.md` with the complete revised summary.
- Maintain exactly one final summary file.
- Do not create backup files or duplicate summaries.
- Extract numerical evidence from previously validated notebook results.
- Do not modify original datasets, historical baselines, or previous analysis outputs.
- Do not perform attack injection or scenario assignment.

**Expected Outcome:** One complete, reproducible, and methodologically justified summary documenting what was evaluated, how candidates were compared, why the final configuration was selected, and which limitations remain.


In [28]:

# ============================================================
# FINAL CELL — NOTEBOOK 06 COMPREHENSIVE SUMMARY
# One summary file | Overwrite existing | No backup
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# STEP 1 — Verify all required results
# ------------------------------------------------------------

required_variables = [
    "ROOT",
    "FORMULAS",
    "MULTIPLIER_GROUPS",
    "evaluation_data",
    "compatible_windows",
    "daily_evidence",
    "edge_sensitivity",
    "cumulative_separation",
    "controlled_behavior",
]

missing = [
    name for name in required_variables
    if name not in globals()
]

if missing:
    raise RuntimeError(
        "Missing analysis variables: "
        + ", ".join(missing)
        + "\nRun Notebook 06 analysis cells first."
    )

report_dir = (
    Path(ROOT)
    / "reports"
    / "06_Stealth_and_Injection_Count_Feasibility"
)

summary_path = report_dir / "notebook_summary.md"
rule_path = report_dir / "selected_injection_rule.json"
validation_path = report_dir / "selected_stealth_validation.csv"
dictionary_path = (
    report_dir / "selected_stealth_validation_dictionary.csv"
)
detailed_path = (
    report_dir / "final_stealth_validation_report.md"
)

for path in [
    rule_path,
    validation_path,
    dictionary_path,
    detailed_path,
]:
    assert path.is_file(), f"Missing output: {path}"

with rule_path.open("r", encoding="utf-8") as f:
    rule = json.load(f)

validation = pd.read_csv(validation_path)
dictionary = pd.read_csv(dictionary_path)

assert set(FORMULAS) == {"A", "B"}
assert len(MULTIPLIER_GROUPS) == 6
assert len(daily_evidence) == 12
assert len(edge_sensitivity) == 12
assert len(cumulative_separation) == 12
assert len(controlled_behavior) == 36

assert len(evaluation_data) == 100_585
assert evaluation_data["user"].nunique() == 936
assert len(compatible_windows) == 288_486

assert rule["formula_id"] == "A"
assert rule["configuration"] == "Balanced"
assert rule["stealth_multipliers"] == {
    "High": 0.25,
    "Medium": 0.50,
    "Low": 0.75,
}

assert len(validation) == 9
assert not validation.duplicated(
    ["channel", "level"]
).any()

assert set(dictionary["column"]) == set(validation.columns)

print("STEP 1 — All source results verified: PASS")


# ------------------------------------------------------------
# STEP 2 — Prepare verified comparison data
# ------------------------------------------------------------

levels = ["High", "Medium", "Low"]
channels = ["EMAIL", "USB", "HTTP"]

config_order = [
    "Conservative",
    "Balanced",
    "Intermediate",
    "V1-A",
    "V1-B",
    "V1-C",
]

# All six sets must match the original notebook definitions.
assert set(config_order) == set(MULTIPLIER_GROUPS)

expected_multipliers = {
    "V1-A": (0.25, 0.75, 1.25),
    "V1-B": (0.50, 1.00, 1.50),
    "V1-C": (0.50, 1.25, 2.00),
    "Conservative": (0.10, 0.25, 0.50),
    "Intermediate": (0.25, 0.50, 1.00),
    "Balanced": (0.25, 0.50, 0.75),
}

for name, expected in expected_multipliers.items():
    actual = tuple(
        MULTIPLIER_GROUPS[name][level]
        for level in levels
    )
    assert np.allclose(actual, expected)


# Combine prior evaluation outputs.
comparison = (
    daily_evidence
    .merge(
        cumulative_separation[
            ["formula", "configuration", "nonoverlap_pct"]
        ],
        on=["formula", "configuration"],
        validate="one_to_one",
    )
)

assert len(comparison) == 12

# Select the final configuration.
chosen = comparison.loc[
    comparison["formula"].eq("A")
    & comparison["configuration"].eq("Balanced")
]

assert len(chosen) == 1
chosen = chosen.iloc[0]

# Verify reported selected metrics.
assert np.isclose(
    chosen["strict_separation_pct"], 91.01,
    atol=0.02
)

assert np.isclose(
    chosen["nonoverlap_pct"], 85.61,
    atol=0.02
)

n1_selected = edge_sensitivity.loc[
    edge_sensitivity["formula"].eq("A")
    & edge_sensitivity["configuration"].eq("Balanced"),
    "strict_N1"
]

assert len(n1_selected) == 1

# Controlled temporal results.
selected_temporal = controlled_behavior.loc[
    controlled_behavior["formula"].eq("A")
    & controlled_behavior["configuration"].eq("Balanced")
].set_index("level")

assert set(selected_temporal.index) == set(levels)

# Natural data coverage.
channel_scope = (
    evaluation_data.groupby("channel")
    .agg(
        observations=("N", "size"),
        users=("user", "nunique"),
        median_n=("N", "median")
    )
)

assert set(channel_scope.index) == set(channels)

print("STEP 2 — Comparison evidence verified: PASS")


# ------------------------------------------------------------
# STEP 3 — Markdown formatting helpers
# No tabulate dependency
# ------------------------------------------------------------

def num(value, digits=2):
    return f"{float(value):.{digits}f}"


def markdown_table(headers, rows):
    """Build a Markdown table without external packages."""

    def cell(value):
        return str(value).replace("|", "\\|").replace("\n", " ")

    result = [
        "| " + " | ".join(map(cell, headers)) + " |",
        "| " + " | ".join(["---"] * len(headers)) + " |",
    ]

    for row in rows:
        result.append(
            "| " + " | ".join(map(cell, row)) + " |"
        )

    return "\n".join(result)


# ------------------------------------------------------------
# STEP 4 — All multiplier configurations
# ------------------------------------------------------------

multiplier_rows = []

for config in config_order:
    m = MULTIPLIER_GROUPS[config]

    multiplier_rows.append([
        config,
        num(m["High"]),
        num(m["Medium"]),
        num(m["Low"]),
    ])

multipliers_table = markdown_table(
    ["Configuration", "High", "Medium", "Low"],
    multiplier_rows,
)


# ------------------------------------------------------------
# STEP 5 — Full 12-combination results
# ------------------------------------------------------------

comparison_rows = []

for formula in ["A", "B"]:
    for config in config_order:

        row = comparison.loc[
            comparison["formula"].eq(formula)
            & comparison["configuration"].eq(config)
        ]

        assert len(row) == 1
        row = row.iloc[0]

        edge = edge_sensitivity.loc[
            edge_sensitivity["formula"].eq(formula)
            & edge_sensitivity["configuration"].eq(config)
        ]

        assert len(edge) == 1
        edge = edge.iloc[0]

        comparison_rows.append([
            formula,
            config,
            num(row["strict_separation_pct"]),
            num(edge["strict_N1"]),
            num(row["nonoverlap_pct"]),
            num(row["newly_above_P95_pct_High"]),
            num(row["newly_above_P95_pct_Medium"]),
            num(row["newly_above_P95_pct_Low"]),
        ])

comparison_table = markdown_table(
    [
        "Formula",
        "Configuration",
        "Strict %",
        "N=1 Strict %",
        "Cumulative Non-overlap %",
        "New P95 High %",
        "New P95 Medium %",
        "New P95 Low %",
    ],
    comparison_rows,
)


# ------------------------------------------------------------
# STEP 6 — Temporal impact comparison
# ------------------------------------------------------------

temporal_rows = []

for formula in ["A", "B"]:
    for config in config_order:

        sub = controlled_behavior.loc[
            controlled_behavior["formula"].eq(formula)
            & controlled_behavior["configuration"].eq(config)
        ].set_index("level")

        assert set(sub.index) == set(levels)

        temporal_rows.append([
            formula,
            config,
            *[
                num(
                    sub.loc[level, "median_increase_30d_pct"]
                )
                for level in levels
            ],
        ])

temporal_table = markdown_table(
    [
        "Formula",
        "Configuration",
        "High 30d %",
        "Medium 30d %",
        "Low 30d %",
    ],
    temporal_rows,
)


# ------------------------------------------------------------
# STEP 7 — Channel-specific selected-rule findings
# ------------------------------------------------------------

channel_rows = []

for channel in channels:

    sub = validation.loc[
        validation["channel"].eq(channel)
    ].set_index("level")

    assert set(sub.index) == set(levels)

    channel_rows.append([
        channel,
        int(sub.loc["High", "observations"]),
        int(sub.loc["High", "users"]),
        num(sub.loc["High", "strict_separation_pct"]),
        num(sub.loc["High", "strict_N1_pct"]),
        num(sub.loc["High", "new_P95_pct"]),
        num(sub.loc["Medium", "new_P95_pct"]),
        num(sub.loc["Low", "new_P95_pct"]),
    ])

channel_table = markdown_table(
    [
        "Channel",
        "Observations",
        "Users",
        "Strict %",
        "N=1 Strict %",
        "New P95 High %",
        "New P95 Medium %",
        "New P95 Low %",
    ],
    channel_rows,
)


# ------------------------------------------------------------
# STEP 8 — Verified numerical selection rationale
# ------------------------------------------------------------

def get_case(formula, config):
    sub = comparison.loc[
        comparison["formula"].eq(formula)
        & comparison["configuration"].eq(config)
    ]
    assert len(sub) == 1
    return sub.iloc[0]


a_bal = get_case("A", "Balanced")
b_bal = get_case("B", "Balanced")
a_cons = get_case("A", "Conservative")
a_inter = get_case("A", "Intermediate")
a_v1c = get_case("A", "V1-C")

# The comparisons below are within the SAME configuration,
# except the separately identified multiplier comparisons.

assert a_bal["strict_separation_pct"] > (
    b_bal["strict_separation_pct"]
)

assert a_bal["strict_separation_pct"] > (
    a_cons["strict_separation_pct"]
)

assert (
    a_v1c["newly_above_P95_pct_High"]
    >= a_bal["newly_above_P95_pct_High"]
)

assert (
    a_inter["newly_above_P95_pct_Low"]
    >= a_bal["newly_above_P95_pct_Low"]
)

# Low-level P95 values for the selected configuration.
low_p95 = validation.loc[
    validation["level"].eq("Low")
].set_index("channel")["new_P95_pct"]


# ------------------------------------------------------------
# STEP 9 — Build complete, concise summary
# ------------------------------------------------------------

summary = f"""# MUBSIR — Notebook 06: Stealth and Injection Count Feasibility

## 1. Objective

Evaluate and select an event-count-based injection formula and stealth multipliers for MUBSIR's simulated channel-rotating low-and-slow insider data exfiltration.

The objective is to establish meaningful differences among High, Medium, and Low Stealth without automatically selecting the most aggressive or least detectable option.

This notebook evaluates hypothetical injection counts and behavioral effects. It does not perform final attacker assignment, attack scheduling, or actual injection.

## 2. Data Scope and Input Verification

- Dataset: CERT r6.2.
- Structurally compatible users: {evaluation_data["user"].nunique():,}.
- Compatible structural windows inherited from Notebook 05: {len(compatible_windows):,}.
- Active User-Day-Channel observations: {len(evaluation_data):,}.
- Channels: EMAIL, USB, HTTP.
- Historical Baseline: immutable.
- Injection analysis: event counts/frequency only.
- Evaluation: hypothetical, not an executed attack.

### Channel Coverage

{markdown_table(
    ["Channel", "Observations", "Users", "Median Natural Events"],
    [
        [
            c,
            int(channel_scope.loc[c, "observations"]),
            int(channel_scope.loc[c, "users"]),
            num(channel_scope.loc[c, "median_n"]),
        ]
        for c in channels
    ]
)}

Input verification covered source schemas, primary keys, natural event counts, historical reference availability, missing values, date boundaries, and reconciliation with processed User-Day sources.

## 3. Key Findings and Selection Evidence

### 3.1 — Candidate Injection Formulas

**Formula A — Historical P95-Based**

`I = max(1, ceil(m × P95))`

Uses the user's historical channel-specific P95 event-count reference.

**Formula B — Natural Daily Count-Based**

`I = max(1, ceil(m × N))`

Uses the user's current natural daily event count in the evaluated channel.

Where:
- `I`: additional injected events.
- `m`: stealth multiplier.
- `P95`: historical user/channel event-count P95.
- `N`: natural daily event count.
- Minimum additional events: 1.

Both formulas were tested on the same verified observations to permit controlled comparisons.

### 3.2 — All Six Tested Multiplier Configurations

{multipliers_table}

Each of the six configurations was tested using both formulas, giving **12 formula/configuration combinations** and **36 formula/configuration/stealth-level cases**.

### 3.3 — Evaluation Criteria

The configurations were evaluated using:

1. **Strict Stealth Separation:** Percentage of observations satisfying `I_High < I_Medium < I_Low`.
2. **Low-Activity Separation:** Strict separation when natural activity is `N = 1`, including additional low-activity and MAD-related sensitivity checks.
3. **Daily Injection Impact:** Distribution and magnitude of additional event counts.
4. **Historical P95 Exposure:** Percentage newly crossing the historical P95 after hypothetical injection, distinguished from pre-existing exceedances.
5. **Cumulative Level Separation:** Non-overlap between feasible cumulative injection-count ranges within compatible windows.
6. **Temporal Behavioral Impact:** Controlled changes in rolling 7-, 14-, and 30-day event-count behavior.
7. **Cross-Channel Consistency:** Whether conclusions remain meaningful across EMAIL, USB, and HTTP.
8. **Realism Trade-off:** Preference for differentiated, measurable attacks without automatically choosing excessive behavioral exposure.

**Comparison method:** Formula A and Formula B were compared with the same multiplier configuration. Multiplier configurations were compared while keeping the formula fixed. The final two candidate configurations were also reviewed across channels, but differences between those finalists cannot be attributed solely to the formula because their multipliers differed.

### 3.4 — Complete Daily and Cumulative Comparison

All percentages below come from the evaluated notebook results.

{comparison_table}

**Column meanings:**
- `Strict %`: daily High < Medium < Low separation.
- `N=1 Strict %`: separation when natural count is exactly one.
- `Cumulative Non-overlap %`: separation of cumulative level ranges.
- `New P95`: newly introduced historical-threshold exceedances, measured across all evaluated active observations.

### 3.5 — Controlled 30-Day Behavioral Impact

Median percentage increase in rolling 30-day event counts under the common hypothetical evaluation pattern:

{temporal_table}

The same controlled pattern was used for all candidates, so the comparison reflects the tested configuration rather than differences in attack scheduling. These are not finalized injected scenarios.

Additional notebook analyses examined rolling 7- and 14-day behavior, channel distribution, dominant-channel changes, cumulative injection bounds, and P95 exposure over compatible structural windows.

### 3.6 — Why Formula A Was Selected

With the **Balanced multipliers held constant**:

- A + Balanced daily strict separation: **{num(a_bal["strict_separation_pct"])}%**.
- B + Balanced daily strict separation: **{num(b_bal["strict_separation_pct"])}%**.
- A + Balanced separation at N=1: **{num(n1_selected.iloc[0])}%**.
- A + Balanced cumulative non-overlap: **{num(a_bal["nonoverlap_pct"])}%**.
- B + Balanced cumulative non-overlap: **{num(b_bal["nonoverlap_pct"])}%**.

Formula A was preferred because its historical P95 reference preserved stronger differentiation among stealth levels, especially when natural daily activity was low.

Formula B was less effective at separating the three levels under N=1 because its minimum-count and rounding behavior often produces identical added-event counts.

This is a rule-selection rationale, **not proof that Formula A improves MUBSIR model accuracy**.

### 3.7 — Why the Balanced Multipliers Were Selected

Formula A was held constant when comparing multiplier configurations.

- A + Conservative strict separation: **{num(a_cons["strict_separation_pct"])}%**; cumulative non-overlap: **{num(a_cons["nonoverlap_pct"])}%**.
- A + Balanced strict separation: **{num(a_bal["strict_separation_pct"])}%**; cumulative non-overlap: **{num(a_bal["nonoverlap_pct"])}%**.
- A + Intermediate strict separation: **{num(a_inter["strict_separation_pct"])}%**; cumulative non-overlap: **{num(a_inter["nonoverlap_pct"])}%**.
- A + V1-C strict separation: **{num(a_v1c["strict_separation_pct"])}%**, but with higher-impact multipliers.

The Conservative configuration yielded weaker level separation.

More aggressive configurations sometimes achieved higher separation but also produced greater added-event counts and stronger P95 exposure. In particular, under Formula A, Intermediate raised Low Stealth's new-P95 exposure to **{num(a_inter["newly_above_P95_pct_Low"])}%**, versus **{num(a_bal["newly_above_P95_pct_Low"])}%** for Balanced.

**Balanced (0.25, 0.50, 0.75)** was selected as a compromise: strong differentiation, measurable behavioral impact, and avoidance of the highest injection levels. It was not the maximum-scoring configuration on every individual metric.

### 3.8 — Structural Window and Cumulative Feasibility

- Reused **{len(compatible_windows):,}** passing structural windows for **936** users from Notebook 05.
- Reconciled compatible windows with actual User–Day timelines and natural channel activity.
- Evaluated candidate cumulative injection-count bounds and level separation.
- Examined cumulative P95 exposure and temporal impacts.
- No attack schedules were finalized and no injection was executed.

This notebook does not replace the switching, Off-Day, or duration eligibility rules approved in Notebook 05.

### 3.9 — Selected Rule Across All Channels

**Final configuration: Formula A + Balanced**

- High Stealth: **0.25**
- Medium Stealth: **0.50**
- Low Stealth: **0.75**

`I = max(1, ceil(m × historical_P95))`

The **same formula and multipliers** apply to EMAIL, USB, and HTTP. The resulting number of added events can differ by user and channel because their historical P95 values differ.

#### Verified Channel-Level Results

{channel_table}

Overall:
- Strict daily separation: **{num(rule["strict_separation_pct"])}%**.
- All three levels identical: **{num(rule["all_equal_pct"])}%**.
- Newly introduced P95 exceedances at Low Stealth: EMAIL **{num(low_p95["EMAIL"])}%**, USB **{num(low_p95["USB"])}%**, HTTP **{num(low_p95["HTTP"])}%**.

These high Low-Stealth exceedance rates, especially for USB and HTTP, are an explicit limitation of the selected configuration.

## 4. Validation

Completed checks include:

- Source discovery and schema/key verification.
- Historical baseline coverage and natural event-count reconciliation.
- Formula definition and multiplier consistency.
- Low-activity and MAD-related sensitivity checks.
- Daily stealth-level ordering and separation.
- Structural window timeline reconciliation.
- Cumulative injection bounds and level-separation checks.
- P95 exposure consistency.
- Controlled temporal and cross-channel evaluation.
- Selected-rule verification across all three channels.
- Reconciliation with previous analysis results.
- Exported file read-back verification.
- Final validation data dictionary coverage.

**Final rule validation: PASS.**

This validates the documented calculations and exported configuration, not the effectiveness of an unexecuted attack or model detection.

## 5. Final Deliverables

All stored under:
`New_Attack_Simulation/reports/06_Stealth_and_Injection_Count_Feasibility/`

- `selected_injection_rule.json` — selected formula and multiplier configuration.
- `selected_stealth_validation.csv` — validated results by channel and level.
- `selected_stealth_validation_dictionary.csv` — output column definitions.
- `final_stealth_validation_report.md` — detailed validation results.
- `notebook_summary.md` — complete final notebook summary.

Intermediate analytical tables remain notebook results unless explicitly exported elsewhere.

## 6. Limitations & Next Step

- Hypothetical evaluations only; no actual attack injection was performed.
- No final attacker selection or assignment was performed.
- No completed channel-rotation schedule was generated.
- Low Stealth can produce substantial newly introduced P95 exceedances.
- Strict separation is not guaranteed for every observation.
- Statistical P95 exposure does not establish model detection.
- Historical Baseline must remain immutable.
- Deployment or model performance is not evaluated in this notebook.

**Next: Notebook 07 — Scenario Catalog Design.**

Carry forward:
- Structural eligibility and switching rules from Notebook 05.
- The unified Formula A + Balanced rule from Notebook 06.
- Forced-only Off-Day constraints.
- A preference for feasible windows with the **fewest forced Off-Days** during final window selection, without changing eligibility rules.
- Later user assignment, dry-run, injection, ground-truth generation, and final QA.

## 7. Conclusion

Notebook 06 compared **two formulas and six multiplier configurations**, evaluated daily and cumulative impacts, tested low-activity behavior, reconciled compatible windows, and assessed cross-channel consistency.

**Formula A + Balanced (High=0.25, Medium=0.50, Low=0.75)** was selected for its strong stealth-level differentiation and comparatively restrained impact relative to the more aggressive alternatives.

The selected rule is documented, validated, and exported for subsequent simulation stages. Its limitations are explicitly recorded.
"""

# ------------------------------------------------------------
# STEP 10 — Final completeness checks BEFORE writing
# ------------------------------------------------------------

required_sections = [
    "## 1. Objective",
    "## 2. Data Scope",
    "## 3. Key Findings",
    "## 4. Validation",
    "## 5. Final Deliverables",
    "## 6. Limitations & Next Step",
    "## 7. Conclusion",
]

for section in required_sections:
    assert section in summary, section

for config in config_order:
    assert config in summary

assert "Formula A" in summary
assert "Formula B" in summary
assert "0.25" in summary
assert "0.50" in summary
assert "0.75" in summary
assert "91.01" in summary
assert "85.61" in summary
assert len(comparison_rows) == 12
assert len(temporal_rows) == 12
assert len(channel_rows) == 3
assert len(summary) > 7000

print("STEP 3 — Summary completeness: PASS")


# ------------------------------------------------------------
# STEP 11 — Replace the SAME summary file
# No backups | No additional files
# ------------------------------------------------------------

assert report_dir.is_dir()

# Overwrite the existing summary in place.
summary_path.write_text(
    summary,
    encoding="utf-8"
)

# Read-back verification.
verified = summary_path.read_text(encoding="utf-8")

assert verified == summary
assert all(
    section in verified
    for section in required_sections
)

print("STEP 4 — Existing summary overwritten: PASS")

print("\n" + "=" * 65)
print("NOTEBOOK 06 — FINAL COMPREHENSIVE SUMMARY")
print("=" * 65)

print("Formulas documented: 2")
print("Multiplier configurations documented: 6")
print("Full combination comparisons: 12")
print("Controlled temporal comparisons: 12")
print("Channel comparisons: 3")
print("Selection criteria and rationale: INCLUDED")
print("Known limitations: INCLUDED")
print("Final selected rule: Formula A + Balanced")
print("Summary files created: 0")
print("Backup files created: 0")
print("Existing summary updated: YES")
print("Source datasets modified: NO")
print("Actual attack injection: NO")

print("\nFINAL SUMMARY: PASS")
print("Path:", summary_path)


STEP 1 — All source results verified: PASS
STEP 2 — Comparison evidence verified: PASS
STEP 3 — Summary completeness: PASS
STEP 4 — Existing summary overwritten: PASS

NOTEBOOK 06 — FINAL COMPREHENSIVE SUMMARY
Formulas documented: 2
Multiplier configurations documented: 6
Full combination comparisons: 12
Controlled temporal comparisons: 12
Channel comparisons: 3
Selection criteria and rationale: INCLUDED
Known limitations: INCLUDED
Final selected rule: Formula A + Balanced
Summary files created: 0
Backup files created: 0
Existing summary updated: YES
Source datasets modified: NO
Actual attack injection: NO

FINAL SUMMARY: PASS
Path: C:\Users\hp\Documents\FYP_Data_Exfiltration_Organized\FYP_Data_Exfiltration\New_Attack_Simulation\reports\06_Stealth_and_Injection_Count_Feasibility\notebook_summary.md
